# California Toy & Collectible Retail — 完整数据清洗代码

本Notebook实际从已下载的四来源原始缓存执行提取、清洗、去重、实体关联、Tract／ACS JOIN、V3规则筛选和邻域特征重算，再导出CSV与Parquet。各步骤的实际函数和SQL均在代码单元中，不调用项目`.py`脚本。

**输入：**Overture原始Parquet分片、OSM固定PBF、四张ACS DAT、TIGER ZIP；以及已保存的人工审核裁决和冻结抽样文件。自动化部分复现的是“应用已完成审核的判断”，不会重新完成人工官网审核，也不会重新抽取留出样本。

**输出：**每次Run All创建独立的`notebook_runs/cleaning_v3_<时间>/`目录。现有发布数据和冻结研究输入保持原样。当前参照结果为58,513行×51列；其中548为目标店代理标签，不能解释为已逐家核实当前营业。

将本文件与`cleaning_notebook_inputs/`放在一起，在已有`data/raw/`的项目中运行。第三方依赖见输入目录的`requirements.txt`。首次运行会读取约1.3GB OSM及完整ACS文件，可能需要数分钟。

In [1]:
from pathlib import Path
from contextlib import contextmanager
from datetime import datetime, timezone
import gc,hashlib,json,os,shutil,sys,time
import duckdb,numpy as np,pyarrow as pa,pyarrow.parquet as pq
from IPython.display import display

# 如在其他位置运行，可手动将SOURCE_ROOT设为含data/raw的项目根目录。
SOURCE_ROOT=next((p for p in [Path.cwd(),*Path.cwd().parents]
                  if (p/'data/raw/osm/california-261024.osm.pbf').exists()),None)
if SOURCE_ROOT is None:
    raise FileNotFoundError('请将SOURCE_ROOT设为含四来源data/raw目录的项目根目录。')
bundle_options=[Path.cwd()/'cleaning_notebook_inputs',SOURCE_ROOT/'notebooks/cleaning_notebook_inputs',SOURCE_ROOT/'output/california_retail_v3/cleaning_notebook_inputs']
INPUTS=next((p for p in bundle_options if (p/'input_manifest.json').exists()),None)
if INPUTS is None:raise FileNotFoundError('缺少cleaning_notebook_inputs；请解压配套输入包。')
RUN_ROOT=SOURCE_ROOT/'notebook_runs'/('cleaning_v3_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ'))
RUN_ROOT.mkdir(parents=True,exist_ok=False)
assert RUN_ROOT.resolve()!=SOURCE_ROOT.resolve()
RUN_STARTED=time.monotonic()

def notebook_sha(path):
    digest=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda:stream.read(1024*1024),b''):digest.update(chunk)
    return digest.hexdigest()

@contextmanager
def notebook_args(*args):
    old=sys.argv[:]
    try:
        sys.argv=['notebook-stage',*map(str,args)]
        yield
    finally:sys.argv=old

def show_counts(path,label):
    with duckdb.connect() as con:
        n,unique=con.execute('SELECT count(*),count(DISTINCT id) FROM read_parquet(?)',[str(path)]).fetchone()
    print(label,':',f'{n:,}','rows;',f'{unique:,}','unique IDs')
    return n

protected=[SOURCE_ROOT/r for r in ['data/clean_v3/california_retail_clean_master_v3.parquet',
  'data/clean_v3/clean_dataset_manifest_v3.json','data/clean_v3/frozen_comparator_frame_v3.parquet',
  'data/clean_v3/quality_frame_manifest_v3.json','validation/comparator_holdout_v3/sample_frame.json',
  'validation/comparator_holdout_v3/sampling_design.json'] if (SOURCE_ROOT/r).exists()]
PROTECTED_BEFORE={str(p):notebook_sha(p) for p in protected}
input_manifest=json.loads((INPUTS/'input_manifest.json').read_text())
for rel,info in input_manifest['files'].items():
    assert notebook_sha(INPUTS/rel)==info['sha256'],rel
    if rel.startswith(('pipeline/','validation/')):
        dst=RUN_ROOT/rel;dst.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(INPUTS/rel,dst)
required=[SOURCE_ROOT/'data/raw/osm/california-261024.osm.pbf',
 SOURCE_ROOT/'data/raw/census/tiger2024/tl_2024_06_tract.zip',
 *[SOURCE_ROOT/f'data/raw/census/acs2024/acsdt5y2024-{t}.dat' for t in ['b01001','b19013','b14001','b11001']]]
assert all(p.is_file() for p in required),'缺少固定版本原始文件，见原数据获取教程。'
con=duckdb.connect()
try:con.execute('LOAD spatial')
except duckdb.Error:con.execute('INSTALL spatial');con.execute('LOAD spatial')
con.close()
print('Read-only sources:',SOURCE_ROOT)
print('Rebuilt outputs:',RUN_ROOT)
print('Verified research inputs:',len(input_manifest['files']))

Read-only sources: <PROJECT_ROOT>
Rebuilt outputs: <PROJECT_ROOT>/notebook_runs/cleaning_v3_20261026T010331_105860Z
Verified research inputs: 19


## 1. Overture：从原始分片提取地点字段

读取固定2026-10-21.0快照的本地原始分片，提取名称、类别、坐标、地址、来源、营业状态与存在性置信度；保留原始来源字段。全国云端检索函数也保留在本节，但默认仅执行本地分片处理，不重新查询网络。后续还会按加州多边形精确裁剪。

函数／常量加阶段前缀，避免同一内核中的同名变量相互覆盖。

In [2]:
# Source: pipeline/collect_overture_ca.py; SHA256: 252c04db81301c456152f0480dc171fead76c15869033709601f363d81b0e7c8
overture___file__ = str(RUN_ROOT / 'pipeline/collect_overture_ca.py')
overture___doc__ = 'Retrieve a frozen Overture Places snapshot for a California retail project.\n\nDefault: a small San Francisco sample, not all California. Full California\nrequires --full-ca. Every scope is a retrieval rectangle, not a state boundary.\nOriginal nested source fields are retained in raw_shards/*.parquet.\n\nRequires: pip install -r requirements_overture.txt'
#!/usr/bin/env python3
"""Retrieve a frozen Overture Places snapshot for a California retail project.

Default: a small San Francisco sample, not all California. Full California
requires --full-ca. Every scope is a retrieval rectangle, not a state boundary.
Original nested source fields are retained in raw_shards/*.parquet.

Requires: pip install -r requirements_overture.txt
"""



import argparse as overture_argparse
from concurrent.futures import ThreadPoolExecutor as overture_ThreadPoolExecutor
from datetime import datetime as overture_datetime, timezone as overture_timezone
import hashlib as overture_hashlib
import json as overture_json
import math as overture_math
from pathlib import Path as overture_Path
import sys as overture_sys
import time as overture_time
from urllib.parse import urlencode as overture_urlencode
from urllib.request import Request as overture_Request, urlopen as overture_urlopen
import xml.etree.ElementTree as overture_ET

import duckdb as overture_duckdb


overture_RELEASE = "2026-10-21.0"
overture_S3_ROOT = "https://overturemaps-us-west-2.s3.us-west-2.amazonaws.com/"
overture_AZURE_ROOT = "https://overturemapswestus2.blob.core.windows.net/"
overture_PREFIX = f"release/{overture_RELEASE}/theme=places/type=place/"
overture_STAC_URL = f"https://stac.overturemaps.org/{overture_RELEASE}/catalog.json"
overture_SF_SAMPLE_BBOX = (-122.420, 37.775, -122.410, 37.785)
# A deliberately enclosing rectangle. It includes ocean and neighboring states.
# Apply TIGER California polygons after retrieval; retain border POIs as context.
overture_CA_RETRIEVAL_BBOX = (-124.50, 32.50, -114.10, 42.02)

# Exact, reviewable paths for a discretionary consumer-retail sample frame.
# These are analytical design choices, not Overture's definitions of suitability.
overture_RETAIL_PARENT_ALLOW = (
    "department_store", "discount_store", "fashion_and_apparel_store",
    "second_hand_store", "specialty_store",
)
overture_RETAIL_PATH_DENY = (
    "adult_store", "auction_house", "building_supply_store", "car_stereo_store",
    "electrical_supply_store", "farming_equipment_and_supply_store", "flea_market",
    "gun_and_ammo_store", "hardware_home_and_garden_store", "junkyard",
    "livestock_feed_and_supply_store", "medical_supply_store", "personal_shopper",
    "pharmacy_and_drug_store", "pick_your_own_farm", "pop_up_store",
    "safety_equipment_store", "smoke_and_vape_store", "tactical_supply_store",
    "vehicle_parts_store", "video_and_video_game_rental", "welding_supply_store",
    "woodworking_supply_store",
)
overture_STRICT_TARGET_PROXY = ("toy_store", "toys_and_games_store")
overture_PENDING_CATEGORY_REVIEW = (
    "hobby_shop", "comic_books_store", "tabletop_games_store",
    "video_game_store", "arts_crafts_and_hobby_store",
)

In [3]:
def overture_log(message: str) -> None:
    print(message, flush=True)

In [4]:
def overture_fetch(url: str, *, method: str = "GET", attempts: int = 4) -> tuple[bytes, dict]:
    """Retry network reads; never replace a missing release with a newer one."""
    for attempt in range(attempts):
        try:
            request = overture_Request(url, method=method, headers={"User-Agent": "CIS5450-California-tutorial/1.0"})
            with overture_urlopen(request, timeout=60) as response:
                return response.read(), {"status": response.status, **dict(response.headers)}
        except Exception as exc:
            if attempt + 1 == attempts:
                raise RuntimeError(f"Could not {method} {url}: {exc}") from exc
            pause = min(2 ** attempt, 8)
            overture_log(f"Network retry {attempt + 1}/{attempts - 1} after {pause}s: {exc}")
            overture_time.sleep(pause)
    raise AssertionError("unreachable")

In [5]:
def overture_discover_manifest() -> list[dict]:
    """Use S3 ListObjectsV2 with pagination rather than hard-coded part names."""
    namespace = {"s": "http://s3.amazonaws.com/doc/2006-03-01/"}
    token = None
    assets = []
    while True:
        params = {"list-type": "2", "prefix": overture_PREFIX}
        if token:
            params["continuation-token"] = token
        body, _ = overture_fetch(overture_S3_ROOT + "?" + overture_urlencode(params))
        root = overture_ET.fromstring(body)
        for item in root.findall("s:Contents", namespace):
            key = item.findtext("s:Key", namespaces=namespace)
            if key and key.endswith(".parquet"):
                assets.append({
                    "key": key,
                    "size_bytes": int(item.findtext("s:Size", namespaces=namespace)),
                    "etag": item.findtext("s:ETag", namespaces=namespace),
                    "last_modified": item.findtext("s:LastModified", namespaces=namespace),
                    "s3_https_url": overture_S3_ROOT + key,
                    "azure_https_url": overture_AZURE_ROOT + key,
                })
        if root.findtext("s:IsTruncated", namespaces=namespace) != "true":
            break
        token = root.findtext("s:NextContinuationToken", namespaces=namespace)
        if not token:
            raise RuntimeError("Truncated S3 listing did not return a continuation token")
    if not assets:
        raise RuntimeError(f"No Parquet assets for frozen release {overture_RELEASE}; do not guess another date")
    return sorted(assets, key=lambda asset: asset["key"])

In [6]:
def overture_expand_bbox(bbox: tuple, buffer_meters: float) -> list[float]:
    """Conservative angular padding for retrieval, not a metric analysis buffer."""
    max_latitude = max(abs(bbox[1]), abs(bbox[3])) + 0.1
    dy = buffer_meters / 110_000.0
    dx = buffer_meters / (110_000.0 * overture_math.cos(overture_math.radians(max_latitude)))
    return [bbox[0] - dx, bbox[1] - dy, bbox[2] + dx, bbox[3] + dy]

In [7]:
def overture_sql_literal(value: str) -> str:
    return "'" + value.replace("'", "''") + "'"

In [8]:
def overture_sql_list(values: tuple) -> str:
    return "[" + ", ".join(overture_sql_literal(value) for value in values) + "]"

In [9]:
def overture_connect() -> overture_duckdb.DuckDBPyConnection:
    connection = overture_duckdb.connect()
    for extension in ("httpfs", "spatial"):
        try:
            connection.execute(f"LOAD {extension}")
        except overture_duckdb.Error:
            connection.execute(f"INSTALL {extension}")
            connection.execute(f"LOAD {extension}")
    connection.execute("SET threads=2")
    connection.execute("SET memory_limit='1GB'")
    connection.execute("SET http_timeout=60")
    connection.execute("SET http_retries=2")
    return connection

In [10]:
def overture_copy_query(connection, query: str, parameters: list, destination: overture_Path) -> None:
    temporary = destination.with_suffix(destination.suffix + ".partial")
    try:
        connection.execute(
            f"COPY ({query}) TO ? (FORMAT PARQUET, COMPRESSION ZSTD)",
            [*parameters, str(temporary)],
        )
        temporary.replace(destination)
    finally:
        temporary.unlink(missing_ok=True)

In [11]:
def overture_collect_part(asset: dict, index: int, bbox: list, output: overture_Path, mode: str) -> dict:
    destination = output / "raw_shards" / f"part_{index:03d}.parquet"
    connection = overture_connect()
    try:
        if destination.exists():
            rows = connection.execute("SELECT count(*) FROM read_parquet(?)", [str(destination)]).fetchone()[0]
            overture_log(f"Resume part {index + 1}: {rows:,} records already present")
            return {"key": asset["key"], "local_file": str(destination), "rows": rows, "cached": True}
        # bbox predicates allow remote Parquet row-group pruning. Check true point
        # coordinates as well, since source bbox coordinates may be float-rounded.
        query = """
            SELECT * EXCLUDE(filename),
                   filename AS audit_source_file,
                   ? AS audit_release,
                   ? AS audit_scope,
                   ST_X(geometry) AS audit_lon,
                   ST_Y(geometry) AS audit_lat
            FROM read_parquet(?, filename=true, hive_partitioning=false)
            WHERE bbox.xmin BETWEEN ? AND ?
              AND bbox.ymin BETWEEN ? AND ?
              AND ST_X(geometry) BETWEEN ? AND ?
              AND ST_Y(geometry) BETWEEN ? AND ?
        """
        # Slightly widen the cheap bbox predicate so float precision cannot drop
        # boundary points; the geometry predicate applies the exact rectangle.
        west, south, east, north = bbox
        parameters = [overture_RELEASE, mode, asset["s3_https_url"], west - 1e-5, east + 1e-5,
                      south - 1e-5, north + 1e-5, west, east, south, north]
        started = overture_time.monotonic()
        for attempt in range(4):
            try:
                overture_copy_query(connection, query, parameters, destination)
                break
            except overture_duckdb.Error as exc:
                if attempt == 3:
                    raise
                overture_log(f"Part {index + 1} query retry {attempt + 1}/3: {str(exc)[:240]}")
                overture_time.sleep(2 ** attempt)
        rows = connection.execute("SELECT count(*) FROM read_parquet(?)", [str(destination)]).fetchone()[0]
        overture_log(f"Part {index + 1}: {rows:,} rows ({overture_time.monotonic() - started:.1f}s)")
        return {"key": asset["key"], "local_file": str(destination), "rows": rows, "cached": False}
    finally:
        connection.close()

In [12]:
def overture_build_local_outputs(output: overture_Path, export_csv: bool) -> dict:
    """Make explicit candidate/background roles without losing raw records."""
    connection = overture_connect()
    raw_glob = str(output / "raw_shards" / "*.parquet")
    try:
        connection.execute(f"CREATE VIEW raw_places AS SELECT * FROM read_parquet({overture_sql_literal(raw_glob)})")
        base_query = """
            SELECT id, names.primary AS name, basic_category,
                   taxonomy.primary AS category, taxonomy.hierarchy AS hierarchy,
                   taxonomy.alternates AS alternate_categories,
                   confidence, operating_status,
                   audit_lon AS lon, audit_lat AS lat,
                   addresses, sources, brand, websites, phones,
                   audit_source_file, audit_release, audit_scope,
                   coalesce(confidence >= 0.8, false)
                   AND trim(coalesce(names.primary, '')) <> ''
                   AND taxonomy.primary IS NOT NULL
                   AND coalesce(operating_status NOT IN
                       ('temporarily_closed', 'permanently_closed'), true)
                   AS passes_initial_quality_screen
            FROM raw_places
        """
        connection.execute(f"CREATE TEMP VIEW base_points AS {base_query}")
        category_expression = (
            f"coalesce(list_extract(hierarchy, 2) IN {overture_sql_list(overture_RETAIL_PARENT_ALLOW)}, false) "
            f"AND NOT coalesce(list_has_any(hierarchy, {overture_sql_list(overture_RETAIL_PATH_DENY)}), false)"
        )
        review_expression = f"""
            CASE WHEN category IN {overture_sql_list(overture_STRICT_TARGET_PROXY)} THEN 'strict_category_proxy'
                 WHEN category IN {overture_sql_list(overture_PENDING_CATEGORY_REVIEW)} THEN 'ambiguous_category'
                 WHEN regexp_matches(coalesce(name, ''),
                    '(?i)pop[ -]?mart|blind[ -]?box|collectibl|designer[ -]?toys')
                 THEN 'name_keyword_only' ELSE NULL END
        """
        points_query = f"""
            SELECT *, {category_expression} AS in_candidate_category_frame,
                   {review_expression} AS target_review_reason,
                   coalesce(category IN {overture_sql_list(overture_STRICT_TARGET_PROXY)}, false)
                   AS is_strict_target_proxy
            FROM base_points
        """
        overture_copy_query(connection, points_query, [], output / "points.parquet")
        connection.execute(f"CREATE VIEW points AS SELECT * FROM read_parquet({overture_sql_literal(str(output / 'points.parquet'))})")
        # Ambiguous/name-only matches are never silently treated as negative.
        candidates_query = """
            SELECT *, CASE WHEN is_strict_target_proxy THEN 1 ELSE 0 END AS label_proxy
            FROM points
            WHERE passes_initial_quality_screen AND in_candidate_category_frame
              AND (target_review_reason IS NULL OR is_strict_target_proxy)
        """
        review_query = "SELECT * FROM points WHERE target_review_reason IS NOT NULL"
        overture_copy_query(connection, candidates_query, [], output / "retail_candidates_unclipped.parquet")
        overture_copy_query(connection, review_query, [], output / "target_review_queue.parquet")
        if export_csv:
            for name in ("retail_candidates_unclipped", "target_review_queue"):
                connection.execute("""
                    COPY (
                        SELECT id, name, lon, lat, category, basic_category,
                               to_json(hierarchy) AS hierarchy_json,
                               confidence, operating_status,
                               to_json(addresses) AS addresses_json,
                               to_json(sources) AS sources_json,
                               audit_release, audit_scope, target_review_reason
                        FROM read_parquet(?)
                    ) TO ? (FORMAT CSV, HEADER TRUE)
                """, [str(output / f"{name}.parquet"), str(output / f"{name}.csv")])
        stats = dict(zip(
            ["bbox_place_rows", "unique_gers_ids", "shopping_rows",
             "passes_initial_quality_screen", "candidate_category_rows",
             "strict_proxy_rows", "target_review_rows"],
            connection.execute("""
                SELECT count(*), count(DISTINCT id),
                       count(*) FILTER (WHERE list_extract(hierarchy, 1)='shopping'),
                       count(*) FILTER (WHERE passes_initial_quality_screen),
                       count(*) FILTER (WHERE in_candidate_category_frame),
                       count(*) FILTER (WHERE is_strict_target_proxy),
                       count(*) FILTER (WHERE target_review_reason IS NOT NULL)
                FROM points
            """).fetchone(),
        ))
        stats["initial_candidate_rows_not_final_cleaned"] = connection.execute(
            "SELECT count(*) FROM read_parquet(?)", [str(output / "retail_candidates_unclipped.parquet")]
        ).fetchone()[0]
        stats["category_inventory"] = [
            {"category": row[0], "rows": row[1]}
            for row in connection.execute("SELECT category,count(*) FROM points GROUP BY 1 ORDER BY 2 DESC").fetchall()
        ]
        stats["operating_status_inventory"] = [
            {"status": row[0], "rows": row[1]}
            for row in connection.execute("SELECT operating_status,count(*) FROM points GROUP BY 1 ORDER BY 2 DESC").fetchall()
        ]
        # Fully resolved SQL makes local steps inspectable and reproducible.
        (output / "queries.sql").write_text(
            "-- The raw_glob path is local to this run.\n"
            f"CREATE VIEW raw_places AS SELECT * FROM read_parquet({overture_sql_literal(raw_glob)});\n\n"
            f"CREATE TEMP VIEW base_points AS {base_query};\n\n"
            f"CREATE VIEW points AS {points_query};\n\n"
            f"-- Candidate frame, not an approved/cleaned California sample:\n{candidates_query};\n\n"
            f"-- Manual label audit queue:\n{review_query};\n",
            encoding="utf-8",
        )
        return stats
    finally:
        connection.close()

In [13]:
def overture_parse_args() -> overture_argparse.Namespace:
    parser = overture_argparse.ArgumentParser(description=overture___doc__, formatter_class=overture_argparse.RawDescriptionHelpFormatter)
    scope = parser.add_mutually_exclusive_group()
    scope.add_argument("--sample", action="store_true", help="Small SF sample (also the default)")
    scope.add_argument("--full-ca", action="store_true", help="Explicitly enable the large California retrieval rectangle")
    parser.add_argument("--output-dir", type=overture_Path, help="Destination; do not reuse it for another scope/buffer")
    parser.add_argument("--buffer-meters", type=float, default=1000, help="Retrieval padding; default 1000m (not an analytical distance buffer)")
    parser.add_argument("--workers", type=int, default=4, help="Concurrent independent part downloads; default 4")
    parser.add_argument("--csv", action="store_true", help="Also export flat candidate/review CSVs; Parquet retains nested audit fields")
    parser.add_argument("--manifest-only", action="store_true", help="Verify mirrors/list assets without extracting any POIs")
    args = parser.parse_args()
    if not 1 <= args.workers <= 8:
        parser.error("--workers must be between 1 and 8")
    if not 0 <= args.buffer_meters <= 10000:
        parser.error("--buffer-meters must be between 0 and 10000")
    return args

In [14]:
def overture_main() -> int:
    args = overture_parse_args()
    mode = "california_bbox_unclipped" if args.full_ca else "sf_sample_bbox_only"
    base_bbox = overture_CA_RETRIEVAL_BBOX if args.full_ca else overture_SF_SAMPLE_BBOX
    query_bbox = overture_expand_bbox(base_bbox, args.buffer_meters)
    output = (args.output_dir or overture_Path("data") / ("overture_ca_bbox" if args.full_ca else "overture_sf_sample")).resolve()
    output.mkdir(parents=True, exist_ok=True)
    (output / "raw_shards").mkdir(exist_ok=True)
    config = {"release": overture_RELEASE, "mode": mode, "base_bbox_wsen": list(base_bbox),
              "query_bbox_wsen": query_bbox, "retrieval_buffer_meters": args.buffer_meters,
              "raw_columns": "all original physical columns plus audit fields",
              "state_polygon_clipped": False}
    config_path = output / "run_config.json"
    if config_path.exists() and overture_json.loads(config_path.read_text()) != config:
        raise RuntimeError("Output directory belongs to another configuration. Choose another --output-dir.")
    config_path.write_text(overture_json.dumps(config, indent=2), encoding="utf-8")
    overture_log(f"Scope={mode}; release={overture_RELEASE}; query_bbox={query_bbox}")
    overture_log("This is a bbox extraction, not a cleaned California dataset or a 50k-row compliance count.")

    assets = overture_discover_manifest()
    catalog_bytes, _ = overture_fetch(overture_STAC_URL)
    overture_json.loads(catalog_bytes)  # Fail if an error page was returned in place of JSON.
    (output / "release_catalog.json").write_bytes(catalog_bytes)
    mirror_checks = []
    for field in ("s3_https_url", "azure_https_url"):
        try:
            _, headers = overture_fetch(assets[0][field], method="HEAD")
            observed_length = next((v for k, v in headers.items() if k.lower() == "content-length"), None)
            if observed_length is not None and int(observed_length) != assets[0]["size_bytes"]:
                raise RuntimeError("Mirror size differs from S3 manifest")
            mirror_checks.append({"url": assets[0][field], "status": headers["status"], "content_length": observed_length})
        except RuntimeError as exc:
            if field == "s3_https_url":
                raise
            mirror_checks.append({"url": assets[0][field], "error": str(exc)})
            overture_log("Azure verification failed; collection continues through the verified S3 endpoint.")
    manifest = {"release": overture_RELEASE, "prefix": overture_PREFIX, "stac_catalog_url": overture_STAC_URL,
                "verified_at_utc": overture_datetime.now(overture_timezone.utc).isoformat(),
                "asset_count": len(assets), "assets": assets, "mirror_checks": mirror_checks}
    (output / "manifest.json").write_text(overture_json.dumps(manifest, indent=2), encoding="utf-8")
    overture_log(f"Manifest: {len(assets)} remote Parquet parts; only bbox-matching rows will be written.")
    if args.manifest_only:
        return 0

    connection = overture_connect()
    try:
        schema = connection.execute("DESCRIBE SELECT * FROM read_parquet(?)", [assets[0]["s3_https_url"]]).fetchall()
        names = {column[0] for column in schema}
        required = {"id", "geometry", "bbox", "names", "sources", "taxonomy", "basic_category", "operating_status", "confidence"}
        if not required <= names:
            raise RuntimeError(f"Frozen release schema is missing {sorted(required - names)}; inspect before changing code")
        (output / "source_schema.json").write_text(overture_json.dumps(schema, indent=2), encoding="utf-8")
    finally:
        connection.close()
    with overture_ThreadPoolExecutor(max_workers=args.workers) as pool:
        parts = list(pool.map(lambda item: overture_collect_part(item[1], item[0], query_bbox, output, mode), enumerate(assets)))
    stats = overture_build_local_outputs(output, args.csv)
    summary = {**config, "completed_at_utc": overture_datetime.now(overture_timezone.utc).isoformat(),
               "duckdb_version": overture_duckdb.__version__, "concurrent_workers": args.workers, "parts": parts, "counts": stats,
               "raw_schema_fingerprint_sha256": overture_hashlib.sha256(overture_json.dumps(schema).encode()).hexdigest(),
               "candidate_parent_allow": overture_RETAIL_PARENT_ALLOW,
               "candidate_path_deny": overture_RETAIL_PATH_DENY,
               "status": "REAL SF SAMPLE ONLY" if not args.full_ca else "REAL CALIFORNIA RETRIEVAL BBOX ONLY",
               "unfinished": ["exact TIGER California polygon clip for candidate rows",
                              "retain adjacent-state buffer POIs as commercial context only",
                              "entity/near-duplicate resolution and storefront verification",
                              "manual target taxonomy/name audit",
                              "Census feature joins and final cleaned model row count"],
               "not_business_success_labels": True}
    (output / "run_summary.json").write_text(overture_json.dumps(summary, indent=2), encoding="utf-8")
    overture_log(overture_json.dumps({"output_dir": str(output), "mode": mode, "counts": {k: v for k, v in stats.items() if not isinstance(v, list)}}, indent=2))
    return 0

In [15]:
overture_out=RUN_ROOT/'data/raw/overture_ca'
(overture_out/'raw_shards').mkdir(parents=True)
shards=sorted((SOURCE_ROOT/'data/raw/overture_ca/raw_shards').glob('*.parquet'))
assert len(shards)==16,'需要固定快照的16个原始区域分片（其中部分可为0行）。'
for source in shards:
    destination=overture_out/'raw_shards'/source.name
    try:os.link(source,destination)
    except OSError:shutil.copy2(source,destination)
overture_stats=overture_build_local_outputs(overture_out,False)
print({k:overture_stats[k] for k in ['bbox_place_rows','unique_gers_ids','shopping_rows','passes_initial_quality_screen']})
assert overture_stats['bbox_place_rows']==overture_stats['unique_gers_ids']
gc.collect()

{'bbox_place_rows': 2375195, 'unique_gers_ids': 2375195, 'shopping_rows': 297412, 'passes_initial_quality_screen': 1429060}


79

## 2. OSM：从PBF提取对象与坐标

分别提取玩具／收藏候选和全部shop背景。保留node／way／relation类型、原始tags和来源，组装几何；缺坐标、开放线形和生命周期信息显式标记，不能把shop标签一概当商品零售。

函数／常量加阶段前缀，避免同一内核中的同名变量相互覆盖。

In [16]:
# Source: pipeline/osm_extract.py; SHA256: 2b637bcee6a7c345f066ef5925f421fa8bf242f7add3ab199afd80d0d0f3ab0b
osm___file__ = str(RUN_ROOT / 'pipeline/osm_extract.py')
osm___doc__ = "Extract reviewable OSM retail candidates, preserving nodes/ways/relations.\n\nInstall: python -m pip install 'osmium>=4,<5' 'shapely>=2,<3'\nThe source is scanned for candidate tags, then BackReferenceWriter includes\ntheir referenced geometry objects. Areas are assembled on that small subset.\nEvery candidate survives in the CSV; unresolved geometry is explicitly flagged."
#!/usr/bin/env python3
"""Extract reviewable OSM retail candidates, preserving nodes/ways/relations.

Install: python -m pip install 'osmium>=4,<5' 'shapely>=2,<3'
The source is scanned for candidate tags, then BackReferenceWriter includes
their referenced geometry objects. Areas are assembled on that small subset.
Every candidate survives in the CSV; unresolved geometry is explicitly flagged.
"""


import argparse as osm_argparse
from collections import Counter as osm_Counter
from datetime import datetime as osm_datetime, timezone as osm_timezone
import json as osm_json
import math as osm_math
import os as osm_os
from pathlib import Path as osm_Path
import re as osm_re
import sys as osm_sys
import tempfile as osm_tempfile
import csv as osm_csv

osm_DEFAULT_NAME_REGEX = (
    r"pop[ -]?mart|blind[ -]?box|collectibl|collectabl|designer[ -]?toys?|"
    r"pok[eé]mon|trading[ -]?cards?|\bTCG\b|gundam|sanrio|\banime\b|\bhobby\b"
)
osm_SHOP_VALUES = {"toys", "collector", "model", "games"}
osm_SHOP_KEYS = ("shop", "disused:shop", "abandoned:shop", "was:shop", "closed:shop")
osm_FIELDS = ["osm_type", "osm_id", "osm_object_key", "name", "shop", "brand", "operator", "website", "address", "city", "postcode", "opening_hours", "check_date", "longitude", "latitude", "coordinate_status", "geometry_method", "match_reasons_json", "tags_json", "relation_members_json", "source_json"]

In [17]:
def osm_encode(value) -> str:
    return osm_json.dumps(value, ensure_ascii=False, separators=(",", ":"))

In [18]:
def osm_main() -> int:
    parser = osm_argparse.ArgumentParser(description=osm___doc__)
    parser.add_argument("--input", type=osm_Path, required=True, help="An ordinary OSM snapshot PBF/XML, not a history or change file.")
    parser.add_argument("--output", type=osm_Path, default=osm_Path("data/processed/osm_toy_candidates.csv"))
    parser.add_argument("--mode", choices=["toy-candidates", "all-shops"], default="toy-candidates", help="all-shops also retains other shop-tagged objects for an optional background cohort; neither mode certifies labels.")
    parser.add_argument("--name-regex", default=osm_DEFAULT_NAME_REGEX, help="Case-insensitive Python regex; replace explicitly to expand a reviewed brand list.")
    parser.add_argument("--relation-depth", type=int, default=3, help="Maximum nested-relation completion depth; each extra depth can require another source scan.")
    parser.add_argument("--source-url", help="Optional original source URL, if the input has no download manifest.")
    parser.add_argument("--overwrite", action="store_true")
    args = parser.parse_args()
    if args.relation_depth < 0:
        parser.error("relation-depth must be nonnegative")
    try:
        import osmium as osm_osmium
        from shapely import from_wkb as osm_from_wkb
        from shapely.errors import ShapelyError as osm_ShapelyError
        from shapely.geometry import LineString as osm_LineString, Polygon as osm_Polygon
    except ImportError as error:
        parser.exit(1, f"Missing dependency: {error}. Install osmium>=4,<5 and shapely>=2,<3.\n")
    try:
        pattern = osm_re.compile(args.name_regex, osm_re.IGNORECASE)
    except osm_re.error as error:
        parser.error(f"Invalid --name-regex: {error}")
    source = args.input.resolve()
    if not source.is_file():
        parser.error(f"Input file not found: {source}")
    destination = args.output.resolve()
    destination.parent.mkdir(parents=True, exist_ok=True)
    unresolved_path = destination.with_name(destination.stem + "_unresolved.csv")
    summary_path = destination.with_name(destination.stem + "_manifest.json")
    for path in [destination, unresolved_path, summary_path]:
        if path.exists() and not args.overwrite:
            parser.error(f"Output exists: {path}; use --overwrite deliberately")
    downloaded = source.with_name(source.name + ".download.json")
    download_manifest = osm_json.loads(downloaded.read_text()) if downloaded.exists() else {}
    with osm_osmium.io.Reader(str(source)) as reader:
        header = reader.header()
        replication_time = header.get("osmosis_replication_timestamp") or None
    source_info = {
        "provider": "OpenStreetMap", "input_file": str(source),
        "source_url": args.source_url or download_manifest.get("url"),
        "input_sha256_from_download_manifest": download_manifest.get("sha256"),
        "osm_replication_timestamp": replication_time,
    }
    candidates: dict[tuple[str, int], dict] = {}
    counts = osm_Counter()
    geometry_errors = osm_Counter()
    type_names = {"n": "node", "w": "way", "r": "relation"}

    def set_point(record: dict, x: float, y: float, method: str) -> None:
        if osm_math.isfinite(x) and osm_math.isfinite(y) and -180 <= x <= 180 and -90 <= y <= 90:
            record.update(longitude=x, latitude=y, coordinate_status="located", geometry_method=method)

    with osm_tempfile.TemporaryDirectory(prefix=".osm_extract_", dir=destination.parent) as work:
        subset = osm_Path(work) / "candidates_with_references.osm.pbf"
        print("Scanning source tags and collecting candidate objects…", file=osm_sys.stderr)
        with osm_osmium.BackReferenceWriter(str(subset), ref_src=str(source), remove_tags=True, relation_depth=args.relation_depth) as writer:
            processor = osm_osmium.FileProcessor(str(source)).with_filter(osm_osmium.filter.KeyFilter(*osm_SHOP_KEYS, "name"))
            seen_tagged = 0
            for obj in processor:
                seen_tagged += 1
                if seen_tagged % 100000 == 0:
                    print(f"Scanned {seen_tagged:,} shop/name-tagged objects; retained {len(candidates):,} candidates", file=osm_sys.stderr, flush=True)
                kind = obj.type_str()
                tags = dict(obj.tags)
                reasons = []
                for shop_key in osm_SHOP_KEYS:
                    if tags.get(shop_key) in osm_SHOP_VALUES:
                        reasons.append("candidate_shop_tag:" + shop_key + "=" + tags[shop_key])
                if pattern.search(tags.get("name", "")):
                    reasons.append("candidate_name_regex")
                if args.mode == "all-shops" and "shop" in tags:
                    reasons.append("background_shop_tag")
                if not reasons:
                    continue
                key = (kind, obj.id)
                if key in candidates:
                    raise ValueError(f"Duplicate OSM {key}; use a single snapshot, not a history file")
                members = [{"type": member.type, "ref": member.ref, "role": member.role} for member in obj.members] if kind == "r" else []
                metadata = {
                    **source_info, "osm_url": f"https://www.openstreetmap.org/{type_names[kind]}/{obj.id}",
                    "osm_version": obj.version, "osm_object_timestamp": str(obj.timestamp),
                    "candidate_only": True,
                }
                candidates[key] = {
                    "osm_type": type_names[kind], "osm_id": obj.id,
                    "osm_object_key": f"{type_names[kind]}/{obj.id}",
                    "name": tags.get("name", ""), "shop": tags.get("shop", ""),
                    "brand": tags.get("brand", ""), "operator": tags.get("operator", ""),
                    "website": tags.get("website", tags.get("contact:website", "")),
                    "address": tags.get("addr:full", " ".join(filter(None, [tags.get("addr:housenumber"), tags.get("addr:street")]))),
                    "city": tags.get("addr:city", ""), "postcode": tags.get("addr:postcode", ""),
                    "opening_hours": tags.get("opening_hours", ""), "check_date": tags.get("check_date", ""),
                    "longitude": None, "latitude": None,
                    "coordinate_status": "unresolved", "geometry_method": "not_yet_resolved",
                    "match_reasons_json": osm_encode(reasons), "tags_json": osm_encode(tags),
                    "relation_members_json": osm_encode(members), "source_json": osm_encode(metadata),
                }
                writer.add(obj)
                counts[type_names[kind]] += 1
        print(f"Resolving geometry for {len(candidates)} candidates (including relation areas)…", file=osm_sys.stderr)
        factory = osm_osmium.geom.WKBFactory()
        processor = osm_osmium.FileProcessor(str(subset)).with_locations().with_areas()
        for obj in processor:
            kind = obj.type_str()
            if kind == "a":
                key = ("w" if obj.from_way() else "r", obj.orig_id())
            else:
                key = (kind, obj.id)
            record = candidates.get(key)
            if record is None:
                continue
            try:
                if kind == "n":
                    if obj.location.valid():
                        set_point(record, obj.lon, obj.lat, "osm_node")
                    else:
                        record["geometry_method"] = "node_missing_location"
                elif kind == "w":
                    locations = [node.location for node in obj.nodes]
                    if not locations or not all(location.valid() for location in locations):
                        record["geometry_method"] = "way_missing_referenced_node"
                        continue
                    coords = [(location.lon, location.lat) for location in locations]
                    if len(coords) >= 4 and obj.is_closed():
                        polygon = osm_Polygon(coords)
                        if polygon.is_valid and not polygon.is_empty:
                            point = polygon.representative_point()
                            set_point(record, point.x, point.y, "closed_way_representative_point")
                        else:
                            record["geometry_method"] = "invalid_closed_way_polygon"
                    elif len(coords) >= 2:
                        point = osm_LineString(coords).interpolate(0.5, normalized=True)
                        set_point(record, point.x, point.y, "open_way_line_midpoint_review_required")
                    else:
                        record["geometry_method"] = "way_insufficient_nodes"
                elif kind == "r":
                    # Areas can be emitted before the original relation callback.
                    # Preserve the already assembled representative-point method.
                    if record["coordinate_status"] != "located":
                        tags = osm_json.loads(record["tags_json"])
                        record["geometry_method"] = "awaiting_relation_area" if tags.get("type") in {"multipolygon", "boundary"} else "non_area_relation_requires_manual_geometry"
                elif kind == "a":
                    polygon = osm_from_wkb(bytes.fromhex(factory.create_multipolygon(obj)))
                    if polygon.is_empty or not polygon.is_valid:
                        raise ValueError("Empty or invalid assembled area")
                    point = polygon.representative_point()
                    set_point(record, point.x, point.y, "assembled_area_representative_point")
            except (RuntimeError, ValueError, osm_ShapelyError) as error:
                geometry_errors[type(error).__name__ + ": " + str(error)[:160]] += 1
                if record["coordinate_status"] != "located":
                    record["geometry_method"] = "geometry_error:" + type(error).__name__
        for record in candidates.values():
            if record["geometry_method"] == "awaiting_relation_area":
                record["geometry_method"] = "relation_area_not_assembled_missing_or_invalid_members"
        records = [candidates[key] for key in sorted(candidates)]
        unresolved = [row for row in records if row["coordinate_status"] != "located"]
        for filename, rows in [(destination.name, records), (unresolved_path.name, unresolved)]:
            staged = osm_Path(work) / filename
            with staged.open("w", encoding="utf-8", newline="") as stream:
                csv_writer = osm_csv.DictWriter(stream, fieldnames=osm_FIELDS)
                csv_writer.writeheader()
                csv_writer.writerows(rows)
                stream.flush()
                osm_os.fsync(stream.fileno())
        summary = {
            **source_info, "input_bytes": source.stat().st_size,
            "mode": args.mode, "candidate_shop_values": sorted(osm_SHOP_VALUES), "candidate_shop_keys": list(osm_SHOP_KEYS), "name_regex": args.name_regex,
            "relation_depth": args.relation_depth, "candidate_records": len(records),
            "candidate_osm_types": dict(counts), "located_records": len(records) - len(unresolved),
            "unresolved_records": len(unresolved),
            "geometry_methods": dict(osm_Counter(row["geometry_method"] for row in records)),
            "geometry_errors": dict(geometry_errors),
            "generated_at_utc": osm_datetime.now(osm_timezone.utc).isoformat(),
            "notes": [
                "Records are candidate OSM objects, not verified unique physical stores or complete market coverage.",
                "OSM type+ID is unique here; node/way/relation records can still describe the same physical store.",
                "Polygon points are inside-geometry representative points, not store entrances or geographic centroids.",
                "Open-way midpoints require review; non-area or unassembled relations remain in the unresolved CSV.",
                "Node geometry is EPSG:4326. No distance or area is computed in longitude/latitude degrees.",
                "Geofabrik extract limits do not replace an exact California county/tract spatial boundary check.",
            ],
        }
        (osm_Path(work) / summary_path.name).write_text(osm_json.dumps(summary, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
        for target in [destination, unresolved_path, summary_path]:
            osm_os.replace(osm_Path(work) / target.name, target)
    print(osm_json.dumps(summary, ensure_ascii=False, indent=2))
    return 0

In [19]:
osm_input=SOURCE_ROOT/'data/raw/osm/california-261024.osm.pbf'
osm_dir=RUN_ROOT/'data/interim/osm';osm_dir.mkdir(parents=True,exist_ok=True)
for mode,filename in [('toy-candidates','toy_candidates.csv'),('all-shops','all_shop_candidates.csv')]:
    print('Extracting OSM:',mode,flush=True)
    with notebook_args('--input',osm_input,'--output',osm_dir/filename,'--mode',mode):
        assert osm_main()==0
gc.collect()

Extracting OSM: toy-candidates


Scanning source tags and collecting candidate objects…


Scanned 100,000 shop/name-tagged objects; retained 105 candidates


Scanned 200,000 shop/name-tagged objects; retained 235 candidates


Scanned 300,000 shop/name-tagged objects; retained 539 candidates


...

Scanned 1,800,000 shop/name-tagged objects; retained 83,936 candidates


Scanned 1,900,000 shop/name-tagged objects; retained 85,624 candidates


Resolving geometry for 86151 candidates (including relation areas)…


{
  "provider": "OpenStreetMap",
  "input_file": "<PROJECT_ROOT>/data/raw/osm/california-261024.osm.pbf",
  "source_url": "https://download.geofabrik.de/north-america/us/california-261024.osm.pbf",
  "input_sha256_from_download_manifest": "74dd4c0e3547668cfd38cfc73b5595f7054af2accc37d67f9ddf852a9c330061",
  "osm_replication_timestamp": "2026-10-24T20:20:51Z",
  "input_bytes": 1333570033,
  "mode": "all-shops",
  "candidate_shop_values": [
    "collector",
    "games",
    "model",
    "toys"
  ],
  "candidate_shop_keys": [
    "shop",
    "disused:shop",
    "abandoned:shop",
    "was:shop",
    "closed:shop"
  ],
  "name_regex": "pop[ -]?mart|blind[ -]?box|collectibl|collectabl|designer[ -]?toys?|pok[eé]mon|trading[ -]?cards?|\\bTCG\\b|gundam|sanrio|\\banime\\b|\\bhobby\\b",
  "relation_depth": 3,
  "candidate_records": 86151,
  "candidate_osm_types": {
    "node": 60943,
    "way": 24971,
    "relation": 237
  },
  "located_records": 86141,
  "unresolved_records": 10,
  "geometry_met

51

## 3. OSM：结构质量与停用／服务标签

将全shop提取结果转换为Parquet并统计结构完整性、服务标签、停用状态和未解析几何。这些标签是数据库证据，不等于当前营业的现场核实。

函数／常量加阶段前缀，避免同一内核中的同名变量相互覆盖。

In [20]:
# Source: validation/describe_osm_allshops.py; SHA256: a75fb21577a9f873af2068761a0e300a5b2bdaf748b049ccb91e661201c1c663
osmqa___file__ = str(RUN_ROOT / 'validation/describe_osm_allshops.py')
osmqa___doc__ = 'Describe all OSM shop candidates without declaring them verified retail.'
"""Describe all OSM shop candidates without declaring them verified retail."""
from pathlib import Path as osmqa_Path
from collections import Counter as osmqa_Counter, defaultdict as osmqa_defaultdict
from datetime import datetime as osmqa_datetime, timezone as osmqa_timezone
import argparse as osmqa_argparse
import csv as osmqa_csv
import hashlib as osmqa_hashlib
import json as osmqa_json
import math as osmqa_math
import pyarrow as osmqa_pa
import pyarrow.parquet as osmqa_pq

osmqa_INACTIVE_VALUES={'vacant','closed','no','disused','abandoned','empty'}
osmqa_SERVICE_HINTS={'hairdresser','beauty','car_repair','laundry','dry_cleaning','tailor','massage','tattoo','travel_agency','money_lender','funeral_directors','copyshop','bookmaker','lottery'}
osmqa_LIFE_KEYS=('disused:shop','abandoned:shop','was:shop','closed:shop')

In [21]:
def osmqa_main():
 p=osmqa_argparse.ArgumentParser(description=osmqa___doc__);p.add_argument('--input',type=osmqa_Path,default=osmqa_Path('data/interim/osm/all_shop_candidates.csv'));p.add_argument('--output',type=osmqa_Path,default=osmqa_Path('data/interim/osm/all_shop_candidates_quality.json'));p.add_argument('--parquet',type=osmqa_Path,default=osmqa_Path('data/interim/osm/all_shop_candidates.parquet'));a=p.parse_args()
 counts=osmqa_Counter();shop=osmqa_Counter();types=osmqa_Counter();methods=osmqa_Counter();inactive=osmqa_Counter();names=osmqa_Counter();seen=set();dups=[];unresolved=[];openways=[];service=osmqa_Counter();coord_digest=osmqa_hashlib.sha256();data_digest=osmqa_hashlib.sha256();batch=[]
 with a.input.open(newline='',encoding='utf-8') as f:
  reader=osmqa_csv.DictReader(f);fields=list(reader.fieldnames)
  extra=[osmqa_pa.field('shop_tag_present',osmqa_pa.bool_()),osmqa_pa.field('active_tag_candidate',osmqa_pa.bool_()),osmqa_pa.field('explicit_inactive_object_tag',osmqa_pa.bool_()),osmqa_pa.field('lifecycle_shop_keys_present',osmqa_pa.string()),osmqa_pa.field('service_shop_tag_hint',osmqa_pa.bool_()),osmqa_pa.field('geometry_requires_review',osmqa_pa.bool_()),osmqa_pa.field('name_present',osmqa_pa.bool_()),osmqa_pa.field('match_ready_active_tag_candidate',osmqa_pa.bool_()),osmqa_pa.field('phone_from_tags',osmqa_pa.string()),osmqa_pa.field('unit_from_tags',osmqa_pa.string())]
  schema=osmqa_pa.schema([osmqa_pa.field(k,osmqa_pa.float64() if k in('longitude','latitude') else osmqa_pa.string())for k in fields]+extra)
  tmp=a.parquet.with_suffix('.parquet.partial');writer=osmqa_pq.ParquetWriter(tmp,schema,compression='zstd')
  try:
   for row in reader:
    counts['records']+=1;key=row['osm_object_key'];tag=osmqa_json.loads(row['tags_json']);s=tag.get('shop','').strip().casefold();shop[s or'<missing>']+=1;types[row['osm_type']]+=1;methods[row['geometry_method']]+=1
    if key in seen:dups.append(key)
    seen.add(key)
    shop_present=bool(s);explicit_off=s in osmqa_INACTIVE_VALUES or any(tag.get(k,'').casefold()=='yes'for k in('disused','abandoned','closed'))or tag.get('opening_hours','').strip().casefold()=='closed'
    if s in osmqa_INACTIVE_VALUES:inactive[s]+=1
    life=[k+'='+tag[k] for k in osmqa_LIFE_KEYS if k in tag]
    active=shop_present and not explicit_off
    name_present=bool(row['name'].strip());geometry_review=row['coordinate_status']!='located'or row['geometry_method']=='open_way_line_midpoint_review_required'
    if name_present:names[row['name']]+=1
    if s in osmqa_SERVICE_HINTS:service[s]+=1
    for k,v in {'shop_tag_present':shop_present,'active_tag_candidate':active,'explicit_inactive_object_tag':explicit_off,'lifecycle_shop_tag_records':bool(life),'active_with_lifecycle_shop_tag':active and bool(life),'name_missing':not name_present,'unresolved_geometry':row['coordinate_status']!='located','geometry_review':geometry_review,'service_shop_tag_hint':s in osmqa_SERVICE_HINTS,'website_present':bool(row['website']),'phone_present':bool(tag.get('phone',tag.get('contact:phone',''))),'check_date_present':bool(row['check_date']),'mixed_shop_value':';'in s}.items():counts[k]+=int(v)
    ready=active and name_present and not geometry_review;counts['match_ready_active_tag_candidate']+=int(ready)
    if row['coordinate_status']!='located':unresolved.append({'osm_object_key':key,'name':row['name'],'shop':s,'geometry_method':row['geometry_method'],'relation_members_json':row['relation_members_json'],'tags':tag})
    if row['geometry_method']=='open_way_line_midpoint_review_required':openways.append({'osm_object_key':key,'name':row['name'],'shop':s,'tags':tag})
    data_digest.update((osmqa_json.dumps({k:v for k,v in row.items()if k!='geometry_method'},sort_keys=True,ensure_ascii=False,separators=(',',':'))+'\n').encode())
    coord_digest.update((key+'|'+row['longitude']+'|'+row['latitude']+'|'+row['coordinate_status']+'\n').encode())
    out=dict(row)
    for k in('longitude','latitude'):out[k]=float(row[k])if row[k]else None
    out.update(shop_tag_present=shop_present,active_tag_candidate=active,explicit_inactive_object_tag=explicit_off,lifecycle_shop_keys_present='|'.join(life),service_shop_tag_hint=s in osmqa_SERVICE_HINTS,geometry_requires_review=geometry_review,name_present=name_present,match_ready_active_tag_candidate=ready,phone_from_tags=tag.get('phone',tag.get('contact:phone','')),unit_from_tags=tag.get('addr:unit',''))
    batch.append(out)
    if len(batch)>=5000:writer.write_table(osmqa_pa.Table.from_pylist(batch,schema=schema));batch=[]
   if batch:writer.write_table(osmqa_pa.Table.from_pylist(batch,schema=schema))
  finally:writer.close()
 tmp.replace(a.parquet)
 assert not dups,dups
 assert counts['records']==len(seen)
 result={'complete':True,'created_at_utc':osmqa_datetime.now(osmqa_timezone.utc).isoformat(),'source_csv':str(a.input.resolve()),'parquet_with_review_flags':str(a.parquet.resolve()),'source_timestamp':'2026-10-24T20:20:51Z','counts':dict(counts),'unique_type_id':len(seen),'duplicate_type_id':len(dups),'osm_types':dict(types),'geometry_methods':dict(methods),'shop_value_counts':[{'shop':k,'records':v}for k,v in shop.most_common()],'explicit_inactive_shop_values':dict(inactive),'service_hint_tag_counts':dict(service),'unresolved_objects':unresolved,'open_way_objects':openways,'frequent_names':[{'name':k,'records':v}for k,v in names.most_common(25)],'fields_except_geometry_method_sha256':data_digest.hexdigest(),'coordinates_and_status_sha256':coord_digest.hexdigest(),'active_rule':'nonempty shop tag, shop value not vacant/closed/no/disused/abandoned/empty, and no explicit disused=yes/abandoned=yes/closed=yes/opening_hours=closed. This is database tag evidence, not verified current business operation.','service_hints_only':sorted(osmqa_SERVICE_HINTS),'limits':['all-shops extraction also retains target-name and lifecycle candidates without an active shop tag; keep the raw records, filter using explicit flags downstream.','shop=* also describes consumer services such as hairdressers and car repair. An OSM tag match cannot by itself certify discretionary goods retail or manual verification.','A lifecycle-specific old shop tag can coexist with a different active shop tag; preserve it for review rather than infer the current business closed.','No entity linkage, California official-polygon clipping, master modification or target-label verification is performed here.','Source type+ID uniqueness does not establish distinct physical stores. Unresolved objects remain present with null coordinates.']}
 a.output.write_text(osmqa_json.dumps(result,ensure_ascii=False,indent=2)+'\n')
 print(osmqa_json.dumps({k:v for k,v in result.items()if k in('counts','unique_type_id','duplicate_type_id','osm_types','geometry_methods','explicit_inactive_shop_values','service_hint_tag_counts')},ensure_ascii=False,indent=2))
 fix_path=a.input.with_name('all_shop_relation_metadata_fix.json')
 if fix_path.exists():
  fix=osmqa_json.loads(fix_path.read_text());fix['fields_except_geometry_method_sha256_v2']=data_digest.hexdigest();fix['non_method_fields_identical_v1_v2']=fix['fields_except_geometry_method_sha256_v1']==data_digest.hexdigest();fix['fixed_located_relation_methods']=sum(1 for x in fix['located_relation_wrong_method_records']);fix['v2_geometry_methods']=dict(methods);fix['v2_unresolved_records']=len(unresolved);fix_path.write_text(osmqa_json.dumps(fix,ensure_ascii=False,indent=2)+'\n');assert fix['non_method_fields_identical_v1_v2']

In [22]:
with notebook_args('--input',osm_dir/'all_shop_candidates.csv',
 '--output',osm_dir/'all_shop_candidates_quality.json','--parquet',osm_dir/'all_shop_candidates.parquet'):
    osmqa_main()
assert json.loads((osm_dir/'all_shop_candidates_quality.json').read_text())['complete']
gc.collect()

{
  "counts": {
    "records": 86151,
    "shop_tag_present": 86122,
    "active_tag_candidate": 84242,
    "explicit_inactive_object_tag": 1880,
    "lifecycle_shop_tag_records": 214,
    "active_with_lifecycle_shop_tag": 120,
    "name_missing": 9216,
    "unresolved_geometry": 10,
    "geometry_review": 18,
    "service_shop_tag_hint": 20805,
    "website_present": 24010,
    "phone_present": 23906,
    "check_date_present": 11852,
    "mixed_shop_value": 136,
    "match_ready_active_tag_candidate": 76791
  },
  "unique_type_id": 86151,
  "duplicate_type_id": 0,
  "osm_types": {
    "node": 60943,
    "relation": 237,
    "way": 24971
  },
  "geometry_methods": {
    "osm_node": 60943,
    "assembled_area_representative_point": 25190,
    "non_area_relation_requires_manual_geometry": 4,
    "relation_area_not_assembled_missing_or_invalid_members": 6,
    "open_way_line_midpoint_review_required": 8
  },
  "explicit_inactive_shop_values": {
    "vacant": 1855,
    "no": 4,
    "abando

28

## 4. ACS／TIGER：缺失值、比例、边界与GEOID

读取B01001、B19013、B14001、B11001全部估计值和MOE；将负数缺失码转NULL，分母为0时保留NULL。读取TIGER ZIP，保留11位GEOID，生成EPSG4326／3310边界、人口密度与年龄／家庭比例。全部9,129个Tract和69个原始MOE保留。

函数／常量加阶段前缀，避免同一内核中的同名变量相互覆盖。

In [23]:
# Source: pipeline/prepare_census.py; SHA256: 7af704832734cea503aabeb03e0746e4ea9657fa370714ac7469f66fface4c70
census___file__ = str(RUN_ROOT / 'pipeline/prepare_census.py')
census___doc__ = "Prepare canonical California ACS/TIGER inputs from already acquired files.\n\nRun with the project's existing DuckDB + spatial + PyArrow environment:\n  validation/.venv/bin/python pipeline/prepare_census.py\n\nOverride --acs-raw-dir and --tract-zip to reproduce from another local download.\nThe four source DATs and the ZIP are archived by hard link (copy fallback).\nOnly missing, small official variable dictionaries may require network access.\nNo Census API key, pandas or GeoPandas is required."
#!/usr/bin/env python3
"""Prepare canonical California ACS/TIGER inputs from already acquired files.

Run with the project's existing DuckDB + spatial + PyArrow environment:
  validation/.venv/bin/python pipeline/prepare_census.py

Override --acs-raw-dir and --tract-zip to reproduce from another local download.
The four source DATs and the ZIP are archived by hard link (copy fallback).
Only missing, small official variable dictionaries may require network access.
No Census API key, pandas or GeoPandas is required.
"""



import argparse as census_argparse
import collections as census_collections
import csv as census_csv
import datetime as census_dt
import hashlib as census_hashlib
import json as census_json
import math as census_math
import os as census_os
from pathlib import Path as census_Path
import re as census_re
import shutil as census_shutil
import sys as census_sys
import time as census_time
import urllib.error
import urllib as census_urllib
import urllib.request
import urllib as census_urllib
import zipfile as census_zipfile

import duckdb as census_duckdb
import pyarrow as census_pa
import pyarrow.parquet as census_pq


census_TABLES = ["B01001", "B19013", "B14001", "B11001"]
census_SUMMARY_BASE = "https://www2.census.gov/programs-surveys/acs/summary_file/2024/table-based-SF/data/5YRData/"
census_TRACT_URL = "https://www2.census.gov/geo/tiger/TIGER2024/TRACT/tl_2024_06_tract.zip"
census_EXPECTED_ROWS = 9129
census_BAY_COUNTIES = ["001", "013", "041", "055", "075", "081", "085", "095", "097"]
census_CHILD_LINES = list(range(3, 7)) + list(range(27, 31))
census_YOUNG_LINES = list(range(7, 13)) + list(range(31, 37))

In [24]:
def census_utc_now():
    return census_dt.datetime.now(census_dt.timezone.utc).isoformat()

In [25]:
def census_sha256(path):
    result = census_hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            result.update(chunk)
    return result.hexdigest()

In [26]:
def census_relative(path, root):
    try:
        return str(path.resolve().relative_to(root))
    except ValueError:
        return str(path.resolve())

In [27]:
def census_resolve_input(value, root):
    value = census_Path(value)
    return (root / value).resolve() if not value.is_absolute() else value.resolve()

In [28]:
def census_atomic_json(path, value):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + f".{census_os.getpid()}.part")
    try:
        temporary.write_text(census_json.dumps(value, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
        census_os.replace(temporary, path)
    finally:
        temporary.unlink(missing_ok=True)

In [29]:
def census_archive_file(source, destination, root, mode):
    if not source.is_file():
        raise FileNotFoundError(f"Required existing input not found: {census_relative(source, root)}")
    destination.parent.mkdir(parents=True, exist_ok=True)
    checksum = census_sha256(source)
    if destination.exists():
        if census_sha256(destination) != checksum:
            raise ValueError(f"Existing canonical input differs: {census_relative(destination, root)}; refusing to overwrite it.")
        storage = ("in_place_hardlink" if destination.stat().st_nlink > 1 else "in_place") if source.resolve() == destination.resolve() else "existing_identical"
    else:
        temporary = destination.with_name(destination.name + f".{census_os.getpid()}.part")
        try:
            storage = "copy"
            if mode == "hardlink":
                try:
                    census_os.link(source, temporary)
                    storage = "hardlink"
                except OSError:
                    census_shutil.copy2(source, temporary)
            else:
                census_shutil.copy2(source, temporary)
            census_os.replace(temporary, destination)
        finally:
            temporary.unlink(missing_ok=True)
    return {"original_local_input": census_relative(source, root), "archived_path": census_relative(destination, root),
            "storage": storage, "hardlink_count": destination.stat().st_nlink,
            "bytes": destination.stat().st_size, "sha256": checksum}

In [30]:
def census_get_metadata(table, destination, metadata_dir, root, mode):
    url = f"https://api.census.gov/data/2024/acs/acs5/groups/{table}.json"
    if destination.exists():
        return census_json.loads(destination.read_text()), {"source_url": url, "path": census_relative(destination, root), "reused": True}
    candidates = [metadata_dir / f"{table}_metadata.json", metadata_dir / f"{table}.json"]
    for source in candidates:
        if source.is_file():
            info = census_archive_file(source, destination, root, mode)
            return census_json.loads(destination.read_text()), {"source_url": url, **info}
    for attempt in range(3):
        try:
            with census_urllib.request.urlopen(url, timeout=45) as response:
                value = census_json.load(response)
            if "variables" not in value:
                raise ValueError(f"Invalid Census variable dictionary: {table}")
            census_atomic_json(destination, value)
            return value, {"source_url": url, "path": census_relative(destination, root), "downloaded_small_metadata": True,
                           "bytes": destination.stat().st_size, "sha256": census_sha256(destination)}
        except (census_urllib.error.URLError, TimeoutError, OSError):
            if attempt == 2:
                raise RuntimeError(f"Cannot read missing {table} variable dictionary. Supply --metadata-dir with official JSON dictionaries.") from None
            census_time.sleep(2 ** attempt)

In [31]:
def census_prepare_shapefile(archive_path, target):
    target.mkdir(parents=True, exist_ok=True)
    with census_zipfile.ZipFile(archive_path) as archive:
        bad = archive.testzip()
        if bad:
            raise ValueError(f"TIGER ZIP CRC failed: {bad}")
        for suffix in [".shp", ".shx", ".dbf", ".prj", ".cpg"]:
            name = "tl_2024_06_tract" + suffix
            data = archive.read(name)
            destination = target / name
            if destination.exists():
                if census_hashlib.sha256(destination.read_bytes()).digest() != census_hashlib.sha256(data).digest():
                    raise ValueError(f"Existing extracted boundary differs: {destination.name}")
                continue
            temporary = destination.with_name(name + f".{census_os.getpid()}.part")
            try:
                temporary.write_bytes(data)
                census_os.replace(temporary, destination)
            finally:
                temporary.unlink(missing_ok=True)
    return target / "tl_2024_06_tract.shp"

In [32]:
def census_parse_table(path, table):
    """Read every estimate AND MOE column, not only the initial feature subset."""
    records = {}
    negative_codes = census_collections.Counter()
    negative_by_field = census_collections.Counter()
    nonnumeric_by_field = census_collections.Counter()
    with path.open(encoding="utf-8-sig", newline="") as stream:
        reader = census_csv.DictReader(stream, delimiter="|")
        fields = []
        for field in reader.fieldnames or []:
            if field == "GEO_ID":
                continue
            match = census_re.fullmatch(rf"{table}_([EM])(\d{{3}})", field)
            if not match:
                raise ValueError(f"Unexpected source column: {field}")
            fields.append((field, f"{table}_{match[2]}{match[1]}"))
        if not fields or "GEO_ID" not in (reader.fieldnames or []):
            raise ValueError(f"Not a valid Summary File: {path.name}")
        for row in reader:
            source_id = row["GEO_ID"]
            if not source_id.startswith("1400000US06"):
                continue
            geoid = source_id.removeprefix("1400000US")
            if len(geoid) != 11 or not geoid.isdigit() or geoid in records:
                raise ValueError(f"Invalid or duplicate California GEOID in {table}")
            output = {}
            for summary_field, api_field in fields:
                value = row[summary_field]
                try:
                    number = float(value)
                    if not census_math.isfinite(number) or not number.is_integer():
                        raise ValueError()
                    number = int(number)
                    if number < 0:
                        negative_codes[str(number)] += 1
                        negative_by_field[api_field] += 1
                        output[api_field] = None
                    else:
                        output[api_field] = number
                except (ValueError, TypeError):
                    output[api_field] = None
                    nonnumeric_by_field[api_field] += 1
            records[geoid] = output
    if len(records) != census_EXPECTED_ROWS:
        raise ValueError(f"{table}: expected 9129 California tracts; read {len(records)}")
    return records, [api for _, api in fields], {
        "row_count": len(records), "source_numeric_columns": len(fields),
        "estimate_columns": sum(name.endswith("E") for _, name in fields),
        "moe_columns": sum(name.endswith("M") for _, name in fields),
        "negative_sentinel_counts": dict(negative_codes),
        "negative_sentinel_by_field": dict(negative_by_field),
        "nonnumeric_missing_by_field": dict(nonnumeric_by_field),
    }

In [33]:
def census_plus(table, lines):
    return " + ".join(f'"{table}_{line:03d}E"' for line in lines)

In [34]:
def census_copy_parquet(connection, query, path):
    temporary = path.with_name(path.name + f".{census_os.getpid()}.part")
    try:
        connection.execute(f"COPY ({query}) TO ? (FORMAT PARQUET, COMPRESSION ZSTD)", [str(temporary)])
        census_os.replace(temporary, path)
    finally:
        temporary.unlink(missing_ok=True)

In [35]:
def census_dictionary_rows(source_fields, metadata, feature_schema, tract_schema):
    result = []
    units = lambda field: ("2024 inflation-adjusted USD" if field.startswith("B19013") else
                           "households" if field.startswith("B11001") else "persons")
    source_meanings = {}
    for field in source_fields:
        table = field.split("_", 1)[0]
        info = metadata[table]["variables"].get(field, {})
        source_meanings[field] = {
            "description": info.get("label", field), "source": table,
            "unit": units(field), "formula": "Original ACS estimate" if field.endswith("E") else "Original ACS margin of error",
            "missing_rule": "Negative Census special codes and nonnumeric symbols -> null; untouched DAT preserves the original code",
        }
    derived = {
        "GEOID": ("11-character California Census Tract identifier; preserve leading zero", "identifier", "STATEFP + COUNTYFP + TRACTCE"),
        "STATEFP": ("State FIPS; California=06", "identifier", "TIGER attribute"),
        "COUNTYFP": ("3-character county FIPS", "identifier", "TIGER attribute"),
        "TRACTCE": ("6-character Census tract code", "identifier", "TIGER attribute"),
        "GEOIDFQ": ("Fully qualified Census tract geographic ID", "identifier", "1400000US + GEOID"),
        "MTFCC": ("MAF/TIGER feature classification code for the tract", "code", "TIGER attribute"),
        "FUNCSTAT": ("TIGER legal/statistical area functional status", "code", "TIGER attribute"),
        "INTPTLAT": ("Latitude of the source tract's internal representative point", "degrees NAD83", "TIGER attribute; retained as source text"),
        "INTPTLON": ("Longitude of the source tract's internal representative point", "degrees NAD83", "TIGER attribute; retained as source text"),
        "county_geoid": ("5-character county GEOID", "identifier", "STATEFP || COUNTYFP"),
        "county_block_id": ("County-based spatial grouping identifier; does not eliminate all spatial dependence", "identifier", "county_geoid"),
        "is_bay_area_9county": ("Member of Alameda, Contra Costa, Marin, Napa, San Francisco, San Mateo, Santa Clara, Solano, Sonoma", "boolean", "COUNTYFP in 001,013,041,055,075,081,085,095,097"),
        "NAME": ("Census tract display name (TIGER NAMELSAD); not a locally named neighbourhood", "text", "TIGER NAMELSAD"),
        "NAMELSAD": ("TIGER tract name with legal/statistical area descriptor", "text", "TIGER attribute"),
        "ALAND": ("Official tract LAND area", "m2", "TIGER attribute; not total polygon area"),
        "AWATER": ("Official tract WATER area", "m2", "TIGER attribute"),
        "land_area_km2": ("Land area used for population density", "km2", "ALAND / 1000000.0"),
        "water_area_km2": ("Water area", "km2", "AWATER / 1000000.0"),
        "polygon_area_km2": ("Total polygon area measured in California Albers; includes water within the polygon", "km2", "ST_Area(geometry_3310) / 1000000.0"),
        "geometry": ("Tract polygon/multipolygon; primary GeoParquet geometry", "WKB EPSG:4326", "Transform NAD83 EPSG:4269 to EPSG:4326 with always_xy=true"),
        "geometry_3310": ("Tract polygon/multipolygon for metre-distance/area operations", "WKB EPSG:3310", "Transform NAD83 EPSG:4269 to NAD83 California Albers EPSG:3310"),
        "acs_period": ("Five-year survey period, not a single-year 2024 observation", "text", "2020-2024"),
        "population": ("Total resident population estimate", "persons", "B01001_001E"),
        "under18_count": ("Male + female residents aged under 18", "persons", "B01001 male 003-006 + female 027-030 estimates"),
        "under18_share": ("Under-18 share of total population", "fraction 0..1", "under18_count / population"),
        "young_adult_18_34_count": ("Male + female residents aged 18 through 34", "persons", "B01001 male 007-012 + female 031-036 estimates"),
        "young_adult_18_34_share": ("Age-18-34 share of total population", "fraction 0..1", "young_adult_18_34_count / population"),
        "median_household_income_2024_usd": ("Median household income, not income per individual", "2024 inflation-adjusted USD", "B19013_001E"),
        "median_household_income_moe": ("Original margin of error for the household income median", "2024 inflation-adjusted USD", "B19013_001M"),
        "population_age3plus": ("Population age 3 and over, the universe for school enrollment", "persons", "B14001_001E"),
        "college_enrolled_count": ("Undergraduate + graduate/professional-school enrollment", "persons", "B14001_008E + B14001_009E"),
        "college_enrolled_share_3plus": ("College-enrolled share of residents age 3+", "fraction 0..1", "college_enrolled_count / population_age3plus"),
        "households": ("Total households", "households", "B11001_001E"),
        "family_household_share": ("Family household share; not the share with children", "fraction 0..1", "B11001_002E / B11001_001E"),
        "living_alone_household_share": ("Householder living alone; row 008, not nonfamily total row 007", "fraction 0..1", "B11001_008E / B11001_001E"),
        "population_density_per_km2": ("Resident population divided by official land area", "persons/km2 of land", "population / NULLIF(ALAND/1000000.0,0)"),
    }
    for dataset, schema in [("ca_acs2024_tract_features.parquet", feature_schema), ("ca_tracts_2024.parquet", tract_schema)]:
        for field in schema:
            if field.name in source_meanings:
                info = source_meanings[field.name]
            else:
                description, unit, formula = derived.get(field.name, ("Original TIGER attribute", "source attribute", "TIGER attribute"))
                if dataset == "ca_tracts_2024.parquet" and field.name == "NAME":
                    description, unit, formula = "Original TIGER tract short name without LSAD prefix", "text", "TIGER NAME"
                info = {"description": description, "source": "derived ACS/TIGER" if field.name in derived else "TIGER2024",
                        "unit": unit, "formula": formula,
                        "missing_rule": "Zero/missing denominators -> null; derived ratio MOEs are not computed"}
            result.append({"dataset": dataset, "field": field.name, "storage_type": str(field.type), **info})
    return result

In [36]:
def census_main():
    parser = census_argparse.ArgumentParser(description=census___doc__, formatter_class=census_argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--project-root", type=census_Path, default=census_Path(census___file__).resolve().parents[1])
    parser.add_argument("--acs-raw-dir", default="tutorial_tests/acs_full/raw")
    parser.add_argument("--tract-zip", default="tutorial_tests/tracts/tl_2024_06_tract.zip")
    parser.add_argument("--metadata-dir", default="validation/geography")
    parser.add_argument("--archive-mode", choices=["hardlink", "copy"], default="hardlink")
    args = parser.parse_args()
    root = args.project_root.resolve()
    source_dir = census_resolve_input(args.acs_raw_dir, root)
    tract_zip = census_resolve_input(args.tract_zip, root)
    metadata_dir = census_resolve_input(args.metadata_dir, root)
    raw_dir = root / "data/raw/census"
    interim_dir = root / "data/interim/census"
    qa_path = root / "validation/census_full_qa.json"
    interim_dir.mkdir(parents=True, exist_ok=True)
    census_atomic_json(interim_dir / "manifest.json", {"complete":False,"started_utc":census_utc_now(),
                "status":"Preparing/validating census outputs; do not treat existing files as this run's success."})
    raw_manifest = {"complete": False, "started_utc": census_utc_now(), "archive_mode_requested": args.archive_mode,
                    "notes": ["Hardlinked raw files are immutable shared inodes; do not modify either copy.",
                              "This pipeline reuses acquired DAT/ZIP files and downloads no large source files."]}
    sources, metadata, metadata_info = [], {}, []
    census_atomic_json(raw_dir / "manifest.json", raw_manifest)
    try:
        for table in census_TABLES:
            filename = f"acsdt5y2024-{table.lower()}.dat"
            info = census_archive_file(source_dir / filename, raw_dir / "acs2024" / filename, root, args.archive_mode)
            sources.append({"table": table, "source_url": census_SUMMARY_BASE + filename, **info})
            metadata[table], info = census_get_metadata(table, raw_dir / "metadata" / f"{table}.json", metadata_dir, root, args.archive_mode)
            metadata_info.append({"table": table, **info})
        zip_info = census_archive_file(tract_zip, raw_dir / "tiger2024" / tract_zip.name, root, args.archive_mode)
        sources.append({"source_url": census_TRACT_URL, **zip_info})
        input_manifests = []
        for source in [source_dir.parent / "manifest_acs.json", tract_zip.parent / "manifest_tracts.json"]:
            if source.is_file():
                input_manifests.append(census_archive_file(source, raw_dir / "input_manifests" / source.name, root, args.archive_mode))
        recorded_manifests = {census_Path(info["archived_path"]).name for info in input_manifests}
        for archived in sorted((raw_dir / "input_manifests").glob("*.json")):
            if archived.name not in recorded_manifests:
                input_manifests.append(census_archive_file(archived, archived, root, args.archive_mode))
        shapefile = census_prepare_shapefile(raw_dir / "tiger2024" / tract_zip.name, raw_dir / "tiger2024" / "shapefile")
        raw_manifest.update({"complete": True, "sources": sources, "variable_dictionaries": metadata_info,
                             "original_download_manifests": input_manifests, "shapefile": census_relative(shapefile, root),
                             "finished_utc": census_utc_now()})
        census_atomic_json(raw_dir / "manifest.json", raw_manifest)

        records, source_fields, table_qa = {}, [], {}
        for table in census_TABLES:
            parsed, fields, table_qa[table] = census_parse_table(raw_dir / "acs2024" / f"acsdt5y2024-{table.lower()}.dat", table)
            if records and set(records) != set(parsed):
                raise ValueError("ACS source tables have different tract identifier sets.")
            for geoid, attributes in parsed.items():
                records.setdefault(geoid, {}).update(attributes)
            source_fields.extend(fields)
        schema = census_pa.schema([census_pa.field("GEOID", census_pa.string())] + [census_pa.field(name, census_pa.int64()) for name in source_fields])
        arrow_source = census_pa.Table.from_pylist([{"GEOID": geoid, **records[geoid]} for geoid in sorted(records)], schema=schema)

        con = census_duckdb.connect()
        try:
            con.execute("LOAD spatial")
        except census_duckdb.Error:
            raise RuntimeError("DuckDB spatial extension must already be installed; install it once with INSTALL spatial in your environment.") from None
        con.execute("SET threads=4")
        con.register("acs_source", arrow_source)
        bay_sql = ",".join("'" + code + "'" for code in census_BAY_COUNTIES)
        con.execute(f"""CREATE TEMP TABLE tracts AS
            SELECT * EXCLUDE (geom,OGC_FID),
                   STATEFP || COUNTYFP AS county_geoid,
                   STATEFP || COUNTYFP AS county_block_id,
                   COUNTYFP IN ({bay_sql}) AS is_bay_area_9county,
                   ALAND / 1000000.0 AS land_area_km2,
                   AWATER / 1000000.0 AS water_area_km2,
                   ST_Area(ST_Transform(geom,'EPSG:4269','EPSG:3310',always_xy:=true)) / 1000000.0 AS polygon_area_km2,
                   ST_SetCRS(ST_Transform(geom,'EPSG:4269','EPSG:4326',always_xy:=true),'EPSG:4326') AS geometry,
                   ST_SetCRS(ST_Transform(geom,'EPSG:4269','EPSG:3310',always_xy:=true),'EPSG:3310') AS geometry_3310
              FROM ST_Read(?) ORDER BY GEOID""", [str(shapefile)])
        con.execute(f"""CREATE TEMP TABLE features AS
            SELECT a.*, b.NAMELSAD AS NAME, '2020-2024' AS acs_period,
                   B01001_001E AS population,
                   {census_plus('B01001', census_CHILD_LINES)} AS under18_count,
                   ({census_plus('B01001', census_CHILD_LINES)}) / NULLIF(B01001_001E,0) AS under18_share,
                   {census_plus('B01001', census_YOUNG_LINES)} AS young_adult_18_34_count,
                   ({census_plus('B01001', census_YOUNG_LINES)}) / NULLIF(B01001_001E,0) AS young_adult_18_34_share,
                   B19013_001E AS median_household_income_2024_usd,
                   B19013_001M AS median_household_income_moe,
                   B14001_001E AS population_age3plus,
                   B14001_008E+B14001_009E AS college_enrolled_count,
                   (B14001_008E+B14001_009E) / NULLIF(B14001_001E,0) AS college_enrolled_share_3plus,
                   B11001_001E AS households,
                   B11001_002E / NULLIF(B11001_001E,0) AS family_household_share,
                   B11001_008E / NULLIF(B11001_001E,0) AS living_alone_household_share,
                   b.STATEFP, b.COUNTYFP, b.county_geoid, b.county_block_id, b.is_bay_area_9county,
                   b.ALAND, b.AWATER, b.land_area_km2, b.water_area_km2,
                   B01001_001E / NULLIF(b.land_area_km2,0) AS population_density_per_km2
              FROM acs_source a INNER JOIN tracts b USING (GEOID) ORDER BY GEOID""")

        geometry_summary = con.execute("""SELECT count(*),count(DISTINCT GEOID),
            count(*) FILTER (WHERE NOT ST_IsValid(geometry)),count(*) FILTER (WHERE NOT ST_IsValid(geometry_3310)),
            count(*) FILTER (WHERE ST_IsEmpty(geometry)),count(*) FILTER (WHERE ALAND=0),
            count(*) FILTER (WHERE is_bay_area_9county),count(DISTINCT county_geoid),
            min(land_area_km2),max(land_area_km2) FROM tracts""").fetchone()
        feature_summary = con.execute("""SELECT count(*),count(DISTINCT GEOID),
            count(*) FILTER (WHERE population>0 AND median_household_income_2024_usd>0),
            count(*) FILTER (WHERE population=0),count(*) FILTER (WHERE households=0),
            count(*) FILTER (WHERE land_area_km2=0),count(*) FILTER (WHERE population_density_per_km2 IS NULL)
            FROM features""").fetchone()
        unmatched = con.execute("SELECT count(*) FROM tracts ANTI JOIN acs_source USING(GEOID)").fetchone()[0]
        unmatched_reverse = con.execute("SELECT count(*) FROM acs_source ANTI JOIN tracts USING(GEOID)").fetchone()[0]
        ratios = ["under18_share", "young_adult_18_34_share", "college_enrolled_share_3plus", "family_household_share", "living_alone_household_share"]
        bad_ratios = con.execute("SELECT count(*) FROM features WHERE " + " OR ".join(f"{field}<0 OR {field}>1" for field in ratios)).fetchone()[0]
        invalid_geoid = con.execute("SELECT count(*) FROM features WHERE length(GEOID)<>11 OR left(GEOID,2)<>'06' OR NOT regexp_full_match(GEOID,'[0-9]{11}')").fetchone()[0]
        negative_cleaned = con.execute("SELECT count(*) FROM features WHERE " + " OR ".join(f'"{field}"<0' for field in source_fields)).fetchone()[0]
        missing_fields = con.execute("SELECT " + ",".join(f'count(*) FILTER (WHERE "{field}" IS NULL) AS "{field}"' for field in source_fields + ratios + ["population_density_per_km2"]) + " FROM features").fetchone()
        missing_counts = dict(zip(source_fields + ratios + ["population_density_per_km2"], missing_fields))
        geometry_types = dict(con.execute("SELECT ST_GeometryType(geometry),count(*) FROM tracts GROUP BY 1").fetchall())
        checks = {
            "geometry_rows_9129": geometry_summary[0] == census_EXPECTED_ROWS,
            "geometry_unique_geoid": geometry_summary[1] == census_EXPECTED_ROWS,
            "wgs84_geometry_valid": geometry_summary[2] == 0,
            "epsg3310_geometry_valid": geometry_summary[3] == 0,
            "geometry_not_empty": geometry_summary[4] == 0,
            "bay9_tracts_1772": geometry_summary[6] == 1772,
            "county_count_58": geometry_summary[7] == 58,
            "acs_join_rows_9129": feature_summary[0] == census_EXPECTED_ROWS,
            "feature_unique_geoid": feature_summary[1] == census_EXPECTED_ROWS,
            "geoid_11_digit_strings": invalid_geoid == 0,
            "no_unmatched_tracts_or_acs_rows": unmatched == 0 and unmatched_reverse == 0,
            "ratios_within_0_1": bad_ratios == 0,
            "negative_source_codes_are_missing": negative_cleaned == 0,
            "zero_land_area_density_is_missing": feature_summary[5] == feature_summary[6],
            "all_69_source_moes_retained": len([name for name in source_fields if name.endswith("M")]) == 69,
        }
        if not all(checks.values()):
            raise ValueError(f"Census QA failed: {[key for key,value in checks.items() if not value]}")
        geometry_path = interim_dir / "ca_tracts_2024.parquet"
        feature_path = interim_dir / "ca_acs2024_tract_features.parquet"
        census_copy_parquet(con, "SELECT * FROM tracts ORDER BY GEOID", geometry_path)
        census_copy_parquet(con, "SELECT * FROM features ORDER BY GEOID", feature_path)
        geometry_schema = census_pq.read_schema(geometry_path)
        feature_schema = census_pq.read_schema(feature_path)
        geo_metadata = census_json.loads(geometry_schema.metadata[b"geo"])
        crs_codes = {name: value.get("crs", {}).get("id", {}).get("code") for name,value in geo_metadata["columns"].items()}
        if crs_codes != {"geometry": 4326, "geometry_3310": 3310}:
            raise ValueError(f"Unexpected GeoParquet CRS metadata: {crs_codes}")
        checks["geoparquet_crs_metadata_complete"] = True
        dictionary_path = interim_dir / "field_dictionary.csv"
        rows = census_dictionary_rows(source_fields, metadata, feature_schema, geometry_schema)
        temporary = dictionary_path.with_name(dictionary_path.name + ".part")
        with temporary.open("w", newline="", encoding="utf-8") as stream:
            writer = census_csv.DictWriter(stream, fieldnames=["dataset","field","storage_type","description","source","unit","formula","missing_rule"])
            writer.writeheader();writer.writerows(rows)
        census_os.replace(temporary, dictionary_path)
        qa = {
            "complete": True, "prepared_utc": census_utc_now(), "checks": checks,
            "inputs_reused_not_redownloaded": True, "source_period": "2020-2024", "boundary_vintage": 2024,
            "table_qa": table_qa, "source_numeric_columns": len(source_fields),
            "source_estimate_columns": 69, "source_moe_columns": 69,
            "tract_rows": geometry_summary[0], "feature_rows": feature_summary[0],
            "feature_column_count": len(feature_schema), "tract_column_count": len(geometry_schema),
            "positive_population_and_income_rows": feature_summary[2],
            "zero_population_tracts": feature_summary[3], "zero_household_tracts": feature_summary[4],
            "zero_land_area_tracts": feature_summary[5], "null_population_density": feature_summary[6],
            "bay9_tracts": geometry_summary[6], "county_count": geometry_summary[7],
            "geometry_types": geometry_types, "geoparquet_crs_codes": crs_codes,
            "land_area_km2_range": [geometry_summary[8],geometry_summary[9]],
            "missing_counts": missing_counts,
            "paths": {"raw_manifest": census_relative(raw_dir / "manifest.json",root),
                      "tract_geoparquet": census_relative(geometry_path,root), "acs_features": census_relative(feature_path,root),
                      "field_dictionary": census_relative(dictionary_path,root)},
            "notes": [
                "All 138 E/M source columns are retained; the initial tutorial source CSV held only 60 selected E/M columns.",
                "Negative source sentinels become null, including controlled-estimate MOE -555555555; immutable raw DATs retain exact codes.",
                "Source MOEs are retained; derived share/density MOEs are not estimated by this pipeline.",
                "ALAND is official land area in m2; water is excluded from population density. EPSG3310 polygon area includes the entire boundary.",
                "County IDs are supplied for grouping, without claiming county groups remove all spatial dependence.",
                "No store labels, counts, candidate-site ranking, or root analytical table are created here.",
            ],
        }
        census_atomic_json(qa_path, qa)
        outputs = [{"path":census_relative(path,root),"bytes":path.stat().st_size,"sha256":census_sha256(path)}
                   for path in [geometry_path,feature_path,dictionary_path]]
        census_atomic_json(interim_dir / "manifest.json", {"complete":True,"prepared_utc":census_utc_now(),"outputs":outputs,
                    "qa_path":census_relative(qa_path,root),
                    "reproduce_command":"validation/.venv/bin/python pipeline/prepare_census.py --acs-raw-dir data/raw/census/acs2024 --tract-zip data/raw/census/tiger2024/tl_2024_06_tract.zip --metadata-dir data/raw/census/metadata",
                    "dependencies":{"duckdb":census_duckdb.__version__,"pyarrow":census_pa.__version__},"geometry_crs":crs_codes})
        print(census_json.dumps({"complete":True,"tract_rows":geometry_summary[0],"acs_rows":feature_summary[0],
                          "source_estimates":69,"source_moes":69,"qa":census_relative(qa_path,root)},indent=2))
        return 0
    except Exception as exc:
        census_atomic_json(qa_path,{"complete":False,"failed_utc":census_utc_now(),"error":str(exc)})
        census_atomic_json(interim_dir / "manifest.json", {"complete":False,"failed_utc":census_utc_now(),
                    "error":str(exc),"qa_path":census_relative(qa_path,root)})
        print(f"Census preparation failed: {exc}",file=census_sys.stderr)
        return 1

In [37]:
with notebook_args('--project-root',RUN_ROOT,
 '--acs-raw-dir',SOURCE_ROOT/'data/raw/census/acs2024',
 '--tract-zip',SOURCE_ROOT/'data/raw/census/tiger2024/tl_2024_06_tract.zip',
 '--metadata-dir',INPUTS/'data/raw/census/metadata','--archive-mode','hardlink'):
    assert census_main()==0
census_qa=json.loads((RUN_ROOT/'validation/census_full_qa.json').read_text())
assert census_qa['complete'] and all(census_qa['checks'].values())
print('Census checks:',len(census_qa['checks']))
gc.collect()

{
  "complete": true,
  "tract_rows": 9129,
  "acs_rows": 9129,
  "source_estimates": 69,
  "source_moes": 69,
  "qa": "validation/census_full_qa.json"
}
Census checks: 16


30

## 5. 地点去重、跨源实体匹配与空间JOIN

规范化名称和地址单元号，处理近距离重复对象，将OSM候选关联到Overture实体；应用保存的V1人工裁决。以点面覆盖关系精确限定加州，再按GEOID关联ACS。此处先生成候选主表，最终资格仍需后面V2／V3筛选。

函数／常量加阶段前缀，避免同一内核中的同名变量相互覆盖。

In [38]:
# Source: pipeline/build_master.py; SHA256: 58b49f849f3853e0e1affa4f5b3b065ba18d049a4cd732556b5a26677333ae7d
base___file__ = str(RUN_ROOT / 'pipeline/build_master.py')
base___doc__ = 'Build a reproducible California retail master from four public sources.\n\nNo training, global imputation, fabricated sites, or duplicated snapshot years.\nMapped retailer categories provide provisional labels, not audited ground truth.'
"""Build a reproducible California retail master from four public sources.

No training, global imputation, fabricated sites, or duplicated snapshot years.
Mapped retailer categories provide provisional labels, not audited ground truth.
"""

import argparse as base_argparse
from collections import Counter as base_Counter, defaultdict as base_defaultdict
from datetime import datetime as base_datetime, timezone as base_timezone
import hashlib as base_hashlib
import json as base_json
import math as base_math
from pathlib import Path as base_Path
import re as base_re
import time as base_time
import unicodedata as base_unicodedata

import duckdb as base_duckdb
import numpy as base_np
import pyarrow as base_pa
import pyarrow.csv as base_pacsv
import pyarrow.parquet as base_pq
from rapidfuzz.fuzz import ratio as base_ratio
from scipy.spatial import cKDTree as base_cKDTree

base_ROOT = base_Path(base___file__).resolve().parents[1]


base_PENDING_PLUS = (*overture_PENDING_CATEGORY_REVIEW, 'gift_shop', 'cards_and_stationery_store')
base_NAME_TARGET = base_re.compile(r"\bpop[ -]?mart\b|blind[ -]?box|collect[ia]bl|designer[ -]?toys|pok[eé]mon|trading[ -]?cards?|\btcg\b|gundam|sanrio|anime|\blego\b", base_re.I)
base_STRONG_NAME = base_re.compile(r"\bpop[ -]?mart\b|blind[ -]?box|designer[ -]?toys|pok[eé]mon|trading[ -]?cards?|\btcg\b|gundam|\blego\b", base_re.I)
base_ADULT = base_re.compile(r"\badult\b|\berotic\b|\bsex(?:y|ual)?\b|\bpleasure\b|\bporn\b|\bhustler\b|adam\s*(?:&|and)\s*eve|sextoy|lovehoney", base_re.I)
base_ONLINE = base_re.compile(r"\.com\b|\bonline[ -]?(?:only|store|shop)\b|\bwholesale\b|\bdistribut(?:or|ion)\b|\bmanufactur", base_re.I)
base_AMBIGUOUS_BRANDS = base_re.compile(r"\bgame[ -]?stop\b|\bhot topic\b|\bbox[ -]?lunch\b|\bspencer'?s\b|\btoys\s*r\s*us\b|robo[ -]?shop|\bvending\b", base_re.I)

In [39]:
def base_q(s): return "'" + str(s).replace("'", "''") + "'"

In [40]:
def base_sql_list(values): return '[' + ','.join(base_q(v) for v in values) + ']'

In [41]:
def base_log(s): print(s, flush=True)

In [42]:
def base_js(path, obj):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.partial')
    tmp.write_text(base_json.dumps(obj, ensure_ascii=False, indent=2, default=str))
    tmp.replace(path)

In [43]:
def base_name_key(s):
    text = base_unicodedata.normalize('NFKD', (s or '').casefold())
    text = ''.join(c for c in text if not base_unicodedata.combining(c))
    text = base_re.sub(r"\b(?:incorporated|inc|llc|ltd|corporation)\b", '', text)
    return ''.join(c for c in text if c.isalnum())

In [44]:
def base_address_key(s):
    text = (s or '').casefold()
    for a,b in [('street','st'),('avenue','ave'),('boulevard','blvd'),('road','rd'),('suite','ste')]:
        text = base_re.sub(r'\b' + a + r'\b', b, text)
    return base_re.sub(r'[^a-z0-9]', '', text)

In [45]:
def base_unit(s):
    # A prefix must end before a unit token; "Stevens" is a street name,
    # whereas "Ste 2255", "Spc2170" and "Unit Suite B" contain unit markers.
    marker=r'\b(?:suite|ste|unit|space|spc|floor)(?=[.\s#]|\d|[a-z](?:[-\s]|$))[.\s]*'
    m = base_re.search(r'(?:'+marker+r'|#\s*)+([a-z0-9-]+)', (s or '').casefold())
    return m.group(1) if m else None

In [46]:
def base_compatible_address(a,b):
    ua,ub = base_unit(a),base_unit(b)
    if ua and ub and ua != ub: return False
    na,nb = base_re.match(r'\s*(\d+)',a or ''),base_re.match(r'\s*(\d+)',b or '')
    return not (na and nb and na.group(1) != nb.group(1))

In [47]:
def base_parquet(path, rows, schema=None):
    path.parent.mkdir(parents=True, exist_ok=True)
    table = base_pa.Table.from_pylist(rows, schema=schema)
    tmp=path.with_suffix('.parquet.partial')
    base_pq.write_table(table,tmp,compression='zstd');tmp.replace(path)
    return table

In [48]:
def base_connect():
    c=base_duckdb.connect()
    try:c.execute('LOAD spatial')
    except base_duckdb.Error:c.execute('INSTALL spatial');c.execute('LOAD spatial')
    c.execute("SET memory_limit='3GB'; SET threads=4")
    return c

In [49]:
def base_prepare_context(points_path, tract_path, out, report):
    c=base_connect()
    frame=f"coalesce(list_extract(hierarchy,2) IN {base_sql_list(overture_RETAIL_PARENT_ALLOW)},false) AND NOT coalesce(list_has_any(hierarchy,{base_sql_list(overture_RETAIL_PATH_DENY)}),false)"
    pattern=base_NAME_TARGET.pattern.replace('\\b',r'\b')
    c.execute(f'CREATE VIEW p AS SELECT * FROM read_parquet({base_q(points_path)})')
    report['raw_overture_rows']=c.execute('SELECT count(*) FROM p').fetchone()[0]
    c.execute(f"""CREATE TEMP TABLE flat AS
      SELECT id,name,category,basic_category,hierarchy,confidence,operating_status,lon,lat,
        addresses[1].freeform AS address,addresses[1].locality AS locality,
        addresses[1].postcode AS postcode,addresses[1].region AS address_region,
        addresses[1].country AS address_country,websites[1] AS website,phones[1] AS phone,
        to_json(sources) AS overture_sources_json,
        ({frame}) AS in_retail_frame,
        hierarchy[1]='shopping' AS is_shopping,
        hierarchy[1]='food_and_drink' AS is_food,
        category='shopping_mall' AS is_mall,
        category='college_university' AS is_university,
        (category IN {base_sql_list((*overture_STRICT_TARGET_PROXY,*base_PENDING_PLUS))}
          OR regexp_matches(coalesce(name,''),{base_q('(?i)'+pattern)})) AS target_signal,
        passes_initial_quality_screen,
        coalesce(addresses[1].country NOT IN ('US','USA'),false) AS address_country_conflict
      FROM p
      WHERE hierarchy[1] IN ('shopping','food_and_drink')
         OR category='college_university'
         OR regexp_matches(coalesce(name,''),{base_q('(?i)'+pattern)})
      """)
    report['context_pre_quality_rows']=c.execute('SELECT count(*) FROM flat').fetchone()[0]
    report['context_quality_flags_overlapping']={
      'source_quality_screen_failed':c.execute('SELECT count(*) FROM flat WHERE NOT passes_initial_quality_screen').fetchone()[0],
      'non_US_address_retained_for_border_context':c.execute('SELECT count(*) FROM flat WHERE address_country_conflict').fetchone()[0],
    }
    c.execute("""CREATE TEMP TABLE context AS
      SELECT *,ST_X(g) AS x_m,ST_Y(g) AS y_m FROM (
        SELECT * EXCLUDE(passes_initial_quality_screen),
        ST_Transform(ST_Point(lon,lat),'EPSG:4326','EPSG:3310',always_xy:=true) AS g
        FROM flat WHERE passes_initial_quality_screen
          AND isfinite(lon) AND isfinite(lat)
      )
    """)
    c.execute(f'CREATE VIEW tracts AS SELECT GEOID,geometry FROM read_parquet({base_q(tract_path)})')
    c.execute("""CREATE TEMP TABLE spatially_tagged AS
      SELECT * EXCLUDE(g,choice) FROM (
        SELECT c.*,t.GEOID,
          count(t.GEOID) OVER (PARTITION BY c.id) AS tract_match_count,
          row_number() OVER (PARTITION BY c.id ORDER BY t.GEOID NULLS LAST) AS choice
        FROM context c LEFT JOIN tracts t ON ST_Covers(t.geometry,ST_Point(c.lon,c.lat))
      ) WHERE choice=1
    """)
    report['context_quality_rows']=c.execute('SELECT count(*) FROM spatially_tagged').fetchone()[0]
    report['context_in_california_rows']=c.execute('SELECT count(*) FROM spatially_tagged WHERE GEOID IS NOT NULL').fetchone()[0]
    report['context_multimatch_rows']=c.execute('SELECT count(*) FROM spatially_tagged WHERE tract_match_count>1').fetchone()[0]
    path=out/'context_pre_dedup.parquet'
    c.execute('COPY spatially_tagged TO ? (FORMAT PARQUET,COMPRESSION ZSTD)',[str(path)])
    rows=c.execute('SELECT * FROM spatially_tagged ORDER BY id').to_arrow_table().to_pylist()
    c.close()
    for row in rows:
        row['norm_name']=base_name_key(row['name']);row['source_overture_ids']=[row['id']]
        row['source_osm_ids']=[];row['osm_records']=[];row['source_provider']='Overture'
    return rows

In [50]:
def base_deduplicate(rows, report, out):
    groups=base_defaultdict(list)
    for i,r in enumerate(rows):
        if r['norm_name']: groups[r['norm_name']].append(i)
    parent=list(range(len(rows))); component_members={i:[i] for i in range(len(rows))}
    def find(x):
        while parent[x]!=x:parent[x]=parent[parent[x]];x=parent[x]
        return x
    pairs=[]
    for key,ix in groups.items():
        if len(ix)<2:continue
        xy=base_np.array([(rows[i]['x_m'],rows[i]['y_m']) for i in ix])
        for a,b in base_cKDTree(xy).query_pairs(20):
            i,j=ix[a],ix[b];ri,rj=rows[i],rows[j]
            if not base_compatible_address(ri['address'],rj['address']):continue
            # Never merge different kinds of mall/institution/retail destinations.
            if (ri['is_mall']!=rj['is_mall']) or (ri['is_university']!=rj['is_university']):continue
            # Cap cluster extent by checking each candidate against its representative.
            ra,rb=find(i),find(j)
            if ra==rb:continue
            members=component_members[ra]+component_members[rb]
            points=base_np.array([(rows[k]['x_m'],rows[k]['y_m']) for k in members])
            if base_np.max(base_np.linalg.norm(points[:,None,:]-points[None,:,:],axis=2))>20:continue
            if any(not base_compatible_address(rows[a]['address'],rows[b]['address']) for a in component_members[ra] for b in component_members[rb]):continue
            parent[rb]=ra
            component_members[ra]=members;del component_members[rb]
            pairs.append({'left_id':ri['id'],'right_id':rj['id'],'distance_m':round(float(base_np.linalg.norm(xy[a]-xy[b])),3),'rule':'same_normalized_name_20m_compatible_address'})
    components=base_defaultdict(list)
    for i in range(len(rows)):components[find(i)].append(i)
    retained=[];mapping=[]
    for ix in components.values():
        pick=min(ix,key=lambda i:(-int(rows[i]['operating_status']=='open'),-rows[i]['confidence'],-int(bool(rows[i]['address'])),rows[i]['id']))
        rep=rows[pick]
        rep['source_overture_ids']=sorted(rows[i]['id'] for i in ix)
        rep['overture_duplicate_count']=len(ix)-1
        # Preserve a target review signal even if the representative category differs.
        rep['target_signal']=any(rows[i]['target_signal'] for i in ix)
        rep['in_retail_frame']=any(rows[i]['in_retail_frame'] for i in ix)
        for i in ix:mapping.append({'overture_id':rows[i]['id'],'canonical_id':rep['id'],'merged':i!=pick})
        retained.append(rep)
    report['context_duplicate_records_removed']=len(rows)-len(retained)
    report['context_canonical_rows']=len(retained)
    base_parquet(out/'overture_entity_map.parquet',mapping)
    if pairs:base_parquet(out/'overture_duplicate_pairs.parquet',pairs)
    return retained

In [51]:
def base_project_points(coords):
    c=base_connect();a=base_pa.table({'lon':[x[0] for x in coords],'lat':[x[1] for x in coords]});c.register('a',a)
    result=c.execute("""SELECT ST_X(g),ST_Y(g) FROM (SELECT ST_Transform(ST_Point(lon,lat),'EPSG:4326','EPSG:3310',always_xy:=true) AS g FROM a)""").fetchall();c.close()
    return result

In [52]:
def base_link_osm(rows, osm_path, report, out):
    import csv as base_csv
    raw=list(base_csv.DictReader(osm_path.open()))
    report['osm_candidate_objects']=len(raw)
    eligible=[];excluded=[]
    for r in raw:
        tags=base_json.loads(r['tags_json']);r['_tags']=tags
        why=[]
        if r['coordinate_status']!='located':why.append('unresolved_geometry')
        if not r['name'].strip():why.append('missing_name')
        if not tags.get('shop'):why.append('no_active_shop_tag')
        if tags.get('shop') in ('vacant','closed','no'):why.append('inactive_shop_tag')
        if tags.get('disused')=='yes' or tags.get('abandoned')=='yes' or tags.get('access')=='private':why.append('inactive_or_private')
        if why:
            excluded.append({'osm_key':r['osm_type']+'/'+r['osm_id'],'name':r['name'],'reason':'|'.join(why),'tags_json':r['tags_json']});continue
        try:r['lon']=float(r['longitude']);r['lat']=float(r['latitude'])
        except ValueError:continue
        eligible.append(r)
    xy=base_project_points([(r['lon'],r['lat']) for r in eligible]) if eligible else []
    tree=base_cKDTree(base_np.array([(r['x_m'],r['y_m']) for r in rows]))
    osm_tree_groups=base_defaultdict(list);links=[];added=[];ambiguous=0
    for o,(x,y) in zip(eligible,xy):
        tags=o['_tags'];key=o['osm_type']+'/'+o['osm_id'];nk=base_name_key(o['name'])
        addr=' '.join(filter(None,[tags.get('addr:housenumber'),tags.get('addr:street'),('Unit '+tags['addr:unit']) if tags.get('addr:unit') else None]))
        choices=[];unit_conflicts=[]
        for i in tree.query_ball_point([x,y],100):
            r=rows[i];sim=base_ratio(nk,r['norm_name'])
            dist=base_math.hypot(r['x_m']-x,r['y_m']-y)
            phone=base_re.sub(r'\D','',tags.get('phone',tags.get('contact:phone','')))
            phone_match=bool(phone and len(phone)>=10 and phone[-10:]==base_re.sub(r'\D','',r.get('phone') or '')[-10:])
            address_match=bool(addr and r.get('address') and base_address_key(addr)==base_address_key(r['address']))
            if not base_compatible_address(addr,r.get('address')):
                if sim>=92 and dist<=100:unit_conflicts.append((i,dist,sim))
                continue
            if (sim>=92 and dist<=60) or (sim>=85 and dist<=25) or (phone_match and sim>=50) or (address_match and sim>=70):
                score=sim+15*phone_match+10*address_match-min(dist,100)/10
                choices.append((score,i,dist,sim))
        choices.sort(reverse=True)
        is_ambiguous=len(choices)>1 and choices[0][0]-choices[1][0]<5
        if choices and not is_ambiguous:
            _,i,dist,sim=choices[0];r=rows[i]
            r['source_osm_ids'].append(key);r['osm_records'].append({'osm_key':key,'shop':tags.get('shop'),'tags':tags,'source':base_json.loads(o['source_json'])})
            r['source_provider']='Overture+OSM';r['target_signal']=True
            links.append({'osm_key':key,'overture_canonical_id':r['id'],'distance_m':dist,'name_similarity':sim,'status':'linked','osm_name':o['name'],'overture_name':r['name']})
        elif unit_conflicts and not choices:
            # Preserve contradictory unit/address evidence without asserting one entity
            # or adding a second possible duplicate into the canonical location table.
            for i,dist,sim in unit_conflicts:
                r=rows[i];r['entity_match_ambiguous']=True
                r['source_osm_ids'].append(key)
                r['osm_records'].append({'osm_key':key,'shop':tags.get('shop'),'tags':tags,'source':base_json.loads(o['source_json']),'entity_link_status':'unit_conflict_review'})
                r['source_provider']='Overture+OSM_conflict'
                links.append({'osm_key':key,'overture_canonical_id':r['id'],'distance_m':dist,'name_similarity':sim,'status':'unit_conflict_review','osm_name':o['name'],'overture_name':r['name']})
            ambiguous+=1
        else:
            ambiguous+=int(is_ambiguous)
            # Collapse duplicate node/way representations only with exact normalized name and <=20m.
            matching=next((j for j in osm_tree_groups[nk] if base_math.hypot(added[j]['x_m']-x,added[j]['y_m']-y)<=20 and base_compatible_address(addr,added[j]['address'])),None)
            if matching is not None:
                r=added[matching];r['source_osm_ids'].append(key);r['osm_records'].append({'osm_key':key,'shop':tags.get('shop'),'tags':tags,'source':base_json.loads(o['source_json'])})
            else:
                r={'id':'osm:'+key,'name':o['name'],'category':'osm_shop_'+tags.get('shop','unknown'),'basic_category':None,'hierarchy':['shopping'],
                  'confidence':None,'operating_status':None,'lon':o['lon'],'lat':o['lat'],'address':addr or None,'locality':tags.get('addr:city'),
                  'postcode':tags.get('addr:postcode'),'address_region':tags.get('addr:state'),'address_country':tags.get('addr:country'),
                  'website':tags.get('website',tags.get('contact:website')),'phone':tags.get('phone',tags.get('contact:phone')),
                  'overture_sources_json':None,'in_retail_frame':False,'is_shopping':True,'is_food':False,'is_mall':False,'is_university':False,
                  'target_signal':True,'address_country_conflict':bool(tags.get('addr:country') and tags.get('addr:country') not in ('US','USA')),'x_m':x,'y_m':y,'GEOID':None,'tract_match_count':None,
                  'norm_name':nk,'source_overture_ids':[],'source_osm_ids':[key],'source_provider':'OSM',
                  'osm_records':[{'osm_key':key,'shop':tags.get('shop'),'tags':tags,'source':base_json.loads(o['source_json'])}],
                  'overture_duplicate_count':0,'entity_match_ambiguous':is_ambiguous,
                  'geometry_requires_review':o['geometry_method'].startswith('open_way')}
                osm_tree_groups[nk].append(len(added));added.append(r)
            links.append({'osm_key':key,'overture_canonical_id':None,'distance_m':None,'name_similarity':None,'status':'ambiguous_review' if is_ambiguous else 'unmatched_osm_candidate','osm_name':o['name'],'overture_name':None})
    rows.extend(added)
    report['osm_source_links']=sum(len(r['source_osm_ids']) for r in rows if r['source_provider']=='Overture+OSM')
    report['osm_only_canonical_candidates']=len(added)
    report['osm_ambiguous_match_objects']=ambiguous
    report['osm_exclusions']=dict(base_Counter(r['reason'] for r in excluded))
    if links:base_parquet(out/'osm_overture_links.parquet',links)
    if excluded:base_parquet(out/'osm_excluded_objects.parquet',excluded)
    return rows

In [53]:
def base_label(r, overrides):
    override=next((overrides[k] for k in r['source_overture_ids'] if k in overrides),None)
    if override:
        d=override['decision'];r['override_evidence_url']=override.get('evidence_url');r['override_reason']=override.get('reason')
        if d=='exclude':return None,'excluded_verified_nonretail',None
        if d=='target_verified':return 1,'target_verified_sample_audit',1
        if d=='comparison_verified':return 0,'comparison_verified_sample_audit',0
        if d=='ambiguous':return None,'manual_review_required',None
    n=r['name'] or '';cat=r['category'] or ''
    if base_ADULT.search(n):return None,'excluded_adult_signal',None
    if r.get('entity_match_ambiguous'):return None,'ambiguous_entity_match',None
    if r.get('geometry_requires_review'):return None,'osm_geometry_review',None
    if base_ONLINE.search(n) and not r['source_osm_ids']:return None,'online_or_wholesale_review',None
    if base_AMBIGUOUS_BRANDS.search(n):return None,'mixed_merchandise_or_concession_review',None
    osm_shops={o['shop'] for o in r['osm_records']}
    strict=cat in overture_STRICT_TARGET_PROXY
    strong=bool(base_STRONG_NAME.search(n))
    osm_target=bool(osm_shops & {'toys','model'})
    # TCG/LEGO also occur in architects, advisors, contractors and attractions.
    # A name alone does not establish a physical retail destination.
    if r['target_signal'] and not r['is_shopping'] and not osm_shops:
        return None,'excluded_nonretail_name_match',None
    if strict or strong or osm_target:
        return 1,('cross_source_target_candidate' if r['source_provider']=='Overture+OSM' else 'mapped_specialist_candidate'),None
    if r['target_signal'] or cat in base_PENDING_PLUS or osm_shops & {'games','collector'}:
        return None,'ambiguous_specialist_review',None
    if r['in_retail_frame']:return 0,'comparison_category_proxy',None
    return None,'outside_sample_frame',None

In [54]:
def base_feature_counts(master, context, report):
    xy=base_np.array([(r['x_m'],r['y_m']) for r in master])
    for kind,predicate in [
      ('other_retail',lambda r:r['is_shopping'] and not r['target_signal'] and not r.get('excluded',False)),
      ('food',lambda r:r['is_food']),
    ]:
        cohort=[r for r in context if predicate(r)];ids={r['id'] for r in cohort}
        tree=base_cKDTree(base_np.array([(r['x_m'],r['y_m']) for r in cohort]))
        own=base_np.array([int(r['id'] in ids) for r in master])
        for radius in [500,1000]:
            cnt=tree.query_ball_point(xy,radius,return_length=True,workers=2)-own
            assert base_np.all(cnt>=0)
            for r,value in zip(master,cnt):r[f'{kind}_count_{radius}m']=int(value)
        report[f'context_{kind}_unique_locations']=len(cohort)
    for kind,predicate in [('mall',lambda r:r['is_mall']),('university',lambda r:r['is_university'])]:
        cohort=[r for r in context if predicate(r)]
        tree=base_cKDTree(base_np.array([(r['x_m'],r['y_m']) for r in cohort]));distance,_=tree.query(xy,workers=2)
        for r,d in zip(master,distance):r['distance_to_'+kind+'_m']=float(d)
        report[f'context_{kind}_unique_locations']=len(cohort)
    targets=[r for r in master if r['target_label_proxy']==1]
    if targets:
        tree=base_cKDTree(base_np.array([(r['x_m'],r['y_m']) for r in targets]));target_ids={r['id'] for r in targets}
        own=base_np.array([int(r['id'] in target_ids) for r in master])
        for radius in [500,1000]:
            cnt=tree.query_ball_point(xy,radius,return_length=True,workers=2)-own
            for r,n in zip(master,cnt):r[f'other_target_proxy_count_{radius}m']=int(n)
        ds,_=tree.query(xy,k=2 if len(targets)>1 else 1,workers=2)
        for i,r in enumerate(master):
            d=ds[i,1] if len(targets)>1 and own[i] else (ds[i,0] if len(targets)>1 else ds[i])
            r['distance_to_other_target_proxy_m']=float(d) if base_np.isfinite(d) and not (len(targets)==1 and own[i]) else None
    report['target_proxy_count_before_demographic_eligibility']=len(targets)

In [55]:
def base_final_join(master, census_path, tract_path, processed, report):
    # All rows undergo the same exact California polygon and ACS join.
    table=base_pa.Table.from_pylist(master);c=base_connect();c.register('master',table)
    c.execute(f'CREATE VIEW tracts AS SELECT GEOID,geometry FROM read_parquet({base_q(tract_path)})')
    c.execute(f'CREATE VIEW acs AS SELECT * FROM read_parquet({base_q(census_path)})')
    features=['acs_period','population','under18_share','young_adult_18_34_share','median_household_income_2024_usd','median_household_income_moe',
      'college_enrolled_share_3plus','households','family_household_share','living_alone_household_share',
      'county_geoid','county_block_id','is_bay_area_9county','land_area_km2','population_density_per_km2',
      'B01001_001M','B14001_001M','B11001_001M']
    c.execute("""CREATE TEMP TABLE matched AS SELECT * EXCLUDE(choice) FROM (
      SELECT m.* EXCLUDE(GEOID,tract_match_count),t.GEOID AS tract_geoid,
        count(t.GEOID) OVER(PARTITION BY m.id) AS tract_match_count,
        row_number() OVER(PARTITION BY m.id ORDER BY t.GEOID NULLS LAST) AS choice
      FROM master m LEFT JOIN tracts t ON ST_Covers(t.geometry,ST_Point(m.lon,m.lat))
    ) WHERE choice=1""")
    report['retail_before_exact_california_clip']=len(master)
    report['retail_outside_california_removed']=c.execute('SELECT count(*) FROM matched WHERE tract_geoid IS NULL').fetchone()[0]
    report['retail_CA_coordinate_foreign_address_removed']=c.execute('SELECT count(*) FROM matched WHERE tract_geoid IS NOT NULL AND address_country_conflict').fetchone()[0]
    sel=','.join('a.'+f for f in features)
    c.execute(f"""CREATE TEMP TABLE joined AS SELECT m.*,{sel},a.GEOID IS NOT NULL AS acs_matched,
      a.median_household_income_2024_usd IS NULL AS income_missing,
      a.median_household_income_moe IS NULL AS income_moe_missing,
      (a.median_household_income_moe / nullif(a.median_household_income_2024_usd,0)) AS income_relative_moe
      FROM matched m LEFT JOIN acs a ON m.tract_geoid=a.GEOID
      WHERE m.tract_geoid IS NOT NULL AND NOT m.address_country_conflict""")
    c.execute("""CREATE TEMP TABLE final AS SELECT *,
      coalesce(acs_matched AND population>0 AND land_area_km2>0
      AND under18_share IS NOT NULL AND young_adult_18_34_share IS NOT NULL
      AND median_household_income_2024_usd>0
      AND college_enrolled_share_3plus IS NOT NULL AND living_alone_household_share IS NOT NULL
      AND target_label_proxy IS NOT NULL,false) AS proxy_model_eligible
      FROM joined""")
    for name,query in [('california_retail_master','SELECT * FROM final ORDER BY tract_geoid,id'),
                       ('california_model_proxy_cohort','SELECT * FROM final WHERE proxy_model_eligible ORDER BY tract_geoid,id')]:
        c.execute(f'COPY ({query}) TO ? (FORMAT PARQUET,COMPRESSION ZSTD)',[str(processed/(name+'.parquet'))])
        c.execute(f'COPY ({query}) TO ? (FORMAT CSV,HEADER TRUE)',[str(processed/(name+'.csv'))])
    c.execute("""CREATE TEMP TABLE tract_summary AS
      SELECT a.*,coalesce(s.retail_locations,0) AS retail_locations,
        coalesce(s.target_proxy_locations,0) AS target_proxy_locations,
        coalesce(s.proxy_model_eligible_rows,0) AS proxy_model_eligible_rows,
        coalesce(s.target_review_rows,0) AS target_review_rows,
        coalesce(s.osm_linked_locations,0) AS osm_linked_locations
      FROM acs a LEFT JOIN (
        SELECT tract_geoid,count(*) AS retail_locations,
          count(*) FILTER(WHERE target_label_proxy=1) AS target_proxy_locations,
          count(*) FILTER(WHERE proxy_model_eligible) AS proxy_model_eligible_rows,
          count(*) FILTER(WHERE target_label_proxy IS NULL) AS target_review_rows,
          count(*) FILTER(WHERE has_osm_evidence) AS osm_linked_locations
        FROM final GROUP BY tract_geoid
      ) s ON a.GEOID=s.tract_geoid
    """)
    for fmt in ['PARQUET','CSV']:
        extra=',COMPRESSION ZSTD' if fmt=='PARQUET' else ',HEADER TRUE'
        c.execute(f'COPY (SELECT * FROM tract_summary ORDER BY GEOID) TO ? (FORMAT {fmt}{extra})',[str(processed/('california_tract_summary.'+fmt.lower()))])
    report['master_rows']=c.execute('SELECT count(*) FROM final').fetchone()[0]
    report['master_columns']=len(c.execute('DESCRIBE final').fetchall())
    report['proxy_model_eligible_rows']=c.execute('SELECT count(*) FROM final WHERE proxy_model_eligible').fetchone()[0]
    report['candidate_proxy_row_count_threshold_met']=report['proxy_model_eligible_rows']>=50000
    report['cleaned_row_threshold_50000_met_for_proxy_cohort']=None
    report['verified_retail_row_count']=None
    report['retail_scope_audit_status']='NOT_FINAL_CLEAN_DATA_RUN_RETAIL_SCOPE_V2_AUDIT'
    report['retail_scope_qualification']='Counts describe rule-selected mapped candidates; retail scope and target labels are not universally verified.'
    report['unique_master_ids']=c.execute('SELECT count(DISTINCT id) FROM final').fetchone()[0]
    report['acs_unmatched_in_master']=c.execute('SELECT count(*) FROM final WHERE NOT acs_matched').fetchone()[0]
    report['master_counties']=c.execute('SELECT count(DISTINCT county_geoid) FROM final').fetchone()[0]
    report['master_tracts']=c.execute('SELECT count(DISTINCT tract_geoid) FROM final').fetchone()[0]
    report['bay_area_rows']=c.execute('SELECT count(*) FROM final WHERE is_bay_area_9county').fetchone()[0]
    report['label_status_counts']=[dict(zip(['status','rows'],r)) for r in c.execute('SELECT label_status,count(*) FROM final GROUP BY 1 ORDER BY 2 DESC').fetchall()]
    report['proxy_label_counts']=[dict(zip(['label','rows'],r)) for r in c.execute('SELECT target_label_proxy,count(*) FROM final GROUP BY 1 ORDER BY 1').fetchall()]
    report['proxy_cohort_label_counts']=[dict(zip(['label','rows'],r)) for r in c.execute('SELECT target_label_proxy,count(*) FROM final WHERE proxy_model_eligible GROUP BY 1 ORDER BY 1').fetchall()]
    report['source_provider_counts']=[dict(zip(['source','rows'],r)) for r in c.execute('SELECT source_provider,count(*) FROM final GROUP BY 1 ORDER BY 2 DESC').fetchall()]
    report['master_missingness']={f:c.execute(f'SELECT count(*) FROM final WHERE {f} IS NULL').fetchone()[0] for f in features}
    assert report['master_rows']==report['unique_master_ids']
    assert report['acs_unmatched_in_master']==0
    assert c.execute("SELECT count(*) FROM final WHERE NOT regexp_full_match(tract_geoid,'06[0-9]{9}')").fetchone()[0]==0
    assert c.execute('SELECT count(*) FROM final WHERE other_retail_count_500m>other_retail_count_1000m OR food_count_500m>food_count_1000m OR other_target_proxy_count_500m>other_target_proxy_count_1000m').fetchone()[0]==0
    assert c.execute('SELECT count(*) FROM tract_summary').fetchone()[0]==9129
    report['assertions']={'unique_master_id':True,'all_master_points_ca':True,'ACS_join_not_multiplying_rows':True,'nested_radius_count_monotonicity':True,'tract_output_rows_9129':True,'no_global_imputation':True}
    c.execute("COPY (SELECT * FROM final WHERE target_label_proxy=1 OR target_label_proxy IS NULL ORDER BY label_status,name) TO ? (FORMAT CSV,HEADER TRUE)",[str(processed/'target_label_review_queue.csv')])
    c.execute('COPY (SELECT * FROM final ORDER BY id LIMIT 100) TO ? (FORMAT CSV,HEADER TRUE)',[str(processed/'master_preview_100.csv')])
    schema=c.execute('DESCRIBE final').fetchall();c.close()
    return schema

In [56]:
def base_main():
    ap=base_argparse.ArgumentParser(description=base___doc__)
    ap.add_argument('--points',type=base_Path,default=base_ROOT/'data/raw/overture_ca/points.parquet')
    ap.add_argument('--osm',type=base_Path,default=base_ROOT/'data/interim/osm/toy_candidates.csv')
    ap.add_argument('--tracts',type=base_Path,default=base_ROOT/'data/interim/census/ca_tracts_2024.parquet')
    ap.add_argument('--acs',type=base_Path,default=base_ROOT/'data/interim/census/ca_acs2024_tract_features.parquet')
    ap.add_argument('--overrides',type=base_Path,default=base_ROOT/'pipeline/label_overrides.json')
    ap.add_argument('--output-root',type=base_Path,default=base_ROOT/'data')
    args=ap.parse_args();started=base_time.monotonic()
    interim=args.output_root/'interim/integration';processed=args.output_root/'processed'
    interim.mkdir(parents=True,exist_ok=True);processed.mkdir(parents=True,exist_ok=True)
    report={'complete':False,'started_at_utc':base_datetime.now(base_timezone.utc).isoformat(),'unit':'one canonical mapped retail candidate; one current snapshot, not tract-year',
      'source_versions':{'Overture':'2026-10-21.0','OSM':'2026-10-24','TIGER':'2024','ACS':'2020-2024 five-year estimates'},
      'label_warning':'target_label_proxy is rule-based and source-corroborated, not universally manually verified. Verified label is separately nullable.',
      'input_paths':{k:str(getattr(args,k).resolve()) for k in ['points','osm','tracts','acs']}}
    base_js(processed/'dataset_manifest.json',report)
    base_js(base_ROOT/'validation/california_master_qa.json',report)
    base_log('1/6 Flatten relevant context and match California polygons')
    rows=base_prepare_context(args.points,args.tracts,interim,report)
    base_log(f'2/6 Conservative near-duplicate resolution on {len(rows):,} context points')
    rows=base_deduplicate(rows,report,interim)
    base_log('3/6 Resolve OSM objects to Overture entities; retain provenance')
    rows=base_link_osm(rows,args.osm,report,interim)
    overrides={}
    if args.overrides.exists():
        overrides={r['id']:r for r in base_json.loads(args.overrides.read_text())['overrides']}
    master=[];excluded=[]
    for r in rows:
        r['target_label_proxy'],r['label_status'],r['is_target_verified']=base_label(r,overrides)
        if r['label_status']=='comparison_verified_sample_audit':r['target_signal']=False
        r['excluded']=r['label_status'].startswith('excluded_')
        r['has_osm_evidence']=bool(r['source_osm_ids'])
        r['source_osm_json']=base_json.dumps(r['osm_records'],ensure_ascii=False)
        r['overture_ids_json']=base_json.dumps(r['source_overture_ids'])
        r['osm_ids_json']=base_json.dumps(r['source_osm_ids'])
        if r['excluded']:
            excluded.append({'id':r['id'],'name':r['name'],'label_status':r['label_status'],'reason':r.get('override_reason'),'evidence_url':r.get('override_evidence_url')});continue
        if r['in_retail_frame'] or r['target_signal']:
            master.append(r)
    if excluded:base_parquet(interim/'excluded_canonical_records.parquet',excluded)
    report['excluded_canonical_records']=len(excluded)
    context_audit=[]
    for r in rows:
        context_audit.append({'id':r['id'],'name':r['name'],'lon':r['lon'],'lat':r['lat'],'x_m':r['x_m'],'y_m':r['y_m'],
          'is_other_retail_context':bool(r['is_shopping'] and not r['target_signal'] and not r.get('excluded',False)),
          'is_food_context':bool(r['is_food']),'is_mall_context':bool(r['is_mall']),'is_university_context':bool(r['is_university']),
          'target_label_proxy':r['target_label_proxy'],'source_provider':r['source_provider']})
    base_parquet(interim/'context_canonical.parquet',context_audit)
    base_log(f'4/6 Calculate metric local context for {len(master):,} candidate locations')
    base_feature_counts(master,rows,report)
    # Drop transient Python structures; keep scalar audit fields in the master.
    for r in master:
        for col in ['source_overture_ids','source_osm_ids','osm_records','norm_name','hierarchy','excluded']:
            r.pop(col,None)
        r.setdefault('entity_match_ambiguous',False)
        r.setdefault('geometry_requires_review',False)
        r.setdefault('override_evidence_url',None);r.setdefault('override_reason',None)
    base_log('5/6 Exact state clip and keyed ACS join; write tables')
    schema=base_final_join(master,args.acs,args.tracts,processed,report)
    report['elapsed_seconds']=round(base_time.monotonic()-started,2)
    report['completed_at_utc']=base_datetime.now(base_timezone.utc).isoformat()
    report['complete']=True
    report['not_done']=['exhaustive manual target/storefront verification','retailer profitability or vacancy measurement','model training and candidate opportunity ranking']
    base_js(base_ROOT/'validation/california_master_qa.json',report)
    base_js(processed/'dataset_manifest.json',report)
    base_log('6/6 Complete: '+base_json.dumps({k:report[k] for k in ['master_rows','master_columns','proxy_model_eligible_rows','bay_area_rows','elapsed_seconds']}))
    base_js(processed/'master_schema.json',[{'field':r[0],'type':r[1]} for r in schema])

In [57]:
with notebook_args('--points',overture_out/'points.parquet',
 '--osm',osm_dir/'toy_candidates.csv','--tracts',RUN_ROOT/'data/interim/census/ca_tracts_2024.parquet',
 '--acs',RUN_ROOT/'data/interim/census/ca_acs2024_tract_features.parquet',
 '--overrides',RUN_ROOT/'pipeline/label_overrides.json','--output-root',RUN_ROOT/'data'):
    base_main()
base_path=RUN_ROOT/'data/processed/california_retail_master.parquet'
show_counts(base_path,'Raw-source integrated candidate table')
archive_path=RUN_ROOT/'data/archive/v1_pre_retail_scope_review/california_retail_master.parquet'
archive_path.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(base_path,archive_path)
gc.collect()

1/6 Flatten relevant context and match California polygons


/var/folders/17/2v0v148x18v8nn3pttyqhn5w0000gn/T/ipykernel_30850/1876137164.py:54: UserWarning: Arrow output on a DB-API result makes an extra copy of the rows, use con.sql(query).to_arrow_table() instead.
  rows=c.execute('SELECT * FROM spatially_tagged ORDER BY id').to_arrow_table().to_pylist()


2/6 Conservative near-duplicate resolution on 370,958 context points


3/6 Resolve OSM objects to Overture entities; retain provenance


4/6 Calculate metric local context for 90,899 candidate locations


5/6 Exact state clip and keyed ACS join; write tables


6/6 Complete: {"master_rows": 81494, "master_columns": 72, "proxy_model_eligible_rows": 75037, "bay_area_rows": 14937, "elapsed_seconds": 19.74}


Raw-source integrated candidate table : 81,494 rows; 81,494 unique IDs


40

## 6. 业态核验、标签未知与独立OSM佐证

将全shop OSM与主表匹配，区分商品零售、服务、生命周期、地址／单元号冲突；应用已保存的审核表。未知标签保持未知。重算V2背景特征并保留所有隔离原因。

函数／常量加阶段前缀，避免同一内核中的同名变量相互覆盖。

In [58]:
# Source: pipeline/audit_retail_scope_v2.py; SHA256: 7117dab52254ec5edd15c746520e7608cfd344d27f97c5409d3f846de0cbbdba
scope___file__ = str(RUN_ROOT / 'pipeline/audit_retail_scope_v2.py')
scope___doc__ = 'Audit retail scope separately from target labels. Never asserts full verification.\n\nRetains V1 rows for traceability, quarantines risk signals rather than calling\nthem proven errors, and adds independently mapped OSM shop evidence.'
"""Audit retail scope separately from target labels. Never asserts full verification.

Retains V1 rows for traceability, quarantines risk signals rather than calling
them proven errors, and adds independently mapped OSM shop evidence.
"""
from pathlib import Path as scope_Path
from collections import Counter as scope_Counter, defaultdict as scope_defaultdict
from datetime import datetime as scope_datetime, timezone as scope_timezone
import argparse as scope_argparse
import csv as scope_csv
import hashlib as scope_hashlib
import json as scope_json
import math as scope_math
import re as scope_re

import duckdb as scope_duckdb
import numpy as scope_np
import pyarrow as scope_pa
import pyarrow.parquet as scope_pq
from rapidfuzz.fuzz import ratio as scope_ratio
from scipy.spatial import cKDTree as scope_cKDTree


scope_ROOT=scope_Path(scope___file__).resolve().parents[1]
scope_PATTERNS={
 'instruction_or_education':r'\b(?:lessons?|tutor(?:ing)?|academy|academies|education|preschool|kindergarten|learning center|dance studios?|music school|driving school|training center)\b',
 'repair_services':r'\b(?:repair|repairs|repairing|smog|auto service|tech support)\b',
 'wholesale_or_manufacturing':r'\b(?:wholesale|distribut(?:or|ion|ors)|manufactur(?:er|ing|ers)|industrial|supply chain|b2b)\b',
 'professional_or_personal_services':r'\b(?:consulting|consultants?|advisors?|attorneys?|law office|architects?|construction|medical center|clinic|optometr(?:y|ist)|dental|insurance|real estate|salon|spa|massage)\b',
 'online_or_home_services':r'\b(?:in[ -]home|at[ -]home|home[ -]based|online only|online store|ecommerce|e-commerce|virtual)\b|\.com\b',
 'rentals_or_events':r'\b(?:rental|rentals|party entertainment|event planning|photo booth|photobooth|dj service)\b',
 'studio_unspecified':r'\bstudio(?:s)?\b',
 'shipping_postal_services':r'\b(?:shipping|mailbox|postal|pack[ -]?(?:and|&)[ -]?ship)\b',
}
scope_TARGET_FORMAT_PATTERNS={
 'toysrus_concession_or_mixed_format':r'\btoys[\W_]*r[\W_]*us\b',
 'vending_or_robo_format':r'robo[ -]?shop|\bvending\b',
 'possible_ticketed_park_format':r"\bpooh corner\b|it's a small world.*toy|lego movie.*4d",
 'numismatic_target_scope':r'\b(?:coins?|stamps?|numismatic|currency)\b',
}
# Explicit shop-tag evidence of goods retail. Every item is a scope decision,
# not a claim that shop=* always denotes retail (it also includes services).
scope_OSM_GOODS={
 'clothes','jewelry','shoes','mobile_phone','electronics','computer','department_store','variety_store',
 'supermarket','convenience','gift','books','toys','games','model','collector','antiques','second_hand',
 'charity','cosmetics','beauty_supply','hair_supply','florist','pet','sports','bicycle','musical_instrument',
 'music','video_games','craft','art','stationery','office_supplies','camera','optician','sunglasses',
 'furniture','houseware','interior_decoration','bag','fashion_accessories','boutique','baby_goods',
 'fabric','sewing','watches','outdoor','fishing','hunting','ski','scuba_diving','surf','skate',
 'golf','trophy','costume','party','wigs','perfumery','bed','leather','kiosk','confectionery',
 'bakery','pastry','deli','greengrocer','nutrition_supplements','alcohol','wine','hardware','doityourself',
 'paint','garden_centre','agrarian','pawnbroker','pawn','car_parts',
}
scope_OSM_SCOPE_REVIEW={'wholesale','trade','yes','mall','retail','industrial','warehouse'}
scope_OSM_SERVICE={'hairdresser','beauty','car_repair','laundry','dry_cleaning','massage','tattoo','pet_grooming',
 'travel_agency','money_lender','insurance','estate_agent','ticket','copyshop','car_rental','storage_rental',
 'rental','funeral_directors','locksmith','tailor','shoe_repair','repair','computer_repair','hearing_aids'}
# Incompatible with the intended *consumer discretionary retail* comparison
# frame, even where the underlying business might genuinely sell goods.
scope_CATEGORY_REVIEW={'cooking_classes','cafe','legal_service','auto_dealer','automotive_repair','warehouse',
 'shopping','specialty_store','building_supply_store','home_improvement_store','flea_market','warehouse_club_store',
 'produce_store','convenience_store','specialty_foods_store'}

In [59]:
def scope_write_json(path,value):
 path.parent.mkdir(parents=True,exist_ok=True);tmp=path.with_suffix(path.suffix+'.partial')
 tmp.write_text(scope_json.dumps(value,ensure_ascii=False,indent=2,default=str));tmp.replace(path)

In [60]:
def scope_risks(name,category,address):
 out=[k for k,p in scope_PATTERNS.items() if scope_re.search(p,name or '',scope_re.I)]
 if category in scope_CATEGORY_REVIEW:out.append('category_scope_requires_review')
 if not address or not address.strip():out.append('missing_street_address')
 return out

In [61]:
def scope_read_manual(path):
 if not path.exists():return []
 obj=scope_json.loads(path.read_text())
 if isinstance(obj,list):return obj
 return obj.get('overrides',obj.get('records',[]))

In [62]:
def scope_project(rows):
 c=scope_duckdb.connect();c.execute('LOAD spatial')
 c.register('xy',scope_pa.table({'lon':[r['lon'] for r in rows],'lat':[r['lat'] for r in rows]}))
 result=c.sql("SELECT ST_X(g) x,ST_Y(g) y FROM (SELECT ST_Transform(ST_Point(lon,lat),'EPSG:4326','EPSG:3310',always_xy:=true) g FROM xy)").to_arrow_table()
 c.close();return scope_np.column_stack([result['x'].to_numpy(),result['y'].to_numpy()])

In [63]:
def scope_match_osm(master,osm_path,out):
 rows=[];excluded=scope_Counter()
 with osm_path.open() as f:
  for r in scope_csv.DictReader(f):
   t=scope_json.loads(r['tags_json']);shop=t.get('shop','').strip().casefold()
   if r['coordinate_status']!='located':excluded['unlocated']+=1;continue
   if not r['name'].strip():excluded['unnamed']+=1;continue
   if shop in ('','no','vacant','closed','disused','abandoned','empty'):excluded['inactive_or_no_active_shop']+=1;continue
   if r['geometry_method'].startswith('open_way'):excluded['open_way_geometry_review']+=1;continue
   if t.get('access')=='private' or any(t.get(k,'').casefold()=='yes' for k in ['disused','abandoned','closed']) or t.get('opening_hours','').strip().casefold()=='closed':excluded['private_or_inactive']+=1;continue
   try:lon,lat=float(r['longitude']),float(r['latitude'])
   except ValueError:excluded['invalid_coords']+=1;continue
   if not scope_math.isfinite(lon+lat):excluded['invalid_coords']+=1;continue
   addr=' '.join(filter(None,[t.get('addr:housenumber'),t.get('addr:street'),('Unit '+t['addr:unit']) if t.get('addr:unit') else None]))
   phone=scope_re.sub(r'\D','',t.get('phone',t.get('contact:phone','')))
   lifecycle=any(k.startswith(('disused:shop','abandoned:shop','demolished:shop','was:shop','removed:shop','closed:shop')) for k in t)
   rows.append({'key':r['osm_type']+'/'+r['osm_id'],'name':r['name'],'nk':base_name_key(r['name']),
    'shop':shop,'lon':lon,'lat':lat,'addr':addr,'phone':phone,'geometry_method':r['geometry_method'],
    'lifecycle_conflict':lifecycle})
 xy=scope_project(rows);tree=scope_cKDTree(scope_np.array([(r['x_m'],r['y_m']) for r in master]))
 for r in master:
  r['_nk']=base_name_key(r['name']);r['_phone']=scope_re.sub(r'\D','',r['phone'] or '')
 by_id=scope_defaultdict(list);links=[]
 for o,point in zip(rows,xy):
  candidates=[];conflicts=[]
  for ix in tree.query_ball_point(point,75):
   r=master[ix];sim=scope_ratio(o['nk'],r['_nk']);d=float(scope_np.linalg.norm(point-[r['x_m'],r['y_m']]))
   pm=bool(len(o['phone'])>=10 and len(r['_phone'])>=10 and o['phone'][-10:]==r['_phone'][-10:])
   am=bool(o['addr'] and r['address'] and base_address_key(o['addr'])==base_address_key(r['address']))
   if not base_compatible_address(o['addr'],r['address']):
    if sim>=92:conflicts.append((ix,d,sim))
    continue
   if (sim>=92 and d<=60) or (sim>=85 and d<=25) or (pm and sim>=70 and d<=60) or (am and sim>=75 and d<=60):
    candidates.append((sim+15*pm+10*am-d/10,ix,d,sim,pm,am))
  candidates.sort(reverse=True)
  if candidates and (len(candidates)==1 or candidates[0][0]-candidates[1][0]>=5):
   _,ix,d,sim,pm,am=candidates[0];mid=master[ix]['id']
   evidence={'master_id':mid,'osm_key':o['key'],'osm_name':o['name'],'osm_shop':o['shop'],
    'distance_m':d,'name_similarity':sim,'phone_match':pm,'address_match':am,'link_status':'accepted',
    'osm_goods_tag':o['shop'] in scope_OSM_GOODS,'osm_service_tag':o['shop'] in scope_OSM_SERVICE,
    'osm_lifecycle_conflict':o['lifecycle_conflict'],'osm_scope_review':o['shop'] in scope_OSM_SCOPE_REVIEW}
   links.append(evidence);by_id[mid].append(evidence)
  elif conflicts and not candidates:
   for ix,d,sim in conflicts:
    evidence={'master_id':master[ix]['id'],'osm_key':o['key'],'osm_name':o['name'],'osm_shop':o['shop'],
     'distance_m':d,'name_similarity':sim,'phone_match':False,'address_match':False,'link_status':'address_unit_conflict',
     'osm_goods_tag':False,'osm_service_tag':False,
     'osm_lifecycle_conflict':o['lifecycle_conflict'],'osm_scope_review':o['shop'] in scope_OSM_SCOPE_REVIEW}
    links.append(evidence);by_id[master[ix]['id']].append(evidence)
  elif candidates:
   for _,ix,d,sim,pm,am in candidates[:2]:
    evidence={'master_id':master[ix]['id'],'osm_key':o['key'],'osm_name':o['name'],'osm_shop':o['shop'],
     'distance_m':d,'name_similarity':sim,'phone_match':pm,'address_match':am,'link_status':'multiple_match_review',
     'osm_goods_tag':False,'osm_service_tag':False,
     'osm_lifecycle_conflict':o['lifecycle_conflict'],'osm_scope_review':o['shop'] in scope_OSM_SCOPE_REVIEW}
    links.append(evidence);by_id[master[ix]['id']].append(evidence)
 scope_pq.write_table(scope_pa.Table.from_pylist(links),out/'osm_allshops_entity_links.parquet',compression='zstd')
 return by_id,{'osm_objects_considered':len(rows),'osm_exclusion_counts':dict(excluded),
  'link_rows':len(links),'link_status_counts':dict(scope_Counter(r['link_status'] for r in links)),
  'master_ids_with_any_osm_link_evidence':len(by_id)}

In [64]:
def scope_decide(r, links, manual):
 flags=scope_risks(r['name'],r['category'],r['address'])
 formatflags=[k for k,p in scope_TARGET_FORMAT_PATTERNS.items() if scope_re.search(p,r['name'] or '',scope_re.I)]
 accepted=[e for e in links if e['link_status']=='accepted']
 osm_goods=any(e['osm_goods_tag'] for e in accepted)
 osm_service=any(e['osm_service_tag'] for e in accepted)
 conflicts=any(e['link_status']!='accepted' for e in links)
 if osm_service:flags.append('osm_service_tag_conflict')
 if any(e['osm_lifecycle_conflict'] for e in accepted):flags.append('osm_lifecycle_conflict')
 if any(e['osm_scope_review'] for e in accepted):flags.append('osm_scope_requires_review')
 if conflicts or r['entity_match_ambiguous']:flags.append('entity_or_address_conflict')
 for old in scope_json.loads(r['source_osm_json'] or '[]'):
  t=old.get('tags',{})
  oa=' '.join(filter(None,[t.get('addr:housenumber'),t.get('addr:street'),('Unit '+t['addr:unit']) if t.get('addr:unit') else None]))
  if not base_compatible_address(oa,r['address']):flags.append('legacy_osm_address_conflict')
 if r['geometry_requires_review']:flags.append('geometry_review')
 old_official=r['is_target_verified'] is not None
 has_overture=bool(scope_json.loads(r['overture_ids_json'] or '[]'))
 scope='review_required' if flags else ('osm_goods_corroborated' if osm_goods and has_overture else 'mapped_retail_candidate')
 retail_verified=False;target=r['target_label_proxy'];target_verified=r['is_target_verified'];url=r['override_evidence_url']
 reasons=[]
 if old_official:
  scope='official_address_confirmed';retail_verified=True;reasons.append('previous_official_audit')
 for m in manual:
  sd=m.get('retail_scope_decision',m.get('decision'))
  td=m.get('target_decision')
  if sd=='nonretail_confirmed':scope='official_scope_exclusion';retail_verified=False;target=None;target_verified=None
  elif sd=='retailer_confirmed':
   if m.get('verification_level')=='business_model_only_location_not_revalidated':scope='review_required';retail_verified=False
   else:scope='official_address_confirmed';retail_verified=True
  elif sd=='ambiguous':scope='review_required';retail_verified=False
  if td=='target_confirmed':target=1;target_verified=1
  elif td=='comparator_confirmed':target=0;target_verified=0
  elif td=='ambiguous':target=None;target_verified=None
  if m.get('evidence_url'):url=m['evidence_url']
  reasons.append(m.get('reason','manual_audit'))
 # Independent toy signal contradicting an old comparator is not silently relabeled.
 if r['target_label_proxy']==0 and any(e['osm_shop'] in {'toys','model','games','collector'} for e in accepted) and target_verified is None:
  target=None;formatflags.append('opposite_target_signal_in_osm')
 if flags and not retail_verified:target=None
 if formatflags and target_verified is None:target=None
 if scope=='official_scope_exclusion':target=None;target_verified=None
 if not retail_verified:target_verified=None
 if r['target_label_proxy']==1 and osm_service and target_verified is None:target=None
 return {'id':r['id'],'retail_scope_status_v2':scope,'retail_scope_officially_verified_v2':retail_verified,
  'retail_scope_flags_json':scope_json.dumps(sorted(set(flags))),
  'target_format_flags_json':scope_json.dumps(sorted(set(formatflags))),
  'osm_goods_tag_evidence_v2':osm_goods,'osm_goods_corroborated_v2':bool(osm_goods and has_overture),'osm_service_conflict_v2':osm_service,
  'osm_scope_evidence_json':scope_json.dumps(links,ensure_ascii=False),
  'scope_evidence_url_v2':url,'scope_review_reason_v2':' | '.join(reasons) or None,
  'target_label_proxy_v2':target,'target_label_official_verified_v2':target_verified,
  'operating_status_evidence_v2':'listed_open' if r['operating_status']=='open' else 'source_status_unknown',
  'manually_reviewed_v2':bool(manual or old_official)}

In [65]:
def scope_rebuild_context(master, audits, output, report):
 """Recompute affected features; keep V1 fields only for traceability.

 Border context is retained, including mapped targets outside the exact state
 polygon. A quarantined business is omitted from these candidate counts; this
 is a sensitivity definition, not proof of absence in the physical world.
 """
 c=scope_duckdb.connect()
 ctx=c.sql(f"""SELECT x.*,p.category,p.address FROM read_parquet({base_q(scope_ROOT/'data/interim/integration/context_canonical.parquet')}) x
  LEFT JOIN read_parquet({base_q(scope_ROOT/'data/interim/integration/context_pre_dedup.parquet')}) p USING(id)""").to_arrow_table().to_pylist()
 c.close();amap={r['id']:r for r in audits};mmap={r['id']:r for r in master}
 goodscopes={'official_address_confirmed','osm_goods_corroborated','mapped_retail_candidate'}
 other=[];targets=[];annotations=[]
 for r in ctx:
  a=amap.get(r['id']);mr=mmap.get(r['id'])
  # Context includes genuine goods retail (e.g. groceries and hardware) even
  # when it lies outside the discretionary comparison sampling frame.
  flags=scope_risks(r['name'],None,r['address'])
  if r['category'] in {'cooking_classes','cafe','legal_service','auto_dealer','automotive_repair','warehouse','shopping','specialty_store'}:
   flags.append('context_category_requires_review')
  if mr:flags=scope_risks(r['name'],mr['category'],mr['address'])
  if a:
   keep=a['retail_scope_status_v2'] in goodscopes
   isother=bool(keep and a['target_label_proxy_v2']==0)
   istarget=bool(keep and a['target_label_proxy_v2']==1)
  else:
   keep=not flags
   isother=bool(r['is_other_retail_context'] and keep)
   istarget=bool(r['target_label_proxy']==1 and keep and not any(scope_re.search(p,r['name'] or '',scope_re.I) for p in scope_TARGET_FORMAT_PATTERNS.values()))
  if isother:other.append(r)
  if istarget:targets.append(r)
  annotations.append({'id':r['id'],'name':r['name'],'x_m':r['x_m'],'y_m':r['y_m'],
   'is_other_retail_context_v1':r['is_other_retail_context'],
   'is_other_retail_context_candidate_v2':isother,'is_target_context_proxy_v2':istarget,
   'scope_filter_flags_json':scope_json.dumps(flags),'in_exact_CA_master':a is not None})
 scope_pq.write_table(scope_pa.Table.from_pylist(annotations),output/'context_scope_review_v2.parquet',compression='zstd')
 xy=scope_np.array([(r['x_m'],r['y_m']) for r in master]);features=[{'id':r['id']} for r in master]
 for kind,cohort in [('other_retail',other),('other_target_proxy',targets)]:
  ids={r['id'] for r in cohort};own=scope_np.array([int(r['id'] in ids) for r in master])
  if cohort:
   tree=scope_cKDTree(scope_np.array([(r['x_m'],r['y_m']) for r in cohort]))
   for radius in [500,1000]:
    counts=tree.query_ball_point(xy,radius,return_length=True,workers=2)-own
    assert (counts>=0).all()
    for row,n in zip(features,counts):row[f'{kind}_count_{radius}m_v2']=int(n)
   if kind=='other_target_proxy':
    ds,_=tree.query(xy,k=2 if len(cohort)>1 else 1,workers=2)
    for i,row in enumerate(features):
     d=(ds[i,1] if own[i] else ds[i,0]) if len(cohort)>1 else ds[i]
     row['distance_to_other_target_proxy_m_v2']=float(d) if scope_np.isfinite(d) and not (len(cohort)==1 and own[i]) else None
  else:
   for row in features:
    for radius in [500,1000]:row[f'{kind}_count_{radius}m_v2']=0
    if kind=='other_target_proxy':row['distance_to_other_target_proxy_m_v2']=None
 report['context_feature_revision']={
  'context_rows':len(ctx),'v1_other_retail_context_rows':sum(r['is_other_retail_context'] for r in ctx),
  'v2_rule_screened_other_retail_context_rows':len(other),'v2_target_proxy_context_rows':len(targets),
  'other_retail_context_rows_removed':sum(r['is_other_retail_context_v1'] and not r['is_other_retail_context_candidate_v2'] for r in annotations),
  'other_retail_context_rows_added':sum(not r['is_other_retail_context_v1'] and r['is_other_retail_context_candidate_v2'] for r in annotations),
  'note':'V2 counts are rule-screened mapped candidates; omitted unknowns do not establish physical absence. Food, mall and university features retain V1 source definitions.'}
 scope_pq.write_table(scope_pa.Table.from_pylist(features),output/'revised_context_features_v2.parquet',compression='zstd')

In [66]:
def scope_main():
 ap=scope_argparse.ArgumentParser(description=scope___doc__)
 frozen=scope_ROOT/'data/archive/v1_pre_retail_scope_review/california_retail_master.parquet'
 ap.add_argument('--master',type=scope_Path,default=frozen if frozen.exists() else scope_ROOT/'data/processed/california_retail_master.parquet')
 ap.add_argument('--osm-all',type=scope_Path,default=scope_ROOT/'data/interim/osm/all_shop_candidates.csv')
 ap.add_argument('--output',type=scope_Path,default=scope_ROOT/'data/review_v2')
 a=ap.parse_args();a.output.mkdir(parents=True,exist_ok=True)
 report={'complete':False,'audit_version':'retail_scope_v2','scope_audit_is_exhaustive':False,
  'verified_50000_retail_rows':None,'started_at_utc':scope_datetime.now(scope_timezone.utc).isoformat()}
 scope_write_json(a.output/'audit_summary.json',report)
 master=scope_pq.read_table(a.master).to_pylist()
 print('Match all-shop OSM evidence to',len(master),'V1 locations',flush=True)
 by_id,linkstats=scope_match_osm(master,a.osm_all,a.output)
 manuals=scope_defaultdict(list)
 for path in [scope_ROOT/'pipeline/preview_scope_overrides_v2.json',scope_ROOT/'pipeline/retail_scope_overrides_v2.json',scope_ROOT/'pipeline/target_scope_overrides_v2.json']:
  for m in scope_read_manual(path):manuals[m['id']].append(m)
 audits=[scope_decide(r,by_id.get(r['id'],[]),manuals.get(r['id'],[])) for r in master]
 scope_pq.write_table(scope_pa.Table.from_pylist(audits),a.output/'retail_scope_annotations.parquet',compression='zstd')
 print('Recompute retail and target context after scope review',flush=True)
 scope_rebuild_context(master,audits,a.output,report)
 c=scope_duckdb.connect();c.execute("SET threads=2; SET memory_limit='2GB'")
 c.execute(f'CREATE VIEW old AS SELECT * FROM read_parquet({base_q(a.master)})')
 c.execute(f'CREATE VIEW au AS SELECT * FROM read_parquet({base_q(a.output/"retail_scope_annotations.parquet")})')
 c.execute(f'CREATE VIEW fx AS SELECT * FROM read_parquet({base_q(a.output/"revised_context_features_v2.parquet")})')
 c.execute("""CREATE TEMP TABLE reviewed AS SELECT old.*,au.* EXCLUDE(id),fx.* EXCLUDE(id),
  coalesce(acs_matched AND population>0 AND land_area_km2>0 AND under18_share IS NOT NULL
    AND young_adult_18_34_share IS NOT NULL AND median_household_income_2024_usd>0
    AND college_enrolled_share_3plus IS NOT NULL AND living_alone_household_share IS NOT NULL,false) AS core_features_complete_v2
  FROM old JOIN au USING(id) JOIN fx USING(id)""")
 c.execute("""CREATE TEMP TABLE final AS SELECT *,
  core_features_complete_v2 AND target_label_proxy_v2 IS NOT NULL
    AND retail_scope_status_v2 IN ('official_address_confirmed','osm_goods_corroborated','mapped_retail_candidate')
    AS rule_screened_proxy_eligible_v2,
  core_features_complete_v2 AND target_label_proxy_v2 IS NOT NULL
    AND retail_scope_status_v2 IN ('official_address_confirmed','osm_goods_corroborated')
    AS cross_evidence_proxy_eligible_v2
  FROM reviewed""")
 for filename,query in [
  ('retail_scope_review_master','SELECT * FROM final ORDER BY tract_geoid,id'),
  ('rule_screened_proxy_candidates','SELECT * FROM final WHERE rule_screened_proxy_eligible_v2 ORDER BY tract_geoid,id'),
  ('cross_evidence_proxy_candidates','SELECT * FROM final WHERE cross_evidence_proxy_eligible_v2 ORDER BY tract_geoid,id'),
  ('retail_scope_review_queue',"SELECT * FROM final WHERE retail_scope_status_v2 IN ('review_required','official_scope_exclusion') OR target_label_proxy_v2 IS NULL ORDER BY retail_scope_status_v2,name"),
 ]:
  c.execute(f'COPY ({query}) TO ? (FORMAT PARQUET,COMPRESSION ZSTD)',[str(a.output/(filename+'.parquet'))])
 cols='id,name,category,address,website,tract_geoid,target_label_proxy,target_label_proxy_v2,retail_scope_status_v2,retail_scope_flags_json,target_format_flags_json,osm_goods_corroborated_v2,scope_evidence_url_v2,scope_review_reason_v2,manually_reviewed_v2'
 c.execute(f'COPY (SELECT {cols} FROM final ORDER BY retail_scope_status_v2,name) TO ? (FORMAT CSV,HEADER TRUE)',[str(a.output/'retail_scope_review_index.csv')])
 c.execute(f'COPY (SELECT {cols} FROM final WHERE proxy_model_eligible AND NOT rule_screened_proxy_eligible_v2 ORDER BY name) TO ? (FORMAT CSV,HEADER TRUE)',[str(a.output/'v1_cohort_quarantined_rows.csv')])
 c.execute("""COPY (SELECT id,name,category,address,tract_geoid,population_density_per_km2,
  young_adult_18_34_share,median_household_income_2024_usd,other_retail_count_500m_v2,
  target_label_proxy_v2,retail_scope_status_v2,scope_review_reason_v2 FROM (
  SELECT *,row_number() OVER(PARTITION BY retail_scope_status_v2 ORDER BY hash(id)) rn FROM final)
  WHERE rn<=10 ORDER BY retail_scope_status_v2,rn) TO ? (FORMAT CSV,HEADER TRUE)""",[str(a.output/'audit_preview.csv')])
 c.execute(f"""COPY (SELECT a.*,coalesce(s.rule_screened_retail_candidates,0) AS rule_screened_retail_candidates,
  coalesce(s.target_proxy_candidates_v2,0) AS target_proxy_candidates_v2,
  coalesce(s.scope_review_required,0) AS scope_review_required,
  coalesce(s.rule_screened_model_rows,0) AS rule_screened_model_rows
  FROM read_parquet({base_q(scope_ROOT/'data/interim/census/ca_acs2024_tract_features.parquet')}) a
  LEFT JOIN (SELECT tract_geoid,
   count(*) FILTER(WHERE retail_scope_status_v2 IN ('official_address_confirmed','osm_goods_corroborated','mapped_retail_candidate')) AS rule_screened_retail_candidates,
   count(*) FILTER(WHERE target_label_proxy_v2=1 AND retail_scope_status_v2 IN ('official_address_confirmed','osm_goods_corroborated','mapped_retail_candidate')) AS target_proxy_candidates_v2,
   count(*) FILTER(WHERE retail_scope_status_v2='review_required' OR target_label_proxy_v2 IS NULL) AS scope_review_required,
   count(*) FILTER(WHERE rule_screened_proxy_eligible_v2) AS rule_screened_model_rows
   FROM final GROUP BY tract_geoid) s ON a.GEOID=s.tract_geoid ORDER BY a.GEOID)
  TO ? (FORMAT PARQUET,COMPRESSION ZSTD)""",[str(a.output/'california_tract_review_v2.parquet')])
 report.update(linkstats)
 report['rows_retained_for_audit']=len(master)
 report['retail_scope_counts']=[dict(zip(['status','rows'],r)) for r in c.execute('SELECT retail_scope_status_v2,count(*) FROM final GROUP BY 1 ORDER BY 2 DESC').fetchall()]
 report['v1_proxy_cohort_rows']=c.execute('SELECT count(*) FROM old WHERE proxy_model_eligible').fetchone()[0]
 report['rule_screened_proxy_rows']=c.execute('SELECT count(*) FROM final WHERE rule_screened_proxy_eligible_v2').fetchone()[0]
 report['cross_evidence_proxy_rows']=c.execute('SELECT count(*) FROM final WHERE cross_evidence_proxy_eligible_v2').fetchone()[0]
 report['rule_screened_proxy_label_counts']=[dict(zip(['label','rows'],r)) for r in c.execute('SELECT target_label_proxy_v2,count(*) FROM final WHERE rule_screened_proxy_eligible_v2 GROUP BY 1 ORDER BY 1').fetchall()]
 report['v1_cohort_quarantined_rows']=c.execute('SELECT count(*) FROM final WHERE proxy_model_eligible AND NOT rule_screened_proxy_eligible_v2').fetchone()[0]
 report['manually_reviewed_rows']=c.execute('SELECT count(*) FROM final WHERE manually_reviewed_v2').fetchone()[0]
 report['official_scope_confirmed_rows']=c.execute('SELECT count(*) FROM final WHERE retail_scope_officially_verified_v2').fetchone()[0]
 report['target_verified_v2_counts']=[dict(zip(['label','rows'],r)) for r in c.execute('SELECT target_label_official_verified_v2,count(*) FROM final WHERE target_label_official_verified_v2 IS NOT NULL GROUP BY 1 ORDER BY 1').fetchall()]
 report['risk_flag_counts']={k:sum(k in scope_json.loads(r['retail_scope_flags_json']) for r in audits) for k in [*scope_PATTERNS,'category_scope_requires_review','missing_street_address','osm_service_tag_conflict','osm_lifecycle_conflict','osm_scope_requires_review','entity_or_address_conflict','legacy_osm_address_conflict','geometry_review']}
 assert c.execute('SELECT count(*),count(DISTINCT id) FROM final').fetchone()==(len(master),len(master))
 assert c.execute("SELECT count(*) FROM final WHERE rule_screened_proxy_eligible_v2 AND (retail_scope_status_v2 IN ('review_required','official_scope_exclusion') OR target_label_proxy_v2 IS NULL)").fetchone()[0]==0
 report['complete']=True;report['completed_at_utc']=scope_datetime.now(scope_timezone.utc).isoformat()
 assert c.execute('SELECT count(*) FROM final WHERE other_retail_count_500m_v2>other_retail_count_1000m_v2 OR other_target_proxy_count_500m_v2>other_target_proxy_count_1000m_v2').fetchone()[0]==0
 report['context_feature_revision']['master_rows_retail_500m_changed']=c.execute('SELECT count(*) FROM final WHERE other_retail_count_500m!=other_retail_count_500m_v2').fetchone()[0]
 report['context_feature_revision']['master_rows_target_500m_changed']=c.execute('SELECT count(*) FROM final WHERE other_target_proxy_count_500m!=other_target_proxy_count_500m_v2').fetchone()[0]
 report['interpretation']=['Rule-screened candidates are not universally verified retailers.','OSM/Overture agreement is cross-source mapped evidence, not guaranteed statistically independent and not current on-site inspection.','Names or failed websites alone never prove a business is nonretail.','The >=50000 count for V1 does not certify >=50000 eligible physical retail businesses.','Use v2 retail context feature names for further audit; v1 columns remain solely for traceability.']
 scope_write_json(a.output/'audit_summary.json',report)
 scope_write_json(a.output/'scope_rules.json',{'name_risk_patterns':scope_PATTERNS,'target_format_patterns':scope_TARGET_FORMAT_PATTERNS,'goods_shop_tags':sorted(scope_OSM_GOODS),'service_shop_tags':sorted(scope_OSM_SERVICE),'category_review':sorted(scope_CATEGORY_REVIEW)})
 predictors=['population_density_per_km2','under18_share','young_adult_18_34_share','median_household_income_2024_usd',
  'college_enrolled_share_3plus','family_household_share','living_alone_household_share',
  'other_retail_count_500m_v2','other_retail_count_1000m_v2','food_count_500m','food_count_1000m','distance_to_mall_m','distance_to_university_m']
 scope_write_json(a.output/'model_feature_spec_v2.json',{'status':'AUDIT_CANDIDATES_NOT_FINAL_MODEL_TRAINING_DATA',
  'candidate_predictors':predictors,'provisional_label':'target_label_proxy_v2','verified_subset_label':'target_label_official_verified_v2',
  'candidate_filters':['rule_screened_proxy_eligible_v2','cross_evidence_proxy_eligible_v2'],
  'forbidden_features':'Names, categories, source indicators, audit status, target counts and target distances must not be classifier inputs.',
  'validation':'Use spatially grouped validation; no model fitted in this audit.'})
 print(scope_json.dumps({k:report[k] for k in ['retail_scope_counts','rule_screened_proxy_rows','cross_evidence_proxy_rows','v1_cohort_quarantined_rows']},indent=2))

In [67]:
with notebook_args('--master',archive_path,'--osm-all',osm_dir/'all_shop_candidates.csv',
 '--output',RUN_ROOT/'data/review_v2'):
    scope_main()
show_counts(RUN_ROOT/'data/review_v2/retail_scope_review_master.parquet','Reviewed candidate table')
gc.collect()

Match all-shop OSM evidence to 81494 V1 locations


Recompute retail and target context after scope review


{
  "retail_scope_counts": [
    {
      "status": "mapped_retail_candidate",
      "rows": 65998
    },
    {
      "status": "osm_goods_corroborated",
      "rows": 11431
    },
    {
      "status": "review_required",
      "rows": 4008
    },
    {
      "status": "official_address_confirmed",
      "rows": 47
    },
    {
      "status": "official_scope_exclusion",
      "rows": 10
    }
  ],
  "rule_screened_proxy_rows": 71540,
  "cross_evidence_proxy_rows": 10653,
  "v1_cohort_quarantined_rows": 3497
}
Reviewed candidate table : 81,494 rows; 81,494 unique IDs


29

## 7. V3：固定清洗规则与重复／定位风险隔离

执行0.90存在性筛选／open或官方地址证据、编号地址、邮编、联系方式、明确商品末级类别及服务／旧连锁风险规则；隔离同名同址或同分店网址的未解重复。0.90不是90%的标签正确率。

函数／常量加阶段前缀，避免同一内核中的同名变量相互覆盖。

In [68]:
# Source: pipeline/build_clean_model_v3.py; SHA256: 74617eafa7b2a47b37a2972c2838ae27398b5ed9a703afa66d0669ffe987fe24
quality___file__ = str(RUN_ROOT / 'pipeline/build_clean_model_v3.py')
quality___doc__ = 'Conservative, reproducible mapped-retailer modeling frame.\n\nThe target is mapped specialist status, not a universally observed business\noutcome. Quality rules are frozen before an independent validation sample.\nUnknowns and source conflicts never become negatives.'
"""Conservative, reproducible mapped-retailer modeling frame.

The target is mapped specialist status, not a universally observed business
outcome. Quality rules are frozen before an independent validation sample.
Unknowns and source conflicts never become negatives.
"""
from pathlib import Path as quality_Path
from collections import defaultdict as quality_defaultdict, Counter as quality_Counter
from datetime import datetime as quality_datetime, timezone as quality_timezone
import argparse as quality_argparse
import hashlib as quality_hashlib
import json as quality_json
import re as quality_re
import sys as quality_sys
from urllib.parse import urlsplit as quality_urlsplit, urlunsplit as quality_urlunsplit, parse_qsl as quality_parse_qsl, urlencode as quality_urlencode
import duckdb as quality_duckdb
import pyarrow as quality_pa
import pyarrow.parquet as quality_pq


quality_ROOT=quality_Path(quality___file__).resolve().parents[1]
quality_OUT=quality_ROOT/'data/clean_v3'
# Product-retail leaves only. Ambiguous services and broad parents are absent.
quality_COMPARATOR_LEAVES=set('''academic_bookstore antique_store aquatic_pet_store archery_store art_supply_store arts_crafts_and_hobby_store audio_visual_equipment_store baby_gear_and_nursery_store battery_store beauty_supply_store bike_store bird_store books_music_and_video_store bookstore bridal_shop camera_and_photography_store cards_and_stationery_store childrens_clothing_store church_supply_store clothing_store comic_books_store computer_store cosmetics_and_fragrance_store costume_store craft_store custom_clothing_store denim_wear_store department_store designer_clothing diamond_dealer discount_store dive_store dj_equipment_store drone_store duty_free_store educational_supply_store electronics_store eyewear_store fabric_store fashion_accessories_store fashion_and_apparel_store fashion_boutique fireworks_store fitness_exercise_store florist flower_market flowers_and_gifts_store formal_wear_store framing_store fur_clothing gemstone_and_mineral_store gift_shop goldsmith golf_equipment_store guitar_store hair_supply_store handbag_store hat_store hockey_equipment_store home_decor_store home_goods_store home_theater_systems_stores horse_equipment_shop hunting_and_fishing_store jewelry_store knitting_supply_store leather_goods_store lingerie_store luggage_store maternity_wear mens_clothing_store mobile_phone_accessory_store mobile_phone_store music_and_dvd_store newspaper_and_magazines_store office_supply_store orthopedic_shoe_store outdoor_store outlet_store party_supply_store pawn_shop pen_store personal_care_and_beauty_store pet_store piano_store plus_size_clothing_store pool_and_billiards_store religious_and_spiritual_goods_store reptile_store safe_store second_hand_clothing_store second_hand_store shoe_store skate_store ski_and_snowboard_store souvenir_store sporting_goods_store sportswear_store sunglasses_store superstore surf_store swimwear_store t_shirt_store traditional_clothing uniform_store used_bookstore video_game_store vinyl_record_store watch_store wig_store womens_clothing_store'''.split())
# These leaves have observed instruction/production/service confusion in the
# development audit. Require additional physical-goods evidence to retain.
quality_REQUIRES_GOODS_EVIDENCE={'dancewear_store','musical_instrument_store','packing_supply_store',
 'customized_merchandise_store','custom_t_shirt_store','embroidery_and_crochet_store','trophy_store'}
quality_SERVICE_PATTERNS={
 'ignition_interlock_service':r'ignition\s*interlock|\binterlock\b|\blife\s*safer\b|\blow\s*cost\s*interlock\b|\bintoxalock\b|\bsmart\s*start\b|\bdraeger\b',
 'education_service':r'\b(?:instructor|instruction|teacher|tutors?|tutoring|lessons?|dance\s*here|dance\s*(?:center|centre|company|school)|music\s*(?:education|school)|kindergarten|childcare|daycare|driving\s*school)\b',
 'medical_or_dental_service':r'\b(?:optometrist|optometry|ophthalmology|dentist|dental|orthodontic|audiology|hearing\s*aid\s*center|physician|podiatry|podiatrist|medical\s*clinic)\b|\bdr\.?\s+[a-z]',
 'postal_or_print_service':r'\b(?:ups\s*store|fedex\s*office|postnet|postal|mailbox|shipping\s*center|printing\s*service|print\s*shop)\b',
 'nonretail_entity_or_format':r'\b(?:corporate\s*(?:office|headquarters)|headquarters|warehouse|distribution\s*center|manufacturer|wholesale|b2b|sales\s*representative|mobile\s*service|by\s*appointment\s*only|pop[ -]?up\s*event)\b',
}
quality_LEGACY_LOCATION_PATTERNS={
 'legacy_chain_location_review':r'\b(?:radio\s*shack|payless(?:\s*shoe\s*source)?|fry.?s\s*electronics|circuit\s*city|mervyn.?s|sports\s*authority|borders(?:\s*books)?|b\.?\s*dalton|anchor\s*blue|orchard\s*supply\s*hardware|jenny\s*craig|wet\s*seal|dress\s*barn)\b',
}

In [69]:
def quality_sha(p):
 h=quality_hashlib.sha256()
 with p.open('rb') as f:
  for b in iter(lambda:f.read(1024*1024),b''):h.update(b)
 return h.hexdigest()

In [70]:
def quality_address_token(s):
 s=(s or '').lower().strip()
 # A comma separates city/state in some freeform addresses. Keep the unit
 # separately so Suite/Ste/Spc spelling does not split the same address.
 u=base_unit(s)
 marker=r'\b(?:suite|ste|unit|space|spc|floor)(?=[.\s#]|\d|[a-z](?:[-\s]|$))|#'
 street=quality_re.split(marker,s,maxsplit=1)[0].split(',')[0]
 for a,b in [('street','st'),('avenue','ave'),('boulevard','blvd'),('road','rd'),('drive','dr'),('lane','ln'),('court','ct'),('place','pl'),('parkway','pkwy'),('highway','hwy'),('north','n'),('south','s'),('east','e'),('west','w')]:
  street=quality_re.sub(r'\b'+a+r'\b',b,street)
 return quality_re.sub(r'[^a-z0-9]','',street),quality_re.sub(r'[^a-z0-9]','',u) if u else ''

In [71]:
def quality_normalized_url(s):
 try:
  u=quality_urlsplit((s or '').strip());host=u.hostname or ''
  if u.scheme not in {'http','https'} or '.' not in host:return ''
  query=[(k,v) for k,v in quality_parse_qsl(u.query) if not k.lower().startswith(('utm_','fbclid','gclid'))]
  return quality_urlunsplit(('https',host.lower().removeprefix('www.'),u.path.rstrip('/'),quality_urlencode(sorted(query)),''))
 except ValueError:return ''

In [72]:
def quality_contact_ok(r):
 return bool(quality_normalized_url(r.get('website')) or len(quality_re.sub(r'\D','',r.get('phone') or ''))>=10)

In [73]:
def quality_duplicate_flags(rows):
 groups=quality_defaultdict(list);urlgroups=quality_defaultdict(list)
 for r in rows:
  street,u=quality_address_token(r.get('address'));zip5=(quality_re.search(r'\d{5}',r.get('postcode') or '') or [None])[0]
  if street and zip5 and r.get('name'):
   groups[(base_name_key(r['name']),street,u,zip5)].append(r['id'])
   url=quality_normalized_url(r.get('website'))
   if url and quality_urlsplit(url).path not in {'','/','/home','/contact','/contact-us','/locations','/stores','/store-locator','/find-a-store'}:
    urlgroups[(url,street,u,zip5)].append(r['id'])
 out=quality_defaultdict(list);evidence=[]
 for basis,gs in [('same_name_address_zip',groups),('same_branch_url_address_zip',urlgroups)]:
  for key,ids in gs.items():
   ids=sorted(set(ids))
   if len(ids)>1:
    gid=quality_hashlib.sha256((basis+str(key)).encode()).hexdigest()[:16]
    for mid in ids:out[mid].append(gid)
    evidence.append({'group_id':gid,'basis':basis,'match_key':list(key),'ids':ids,'action':'quarantine_all_unresolved_members'})
 return out,evidence

In [74]:
def quality_rules():
 return {'version':'v3','frozen_before_validation':True,
  'unit':'one mapped consumer-facing product-retail location in California, one snapshot',
  'target':'mapped toy/collectible/TCG/model specialist status; not profit, success, demand, or current on-site truth',
  'minimum_confidence':0.90,'source_status':'open',
  'official_address_confirmation_alternative':'Current exact-address official physical-goods confirmation may replace missing confidence/status, never conflict/geometry/demographic checks.',
  'street':'numbered street, nonempty locality, valid ZIP5, no explicit non-CA region',
  'contact':'HTTP(S) website with hostname, or at least 10-digit phone; syntactic availability, not proof of current operation',
  'comparator_leaf_allowlist':sorted(quality_COMPARATOR_LEAVES),
  'requires_physical_goods_evidence_categories':sorted(quality_REQUIRES_GOODS_EVIDENCE),
  'new_service_review_patterns':quality_SERVICE_PATTERNS,'legacy_location_review_patterns':quality_LEGACY_LOCATION_PATTERNS,
  'duplicates':'Quarantine all unresolved same normalized name+address+ZIP or same specific branch URL+address+ZIP groups at any coordinate distance.',
  'missing_target_policy':'Never convert unknown to zero. Source labels remain explicitly distinguished from audited labels.',
  'comparison_interpretation':'Specialist distribution relative to consumer discretionary goods retailers. No grocery/pharmacy expansion for row counts.',
  'calibration_warning':'Confidence is an existence-oriented relative filter, not a calibrated label probability; 0.90 does not establish 90% correctness.'}

In [75]:
def quality_evaluate(r,dups):
 flags=[];official=bool(r['retail_scope_officially_verified_v2'])
 if not r['rule_screened_proxy_eligible_v2']:flags.append('v2_scope_or_label_or_demographic_ineligible')
 if not official and not (r.get('confidence') is not None and r['confidence']>=0.90 and r.get('operating_status')=='open'):flags.append('source_confidence_or_status')
 if not quality_re.match(r'^\s*\d+[a-z]?(?:\s|[-/])',r.get('address') or '',quality_re.I):flags.append('numbered_street_required')
 if not (r.get('locality') or '').strip():flags.append('missing_locality')
 if not quality_re.fullmatch(r'\d{5}(?:-\d{4})?',(r.get('postcode') or '').strip()):flags.append('invalid_postcode')
 if (r.get('address_region') or '').strip().upper() not in {'','CA','CALIFORNIA','US-CA'}:flags.append('non_CA_region_conflict')
 if not quality_contact_ok(r):flags.append('missing_valid_contact_format')
 if r['id'] in dups:flags.append('unresolved_duplicate_or_coordinate_group')
 cat=r.get('category') or ''
 if r['target_label_proxy_v2']==0:
  if cat not in quality_COMPARATOR_LEAVES and cat not in quality_REQUIRES_GOODS_EVIDENCE:flags.append('outside_explicit_product_leaf_frame')
  if cat in quality_REQUIRES_GOODS_EVIDENCE and not (official or r['osm_goods_tag_evidence_v2']):flags.append('mixed_service_leaf_without_goods_evidence')
 for k,p in quality_SERVICE_PATTERNS.items():
  if quality_re.search(p,r.get('name') or '',quality_re.I) and not official:flags.append(k)
 for k,p in quality_LEGACY_LOCATION_PATTERNS.items():
  if quality_re.search(p,r.get('name') or '',quality_re.I) and not official:flags.append(k)
 return flags

In [76]:
def quality_main():
 ap=quality_argparse.ArgumentParser(description=quality___doc__);ap.add_argument('--freeze-comparators',action='store_true');a=ap.parse_args()
 quality_OUT.mkdir(parents=True,exist_ok=True)
 path=quality_ROOT/'data/review_v2/retail_scope_review_master.parquet'
 rows=quality_pq.read_table(path).to_pylist();dups,groups=quality_duplicate_flags(rows)
 (quality_OUT/'duplicate_review_groups_v3.json').write_text(quality_json.dumps(groups,ensure_ascii=False,indent=2))
 spec=quality_rules();spec_path=quality_OUT/'cleaning_rules_v3.json'
 if spec_path.exists():
  assert quality_json.loads(spec_path.read_text())==spec,'Do not silently replace frozen rules; version any amendment.'
 else:spec_path.write_text(quality_json.dumps(spec,ensure_ascii=False,indent=2))
 annotations=[]
 for r in rows:
  f=quality_evaluate(r,dups)
  annotations.append({'id':r['id'],'v3_rule_eligible':not f,'v3_exclusion_reasons_json':quality_json.dumps(f),'duplicate_review_groups_json':quality_json.dumps(dups.get(r['id'],[]))})
 apath=quality_OUT/'quality_annotations_v3.parquet';quality_pq.write_table(quality_pa.Table.from_pylist(annotations),apath,compression='zstd')
 c=quality_duckdb.connect();c.read_parquet(str(path)).create_view('m');c.read_parquet(str(apath)).create_view('a')
 c.execute('CREATE TEMP TABLE quality AS SELECT m.*,a.* EXCLUDE(id) FROM m JOIN a USING(id)')
 stats={'version':'v3','built_at_utc':quality_datetime.now(quality_timezone.utc).isoformat(),'input_v2_sha256':quality_sha(path),'rules_sha256':quality_sha(spec_path),
  'source_rows':len(rows),'quality_rule_eligible_rows':c.execute('SELECT count(*) FROM quality WHERE v3_rule_eligible').fetchone()[0],
  'class_counts':c.execute('SELECT target_label_proxy_v2,count(*) FROM quality WHERE v3_rule_eligible GROUP BY 1 ORDER BY 1').fetchall(),
  'duplicate_groups':len(groups),'duplicate_review_ids':len(dups),
  'flag_counts':dict(quality_Counter(k for r in annotations for k in quality_json.loads(r['v3_exclusion_reasons_json']))),
  'status':'QUALITY_FRAME_FROZEN_FOR_INDEPENDENT_VALIDATION_NOT_FINAL_RELEASE'}
 if a.freeze_comparators:
  frozen=quality_OUT/'frozen_comparator_frame_v3.parquet'
  if frozen.exists():raise FileExistsError('Frozen comparator frame exists; refuse silent replacement.')
  c.execute("COPY (SELECT * FROM quality WHERE v3_rule_eligible AND target_label_proxy_v2=0 ORDER BY id) TO ? (FORMAT PARQUET,COMPRESSION ZSTD)",[str(frozen)])
  stats['frozen_comparator_rows']=c.execute('SELECT count(*) FROM quality WHERE v3_rule_eligible AND target_label_proxy_v2=0').fetchone()[0]
  stats['frozen_comparator_sha256']=quality_sha(frozen)
 c.execute('COPY (SELECT * FROM quality ORDER BY id) TO ? (FORMAT PARQUET,COMPRESSION ZSTD)',[str(quality_OUT/'quality_review_master_v3.parquet')])
 (quality_OUT/'quality_frame_manifest_v3.json').write_text(quality_json.dumps(stats,ensure_ascii=False,indent=2))
 print(quality_json.dumps(stats,ensure_ascii=False,indent=2))

In [77]:
with notebook_args():quality_main()
new_quality=RUN_ROOT/'data/clean_v3/quality_review_master_v3.parquet'
frozen_input=INPUTS/'data/clean_v3/frozen_comparator_frame_v3.parquet'
with duckdb.connect() as con:
    recalculated={r[0] for r in con.execute('SELECT id FROM read_parquet(?) WHERE v3_rule_eligible AND target_label_proxy_v2=0',[str(new_quality)]).fetchall()}
    frozen_ids={r[0] for r in con.execute('SELECT id FROM read_parquet(?)',[str(frozen_input)]).fetchall()}
assert recalculated==frozen_ids,{'new_only':len(recalculated-frozen_ids),'reference_only':len(frozen_ids-recalculated)}
# 复用当时已冻结的研究输入，不重抽样、不重新冻结出不同总体。
clean_out=RUN_ROOT/'data/clean_v3'
assert json.loads((clean_out/'cleaning_rules_v3.json').read_text())==json.loads((INPUTS/'data/clean_v3/cleaning_rules_v3.json').read_text())
shutil.copy2(frozen_input,clean_out/'frozen_comparator_frame_v3.parquet')
# 原函数非freeze重跑不写这两个字段；在新运行目录补回已验证的冻结信息。
quality_manifest_path=clean_out/'quality_frame_manifest_v3.json'
quality_manifest=json.loads(quality_manifest_path.read_text())
quality_manifest.update(frozen_comparator_rows=len(frozen_ids),frozen_comparator_sha256=notebook_sha(frozen_input),
 research_input_mode='Reused immutable original frozen frame after exact recomputed-ID agreement; no resampling')
quality_manifest_path.write_text(json.dumps(quality_manifest,ensure_ascii=False,indent=2))
print('Exact agreement with frozen comparator population:',len(frozen_ids))
gc.collect()

{
  "version": "v3",
  "built_at_utc": "2026-10-26T01:06:13.499111+00:00",
  "input_v2_sha256": "620e080c0192246ebc43c531e4c08e06bd296ced79ebb9e7b4c8c80c5b17ff7c",
  "rules_sha256": "7532001002772b15bf19e71eb4a35e72919bbf3071d90b331348ecd1d39a3f82",
  "source_rows": 81494,
  "quality_rule_eligible_rows": 58608,
  "class_counts": [
    [
      0,
      58030
    ],
    [
      1,
      578
    ]
  ],
  "duplicate_groups": 971,
  "duplicate_review_ids": 1690,
  "flag_counts": {
    "v2_scope_or_label_or_demographic_ineligible": 9954,
    "source_confidence_or_status": 9852,
    "unresolved_duplicate_or_coordinate_group": 1690,
    "mixed_service_leaf_without_goods_evidence": 1669,
    "education_service": 97,
    "missing_locality": 232,
    "numbered_street_required": 1289,
    "nonretail_entity_or_format": 473,
    "invalid_postcode": 452,
    "legacy_chain_location_review": 1033,
    "missing_valid_contact_format": 1213,
    "medical_or_dental_service": 107,
    "ignition_interlock_se

Exact agreement with frozen comparator population: 58030


18

## 8. 应用最终裁决、重算邻域、导出总表

读取已经完成的目标店审核、100条冻结留出审查和额外实体排除表；隔离ROBO、地址／营业冲突和追加重复风险。重新筛选零售／餐饮／商场／大学背景，排除自身，计算500m／1km数量及最近距离，导出CSV与Parquet。所有审核输入均验SHA，避免半成品或混版裁决。

函数／常量加阶段前缀，避免同一内核中的同名变量相互覆盖。

In [78]:
# Source: pipeline/release_clean_model_v3.py; SHA256: 371d8967d63754ad1fb56c96082674c5442c5bf5d2287dcca087482b87ab2a6c
release___file__ = str(RUN_ROOT / 'pipeline/release_clean_model_v3.py')
release___doc__ = 'Release source-qualified mapped-retailer data with explicit proxy labels.\n\nConsumes frozen quality rules, separate target evidence, and untouched holdout\nadjudication. It does not manufacture a 50k target or claim universal manual\nverification. All exclusions and label decisions remain separately traceable.'
"""Release source-qualified mapped-retailer data with explicit proxy labels.

Consumes frozen quality rules, separate target evidence, and untouched holdout
adjudication. It does not manufacture a 50k target or claim universal manual
verification. All exclusions and label decisions remain separately traceable.
"""
from pathlib import Path as release_Path
from collections import Counter as release_Counter, defaultdict as release_defaultdict
from datetime import datetime as release_datetime, timezone as release_timezone
import argparse as release_argparse
import csv as release_csv
import hashlib as release_hashlib
import json as release_json
import re as release_re
import duckdb as release_duckdb
import numpy as release_np
import pyarrow as release_pa
import pyarrow.parquet as release_pq
from scipy.spatial import cKDTree as release_cKDTree




release_PREDICTORS=['population_density_per_km2','under18_share','young_adult_18_34_share',
 'median_household_income_2024_usd','college_enrolled_share_3plus','family_household_share',
 'living_alone_household_share','other_retail_count_500m_clean','other_retail_count_1000m_clean',
 'food_count_500m_clean','food_count_1000m_clean','distance_to_mall_m_clean','distance_to_university_m_clean']
release_TARGET_RULES={'primary_target_categories':['toy_store','osm_shop_toys','osm_shop_model','tabletop_games_store'],
 'other_categories_require':'Official exact-address specialist confirmation, or accepted explicit OSM toys/model evidence with no location conflicts.',
 'toy_primary_label':'Source-qualified mapped toy-store category remains a proxy; source confidence is not label accuracy.',
 'not_automatically_targets':'General gift/party/hobby, gaming, apparel and mixed-lifestyle stores require category-specific evidence.',
 'excluded_formats':'Unresolved vending/ROBO, park/concession, B2B-only, online-only, duplicate, address and opening-status conflicts.'}

In [79]:
def release_sha(path):
 h=release_hashlib.sha256()
 with path.open('rb') as f:
  for b in iter(lambda:f.read(1024*1024),b''):h.update(b)
 return h.hexdigest()

In [80]:
def release_load_rows(path):
 if not path.exists():return []
 v=release_json.loads(path.read_text())
 return v if isinstance(v,list) else v.get('overrides',v.get('records',v.get('adjudications',[])))

In [81]:
def release_target_decision(r,m):
 result={'label':r['target_label_proxy_v2'],'verified_label':r['target_label_official_verified_v2'],
  'retail_official':bool(r['retail_scope_officially_verified_v2']),
  'subtype':None,'reason':None,'evidence_url':r['scope_evidence_url_v2'],
  'new_exclusion':None,'record_status':'source_qualified'}
 if not m:return result
 sd=m.get('retail_scope_decision',m.get('decision'));td=m.get('target_decision')
 result.update(reason=m.get('reason'),evidence_url=m.get('evidence_url') or result['evidence_url'],
  subtype=m.get('target_subtype'),record_status=m.get('record_status','audited'))
 if sd=='retailer_confirmed':result['retail_official']=True
 if td=='target_confirmed':result['label']=1;result['verified_label']=1 if result['retail_official'] else None
 elif td=='comparator_confirmed':result['label']=0;result['verified_label']=0 if result['retail_official'] else None
 elif td=='ambiguous':result['label']=None;result['verified_label']=None
 if sd=='nonretail_confirmed' or m.get('exclude_from_training') is True:
  result['new_exclusion']=m.get('record_status','official_evidence_exclusion');result['label']=None;result['verified_label']=None
 if sd=='ambiguous':
  result['retail_official']=False;result['verified_label']=None
  if m.get('record_status') in {'address_or_opening_conflict','duplicate_review','non_target','nonretail','opening_status_conflict','address_conflict'}:
   result['new_exclusion']=m['record_status'];result['label']=None
 return result

In [82]:
def release_context_metadata():
 c=release_duckdb.connect()
 p=quality_ROOT/'data/interim/integration/context_pre_dedup.parquet'
 old=quality_ROOT/'data/interim/integration/context_canonical.parquet'
 rows=c.execute(f"""SELECT x.*,p.category,p.confidence,p.operating_status,p.address,p.locality,p.postcode,
   p.address_region,p.address_country,p.website,p.phone
   FROM read_parquet({base_q(old)}) x LEFT JOIN read_parquet({base_q(p)}) p USING(id)""").to_arrow_table().to_pylist()
 c.close();return rows

In [83]:
def release_count_features(final,context,decisions,excluded_ids,report):
 """Every retained context row has an explicit source-quality definition."""
 
 fm={r['id']:r for r in final};final_ids=set(fm)
 background_duplicates,_=quality_duplicate_flags(context)
 # Background quality uses the same address/contact/existence filter, without
 # California-only county restriction or ACS completeness.
 accepted=[];background_flags=release_Counter()
 for r in context:
  mid=r['id'];m=fm.get(mid);d=decisions.get(mid)
  if m:
   retail_kind='target' if m['is_mapped_specialist']==1 else 'comparison'
   accepted.append({'id':mid,'name':m['name'],'x_m':m['x_m'],'y_m':m['y_m'],
    'other_retail':retail_kind=='comparison','target':retail_kind=='target','food':False,'mall':False,'university':False})
   continue
  if mid in excluded_ids:background_flags['review_or_excluded_master']+=1;continue
  if mid in background_duplicates:background_flags['background_duplicate_group']+=1;continue
  if r.get('address_country') not in (None,'','US','USA'):background_flags['non_US_context']+=1;continue
  if not r.get('confidence') or r['confidence']<0.90:background_flags['context_confidence']+=1;continue
  cat=r.get('category') or ''
  # Distances to mapped campuses and malls use existence screening; missing
  # operating status is allowed for the mapped physical destination itself.
  if r['is_mall_context'] or r['is_university_context']:
   if r.get('operating_status') in {'permanently_closed','temporarily_closed'}:continue
   accepted.append({'id':mid,'name':r['name'],'x_m':r['x_m'],'y_m':r['y_m'],
    'other_retail':False,'target':False,'food':False,'mall':r['is_mall_context'],'university':r['is_university_context']})
   continue
  if r.get('operating_status')!='open':background_flags['context_status']+=1;continue
  if not release_re.match(r'^\s*\d+[a-z]?(?:\s|[-/])',r.get('address') or '',release_re.I) or not (r.get('locality') or '').strip() or not release_re.fullmatch(r'\d{5}(?:-\d{4})?',(r.get('postcode') or '').strip()) or not quality_contact_ok(r):
   background_flags['context_address_or_contact']+=1;continue
  if r['is_food_context']:
   accepted.append({'id':mid,'name':r['name'],'x_m':r['x_m'],'y_m':r['y_m'],
    'other_retail':False,'target':False,'food':True,'mall':False,'university':False});continue
  if cat not in quality_COMPARATOR_LEAVES|quality_REQUIRES_GOODS_EVIDENCE|{'toy_store','tabletop_games_store','hobby_shop'}:continue
  if scope_risks(r['name'],cat,r.get('address')):continue
  if any(release_re.search(p,r['name'] or '',release_re.I) for p in [*quality_SERVICE_PATTERNS.values(),*quality_LEGACY_LOCATION_PATTERNS.values()]):continue
  if cat in quality_REQUIRES_GOODS_EVIDENCE:continue
  label=d['label'] if d else r['target_label_proxy']
  if label is None:continue
  if label==1 and any(release_re.search(p,r['name'] or '',release_re.I) for p in scope_TARGET_FORMAT_PATTERNS.values()):continue
  accepted.append({'id':mid,'name':r['name'],'x_m':r['x_m'],'y_m':r['y_m'],
   'other_retail':label==0,'target':label==1,'food':False,'mall':False,'university':False})
 missing=final_ids-{r['id'] for r in accepted}
 for mid in missing:
  r=fm[mid];accepted.append({'id':mid,'name':r['name'],'x_m':r['x_m'],'y_m':r['y_m'],
   'other_retail':r['is_mapped_specialist']==0,'target':r['is_mapped_specialist']==1,'food':False,'mall':False,'university':False})
 assert len(accepted)==len({r['id'] for r in accepted})
 xy=release_np.array([(r['x_m'],r['y_m']) for r in final])
 context_counts={}
 for kind,field in [('other_retail','other_retail'),('food','food'),('other_target','target')]:
  subset=[r for r in accepted if r[field]];ids={r['id'] for r in subset};own=release_np.array([int(r['id'] in ids) for r in final])
  context_counts[kind]=len(subset)
  tree=release_cKDTree(release_np.array([(r['x_m'],r['y_m']) for r in subset])) if subset else None
  for radius in [500,1000]:
   values=tree.query_ball_point(xy,radius,return_length=True,workers=2)-own if tree else release_np.zeros(len(final),dtype=int)
   assert (values>=0).all()
   for row,n in zip(final,values):row[f'{kind}_count_{radius}m_clean']=int(n)
  if kind=='other_target':
   distances,_=tree.query(xy,k=2 if len(subset)>1 else 1,workers=2) if tree else (None,None)
   for i,row in enumerate(final):
    value=(distances[i,1] if own[i] else distances[i,0]) if len(subset)>1 else (distances[i] if subset else release_np.inf)
    row['distance_to_other_target_m_clean']=float(value) if release_np.isfinite(value) and not (len(subset)==1 and own[i]) else None
 for kind in ['mall','university']:
  subset=[r for r in accepted if r[kind]];context_counts[kind]=len(subset)
  if not subset:raise ValueError('No eligible '+kind+' context for distance feature')
  tree=release_cKDTree(release_np.array([(r['x_m'],r['y_m']) for r in subset]));ds,_=tree.query(xy,workers=2)
  for row,d in zip(final,ds):row['distance_to_'+kind+'_m_clean']=float(d)
 release_pq.write_table(release_pa.Table.from_pylist(accepted),quality_OUT/'context_clean_v3.parquet',compression='zstd')
 report['context_counts']=context_counts;report['context_exclusion_counts_overlapping']=dict(background_flags)
 report['context_interpretation']='Source-qualified US mapped context; food requires listed-open/.90/numbered address/locality/ZIP/contact. Mall/campus distances allow unknown operation status with .90 existence confidence. Border coverage limited to the original retrieval rectangle.'

In [84]:
def release_main():
 ap=release_argparse.ArgumentParser(description=release___doc__);ap.add_argument('--allow-unfinished-audit',action='store_true');a=ap.parse_args()
 spec=release_json.loads((quality_OUT/'cleaning_rules_v3.json').read_text());assert spec==quality_rules()
 frame_manifest=release_json.loads((quality_OUT/'quality_frame_manifest_v3.json').read_text())
 frozen=quality_OUT/'frozen_comparator_frame_v3.parquet';assert release_sha(frozen)==frame_manifest['frozen_comparator_sha256']
 targets_path=quality_ROOT/'pipeline/target_scope_overrides_v3.json'
 audit_path=quality_ROOT/'validation/comparator_holdout_v3/final_adjudication.json'
 if not a.allow_unfinished_audit:
  assert targets_path.exists(),'Complete target audit before release.'
  assert audit_path.exists(),'Complete frozen comparator validation before release.'
  assert release_json.loads(targets_path.read_text()).get('complete') is True,'Target adjudication is not finalized.'
  design=release_json.loads((quality_ROOT/'validation/comparator_holdout_v3/sampling_design.json').read_text())
  sample_path=quality_ROOT/'validation/comparator_holdout_v3/sample_frame.json'
  sample=release_json.loads(sample_path.read_text());adjudicated=release_load_rows(audit_path)
  assert len(adjudicated)==100 and len({r['id'] for r in adjudicated})==100
  assert {r['id'] for r in adjudicated}=={r['id'] for r in sample}
  assert release_sha(sample_path)==design['sample_json_sha256']
  assert release_sha(frozen)==design['frame_sha256']
  assert release_sha(quality_OUT/'cleaning_rules_v3.json')==design['rule_spec_sha256']
 additional_entity_path=quality_ROOT/'pipeline/additional_entity_exclusions_v3.json'
 additional_entity_ids=set(release_json.loads(additional_entity_path.read_text()).get('exclude_ids',[])) if additional_entity_path.exists() else set()
 audit_input_paths=[p for p in [targets_path,audit_path,quality_ROOT/'validation/target_scope_audit_v3.json',additional_entity_path] if p.exists()]
 audit_input_hashes={str(p.relative_to(quality_ROOT)):release_sha(p) for p in audit_input_paths}
 (quality_OUT/'clean_dataset_manifest_v3.json').write_text(release_json.dumps({'complete':False,'version':'v3','status':'BUILD_IN_PROGRESS','started_at_utc':release_datetime.now(release_timezone.utc).isoformat()}))
 rows=release_pq.read_table(quality_OUT/'quality_review_master_v3.parquet').to_pylist();byid={r['id']:r for r in rows}
 target_overrides={r['id']:r for r in release_load_rows(targets_path)}
 holdout={r['id']:r for r in release_load_rows(audit_path)}
 negative_ids=set(release_pq.read_table(frozen,columns=['id'])['id'].to_pylist())
 dupids={mid for g in release_json.loads((quality_OUT/'duplicate_review_groups_v3.json').read_text()) for mid in g['ids']}
 target_group_report=quality_ROOT/'validation/target_scope_audit_v3.json'
 target_groups=release_json.loads(target_group_report.read_text()).get('duplicate_review_groups',[]) if target_group_report.exists() else []
 target_duplicate_ids={r['id'] for g in target_groups for r in g['rows']}
 dupids.update(target_duplicate_ids)
 decisions={};trace=[];final=[];excluded_ids=set(additional_entity_ids)
 for r in rows:
  mid=r['id'];m=target_overrides.get(mid);d=release_target_decision(r,m);decisions[mid]=d
  reason=[]
  if mid in negative_ids:
   # Preserve the frozen controls. Only documented record-level audit decisions
   # may remove a sampled record; no new global rule is learned from holdout.
   h=holdout.get(mid)
   confirmed=bool(h and h.get('decision') in {'exact_address_goods_confirmed','retailer_confirmed'} and h.get('official_address_confirmed',True))
   if h and not confirmed:reason.append('holdout_record_not_confirmed_for_strict_release')
   if confirmed:
    d['retail_official']=True
    if h.get('evidence'):d['evidence_url']=h['evidence'][0].get('url') or d['evidence_url']
   if m and (d['label']!=0 or d['new_exclusion']):reason.append('new_target_or_scope_evidence_conflict')
   label=0
  else:
   rr=dict(r)
   if m:
    rr['target_label_proxy_v2']=d['label'];rr['retail_scope_officially_verified_v2']=d['retail_official']
    hard_location={'entity_or_address_conflict','legacy_osm_address_conflict','geometry_review'}
    rr['rule_screened_proxy_eligible_v2']=bool(r['core_features_complete_v2'] and d['label'] is not None and d['retail_official'] and not hard_location.intersection(release_json.loads(r['retail_scope_flags_json'])))
   reason.extend(quality_evaluate(rr,{mid:['review']} if mid in dupids else {}))
   if r['category'] not in release_TARGET_RULES['primary_target_categories'] and not (d['retail_official'] and d['verified_label']==1):
    linked=release_json.loads(r['osm_scope_evidence_json'] or '[]')
    if not any(e.get('link_status')=='accepted' and e.get('osm_shop') in {'toys','model'} for e in linked):reason.append('nonprimary_specialist_without_specific_evidence')
   if d['new_exclusion']:reason.append('target_audit_'+d['new_exclusion'])
   if d['label']!=1:reason.append('not_a_retained_target_or_frozen_comparator')
   label=1
  if r['operating_status'] in {'permanently_closed','temporarily_closed'}:reason.append('explicit_closed_status')
  if mid in additional_entity_ids:reason.append('independent_entity_address_normalization_duplicate_review')
  if reason:excluded_ids.add(mid)
  evidence_level='official_address' if d['retail_official'] else ('overture_and_osm_goods' if r['osm_goods_corroborated_v2'] else 'source_qualified_map_record')
  trace.append({'id':mid,'retained':not reason,'exclusion_reasons_json':release_json.dumps(sorted(set(reason))),
   'retail_evidence_level':evidence_level,'target_decision_json':release_json.dumps(d,ensure_ascii=False)})
  if reason:continue
  fields=['id','name','category','basic_category','address','locality','postcode','website','phone','lon','lat','x_m','y_m',
   'tract_geoid','county_geoid','county_block_id','is_bay_area_9county','population','households','land_area_km2',
   'acs_period','population_density_per_km2','under18_share','young_adult_18_34_share','median_household_income_2024_usd',
   'median_household_income_moe','income_relative_moe','college_enrolled_share_3plus','family_household_share',
   'living_alone_household_share','confidence','operating_status','source_provider']
  row={k:r[k] for k in fields}
  row.update(is_mapped_specialist=label,specialist_label_verified=d['verified_label'],
   specialist_label_basis='official_exact_address' if d['verified_label'] is not None else 'mapped_category_and_review_rules_proxy',
   retail_evidence_level=evidence_level,target_subtype=d['subtype'],audit_evidence_url=d['evidence_url'],
   snapshot_date='2026-10-25',coordinate_crs='EPSG:4326',metric_crs='EPSG:3310')
  final.append(row)
 report={'complete':False,'version':'v3','released_at_utc':release_datetime.now(release_timezone.utc).isoformat(),
  'rows':len(final),'label_counts':dict(release_Counter(r['is_mapped_specialist'] for r in final)),
  'frozen_comparator_rows':len(negative_ids),'removed_after_record_audit':len(negative_ids)-sum(r['is_mapped_specialist']==0 for r in final),
  'all_rows_manually_verified':False,'target_meaning':spec['target'],
  'rules_sha256':release_sha(quality_OUT/'cleaning_rules_v3.json'),'frozen_frame_sha256':release_sha(frozen),
  'input_sha256':release_sha(quality_OUT/'quality_review_master_v3.parquet')}
 report['additional_target_duplicate_review_groups']=len(target_groups)
 report['additional_target_duplicate_review_ids']=len(target_duplicate_ids)
 report['audit_input_sha256']=audit_input_hashes
 (quality_OUT/'target_label_rules_v3.json').write_text(release_json.dumps(release_TARGET_RULES,ensure_ascii=False,indent=2))
 release_count_features(final,release_context_metadata(),decisions,excluded_ids,report)
 release_pq.write_table(release_pa.Table.from_pylist(trace),quality_OUT/'final_cleaning_decisions_v3.parquet',compression='zstd')
 master=quality_OUT/'california_retail_clean_master_v3.parquet';release_pq.write_table(release_pa.Table.from_pylist(final),master,compression='zstd')
 c=release_duckdb.connect();c.read_parquet(str(master)).create_view('m')
 cols=c.execute('DESCRIBE m').fetchall();report['columns']=len(cols)
 report['predictor_missingness']={k:c.execute('SELECT count(*) FROM m WHERE '+k+' IS NULL').fetchone()[0] for k in release_PREDICTORS}
 assert not any(report['predictor_missingness'].values())
 assert len(final)==c.execute('SELECT count(DISTINCT id) FROM m').fetchone()[0]
 assert c.execute('SELECT count(*) FROM m WHERE other_retail_count_500m_clean>other_retail_count_1000m_clean OR food_count_500m_clean>food_count_1000m_clean OR other_target_count_500m_clean>other_target_count_1000m_clean').fetchone()[0]==0
 c.execute('COPY (SELECT * FROM m ORDER BY tract_geoid,id) TO ? (FORMAT CSV,HEADER TRUE)',[str(quality_OUT/'california_retail_clean_master_v3.csv')])
 preview=['name','address','tract_geoid','population_density_per_km2','young_adult_18_34_share','median_household_income_2024_usd','other_retail_count_500m_clean','food_count_500m_clean','is_mapped_specialist','specialist_label_basis','retail_evidence_level']
 c.execute('COPY (SELECT '+','.join(preview)+' FROM m QUALIFY row_number() OVER(PARTITION BY is_mapped_specialist ORDER BY hash(id))<=15 ORDER BY is_mapped_specialist,name) TO ? (FORMAT CSV,HEADER TRUE)',[str(quality_OUT/'clean_preview_30.csv')])
 acs=quality_ROOT/'data/interim/census/ca_acs2024_tract_features.parquet'
 c.execute(f"""COPY (SELECT a.*,coalesce(t.clean_retail_rows,0) clean_retail_rows,coalesce(t.mapped_specialist_rows,0) mapped_specialist_rows
  FROM read_parquet({base_q(acs)}) a LEFT JOIN (SELECT tract_geoid,count(*) clean_retail_rows,
    count(*) FILTER(WHERE is_mapped_specialist=1) mapped_specialist_rows FROM m GROUP BY 1) t ON a.GEOID=t.tract_geoid ORDER BY GEOID)
  TO ? (FORMAT PARQUET,COMPRESSION ZSTD)""",[str(quality_OUT/'california_tract_clean_summary_v3.parquet')])
 specout={'status':'SOURCE_QUALIFIED_MAPPED_RETAIL_MODELING_TABLE','predictor_allowlist':release_PREDICTORS,'target':'is_mapped_specialist',
  'label_is_proxy_unless':'specialist_label_verified IS NOT NULL','target_semantics':spec['target'],
  'groups':'county_geoid / larger spatial blocks; do not split one tract across train/test',
  'forbidden_X':[r[0] for r in cols if r[0] not in release_PREDICTORS],
  'training_only_preprocessing':'Fit scaling, imputation, feature selection, resampling and tuning inside training folds.',
  'imbalance':'Use PR-AUC, precision/recall, calibration and suitable class weights; accuracy alone is unsuitable.',
  'competition_features':'other_target* and distance_to_other_target* are screening-only and forbidden for classifier X.'}
 (quality_OUT/'model_feature_spec_v3.json').write_text(release_json.dumps(specout,ensure_ascii=False,indent=2))
 with (quality_OUT/'field_dictionary_v3.csv').open('w',encoding='utf-8-sig',newline='') as f:
  w=release_csv.writer(f);w.writerow(['field','type','role'])
  for name,dtype,*_ in cols:w.writerow([name,dtype,'predictor' if name in release_PREDICTORS else ('proxy_target' if name=='is_mapped_specialist' else 'audit_context_or_identifier')])
 report['clean_row_threshold_50000_met']=len(final)>=50000
 report['evidence_level_counts']=dict(release_Counter(r['retail_evidence_level'] for r in final))
 report['verified_label_counts']=dict(release_Counter(str(r['specialist_label_verified']) for r in final))
 for relative,before_hash in audit_input_hashes.items():assert release_sha(quality_ROOT/relative)==before_hash,'Audit inputs changed during release; rebuild from finalized evidence.'
 report['master_sha256']=release_sha(master);report['complete']=True
 (quality_OUT/'clean_dataset_manifest_v3.json').write_text(release_json.dumps(report,ensure_ascii=False,indent=2))
 print(release_json.dumps(report,ensure_ascii=False,indent=2))

In [85]:
with notebook_args():release_main()
rebuilt=clean_out/'california_retail_clean_master_v3.parquet'
rebuilt_manifest=json.loads((clean_out/'clean_dataset_manifest_v3.json').read_text())
assert rebuilt_manifest['complete']
show_counts(rebuilt,'Rebuilt modeling table')
display(duckdb.connect().sql("SELECT is_mapped_specialist,count(*) rows FROM read_parquet('"+str(rebuilt).replace("'","''")+"') GROUP BY 1 ORDER BY 1").to_arrow_table())
gc.collect()

/var/folders/17/2v0v148x18v8nn3pttyqhn5w0000gn/T/ipykernel_30850/1241790895.py:7: UserWarning: Arrow output on a DB-API result makes an extra copy of the rows, use con.sql(query).to_arrow_table() instead.
  FROM read_parquet({base_q(old)}) x LEFT JOIN read_parquet({base_q(p)}) p USING(id)""").to_arrow_table().to_pylist()


{
  "complete": true,
  "version": "v3",
  "released_at_utc": "2026-10-26T01:06:15.631223+00:00",
  "rows": 58513,
  "label_counts": {
    "0": 57965,
    "1": 548
  },
  "frozen_comparator_rows": 58030,
  "removed_after_record_audit": 65,
  "all_rows_manually_verified": false,
  "target_meaning": "mapped toy/collectible/TCG/model specialist status; not profit, success, demand, or current on-site truth",
  "rules_sha256": "7532001002772b15bf19e71eb4a35e72919bbf3071d90b331348ecd1d39a3f82",
  "frozen_frame_sha256": "e526de04f645027ae0494d61ee48948780c4d98e7226fba6afd708484d8fb1de",
  "input_sha256": "e61b5ea0b4762f1a82d1d0ee574738afbcc774e7e988113ddcdc40350e0bfc58",
  "additional_target_duplicate_review_groups": 36,
  "additional_target_duplicate_review_ids": 78,
  "audit_input_sha256": {
    "pipeline/target_scope_overrides_v3.json": "f8d5629fa9f2acac59e3864f1d36b7a30289ba3231605c3ed897f9663a658a45",
    "validation/comparator_holdout_v3/final_adjudication.json": "c55f0cb845b8ccfb63e365

Rebuilt modeling table : 58,513 rows; 58,513 unique IDs


pyarrow.Table
is_mapped_specialist: int64
rows: int64
----
is_mapped_specialist: [[0,1]]
rows: [[57965,548]]

18

## 9. 独立验证：类型、JOIN与暴力距离重算

此验证不调用生产清洗函数或cKDTree。检查CSV／Parquet一致性、全部13个X字段、加州点面关系、ACS值和69个MOE、冻结样本与裁决传播，并对50个随机点＋10个目标点以NumPy逐点暴力重算540项计数／距离。

函数／常量加阶段前缀，避免同一内核中的同名变量相互覆盖。

In [86]:
# Source: validation/check_clean_model_v3_independent.py; SHA256: b9e405fdf185c7a22cc2dbe5a284b83610b778275aed1224addb00ee0311b396
qa___file__ = str(RUN_ROOT / 'validation/check_clean_model_v3_independent.py')
qa___doc__ = 'Independent V3 QA: direct NumPy distances, SQL joins, no builder imports.\n\nThis tests declared rules/arithmetic, not business truth or universal manual\nverification. Run only after the release manifest is complete.'
"""Independent V3 QA: direct NumPy distances, SQL joins, no builder imports.

This tests declared rules/arithmetic, not business truth or universal manual
verification. Run only after the release manifest is complete.
"""
from pathlib import Path as qa_Path
from urllib.parse import urlsplit as qa_urlsplit
import argparse as qa_argparse
import datetime as qa_datetime
import hashlib as qa_hashlib
import json as qa_json
import re as qa_re
import sys as qa_sys
import duckdb as qa_duckdb
import numpy as qa_np
import pyarrow.parquet as qa_pq
qa_ROOT=qa_Path(qa___file__).resolve().parents[1]
qa_X=['population_density_per_km2','under18_share','young_adult_18_34_share','median_household_income_2024_usd','college_enrolled_share_3plus','family_household_share','living_alone_household_share','other_retail_count_500m_clean','other_retail_count_1000m_clean','food_count_500m_clean','food_count_1000m_clean','distance_to_mall_m_clean','distance_to_university_m_clean']

In [87]:
def qa_sqlpath(p):return "'"+str(p).replace("'","''")+"'"

In [88]:
def qa_sha(p):
 h=qa_hashlib.sha256()
 with p.open('rb') as stream:
  for data in iter(lambda:stream.read(1048576),b''):h.update(data)
 return h.hexdigest()

In [89]:
def qa_recs(obj):return obj if isinstance(obj,list) else obj.get('records',obj.get('overrides',obj.get('adjudications',[])))

In [90]:
def qa_main():
 ap=qa_argparse.ArgumentParser(description=qa___doc__);ap.add_argument('--data-dir',type=qa_Path,default=qa_ROOT/'data/clean_v3');ap.add_argument('--report',type=qa_Path,default=qa_ROOT/'validation/clean_model_v3_independent_qa.json');a=ap.parse_args();out=a.data_dir
 paths={k:out/v for k,v in {'m':'california_retail_clean_master_v3.parquet','ctx':'context_clean_v3.parquet','frozen':'frozen_comparator_frame_v3.parquet','quality':'quality_review_master_v3.parquet','trace':'final_cleaning_decisions_v3.parquet','tract':'california_tract_clean_summary_v3.parquet'}.items()}
 paths.update(acs=qa_ROOT/'data/interim/census/ca_acs2024_tract_features.parquet',geo=qa_ROOT/'data/interim/census/ca_tracts_2024.parquet',pre=qa_ROOT/'data/interim/integration/context_pre_dedup.parquet')
 metadata={k:out/v for k,v in {'manifest':'clean_dataset_manifest_v3.json','csv':'california_retail_clean_master_v3.csv','spec':'model_feature_spec_v3.json','rules':'cleaning_rules_v3.json','duplicates':'duplicate_review_groups_v3.json'}.items()}
 metadata.update(design=qa_ROOT/'validation/comparator_holdout_v3/sampling_design.json',sample=qa_ROOT/'validation/comparator_holdout_v3/sample_frame.json',holdout=qa_ROOT/'validation/comparator_holdout_v3/final_adjudication.json',target=qa_ROOT/'pipeline/target_scope_overrides_v3.json',target_audit=qa_ROOT/'validation/target_scope_audit_v3.json')
 metadata['extra_exclusions']=qa_ROOT/'pipeline/additional_entity_exclusions_v3.json'
 tracked={**paths,**metadata};before={k:{'path':str(p),'sha256':qa_sha(p),'bytes':p.stat().st_size,'mtime_utc':qa_datetime.datetime.fromtimestamp(p.stat().st_mtime,qa_datetime.timezone.utc).isoformat()} for k,p in tracked.items()}
 js={k:qa_json.loads(p.read_text()) for k,p in metadata.items() if p.suffix=='.json'};mfest=js['manifest'];checks={};metrics={};details={};c=qa_duckdb.connect();c.execute('LOAD spatial');c.execute("SET threads=2;SET memory_limit='2GB'")
 for k,p in paths.items():c.execute(f'CREATE VIEW {k} AS SELECT * FROM read_parquet({qa_sqlpath(p)})')
 def ck(k,condition):checks[k]=bool(condition)
 def zero(k,query):
  n=c.execute(query).fetchone()[0];metrics[k+'_violations']=n;ck(k,n==0)
 ck('manifest_complete',mfest.get('complete') is True);ck('not_universally_manually_verified',mfest.get('all_rows_manually_verified') is False)
 ck('master_hash_matches_manifest',before['m']['sha256']==mfest.get('master_sha256'));ck('quality_input_hash_matches_manifest',before['quality']['sha256']==mfest.get('input_sha256'))
 hashes=mfest.get('audit_input_sha256',{});required_hash_paths={'pipeline/target_scope_overrides_v3.json','validation/comparator_holdout_v3/final_adjudication.json','validation/target_scope_audit_v3.json','pipeline/additional_entity_exclusions_v3.json'};ck('all_four_audit_input_hashes_match_manifest',set(hashes)==required_hash_paths and all((qa_ROOT/p).is_file() and qa_sha(qa_ROOT/p)==h for p,h in hashes.items()))
 ck('frozen_frame_hash_consistent',before['frozen']['sha256']==mfest['frozen_frame_sha256']==js['design']['frame_sha256']==js['holdout']['frame_sha256'])
 ck('frozen_rules_hash_consistent',before['rules']['sha256']==mfest['rules_sha256']==js['design']['rule_spec_sha256']==js['holdout']['rules_sha256'])
 ck('holdout_sample_hash_consistent',before['sample']['sha256']==js['design']['sample_json_sha256']==js['holdout']['sample_json_sha256']);ck('target_overrides_complete',js['target'].get('complete') is True)
 n,u=c.execute('SELECT count(*),count(DISTINCT id) FROM m').fetchone();metrics['master_rows']=n;ck('at_least_50000_rule_qualified_rows',n>=50000);ck('master_ids_unique_and_count_matches_manifest',n==u==mfest['rows'])
 labels=dict(c.execute('SELECT is_mapped_specialist,count(*) FROM m GROUP BY 1').fetchall());metrics['label_counts']={str(k):v for k,v in labels.items()};ck('binary_target_both_classes',set(labels)=={0,1})
 ck('exact_13_predictor_allowlist',js['spec'].get('predictor_allowlist')==qa_X and len(qa_X)==13)
 zero('predictors_no_null_or_infinite','SELECT count(*) FROM m WHERE '+' OR '.join(f'"{f}" IS NULL OR NOT isfinite("{f}")' for f in qa_X))
 ck('competition_features_not_classifier_X',not any('target' in f or 'specialist' in f for f in qa_X))
 c.execute(f"CREATE VIEW csv_raw AS SELECT * FROM read_csv({qa_sqlpath(metadata['csv'])},header=true,all_varchar=true,allow_quoted_nulls=false)")
 cols=c.execute('DESCRIBE m').fetchall();c.execute('CREATE VIEW csv_typed AS SELECT '+','.join(f'try_cast("{name}" AS {dtype}) AS "{name}"' for name,dtype,*_ in cols)+' FROM csv_raw')
 zero('csv_parquet_all_columns_exact','SELECT count(*) FROM ((SELECT * FROM csv_typed EXCEPT ALL SELECT * FROM m) UNION ALL (SELECT * FROM m EXCEPT ALL SELECT * FROM csv_typed))')
 zero('master_subset_of_quality_input','SELECT count(*) FROM m LEFT JOIN quality q USING(id) WHERE q.id IS NULL')
 zero('US_country_CA_region_conflicts_absent',"SELECT count(*) FROM m JOIN quality q USING(id) WHERE q.address_country_conflict OR coalesce(q.address_country,'') NOT IN ('','US','USA') OR upper(trim(coalesce(q.address_region,''))) NOT IN ('','CA','CALIFORNIA','US-CA')")
 jn,ju,jbad=c.execute('SELECT count(*),count(DISTINCT m.id),count(*) FILTER(WHERE NOT ST_Covers(g.geometry,ST_Point(m.lon,m.lat))) FROM m JOIN geo g ON m.tract_geoid=g.GEOID').fetchone();metrics['exact_CA_join_rows']=jn;ck('exact_CA_polygon_join_no_multiplication',jn==ju==n and jbad==0)
 zero('GEOID_county_string_format',"SELECT count(*) FROM m WHERE NOT regexp_full_match(tract_geoid,'06[0-9]{9}') OR county_geoid<>left(tract_geoid,5) OR county_block_id<>county_geoid")
 zero('source_address_coordinates_unchanged','SELECT count(*) FROM m JOIN quality q USING(id) WHERE m.address IS DISTINCT FROM q.address OR m.lon IS DISTINCT FROM q.lon OR m.lat IS DISTINCT FROM q.lat OR m.x_m IS DISTINCT FROM q.x_m OR m.y_m IS DISTINCT FROM q.y_m OR m.tract_geoid IS DISTINCT FROM q.tract_geoid')
 projection=c.execute("SELECT max(abs(x_m-ST_X(ST_Transform(ST_Point(lon,lat),'EPSG:4326','EPSG:3310',always_xy:=true)))),max(abs(y_m-ST_Y(ST_Transform(ST_Point(lon,lat),'EPSG:4326','EPSG:3310',always_xy:=true)))) FROM m").fetchone();metrics['projection_max_error_m']=list(projection);ck('EPSG3310_metric_coordinates',max(projection)<1e-6)
 acs_cols=qa_pq.read_schema(paths['acs']).names;shared=[f for f in acs_cols if f in qa_pq.read_schema(paths['m']).names and f!='GEOID'];metrics['retained_ACS_columns_compared']=shared
 zero('master_ACS_values_and_nulls_unchanged','SELECT count(*) FROM m JOIN acs s ON m.tract_geoid=s.GEOID WHERE '+' OR '.join(f'm."{f}" IS DISTINCT FROM s."{f}"' for f in shared))
 zero('density_uses_official_land_km2','SELECT count(*) FROM m WHERE population_density_per_km2 IS DISTINCT FROM population/NULLIF(land_area_km2,0)')
 tn,tu=c.execute('SELECT count(*),count(DISTINCT GEOID) FROM tract').fetchone();ck('all_9129_tracts_retained_once',tn==tu==9129)
 zero('all_tract_source_ACS_and_MOE_values_preserved','SELECT count(*) FROM tract t JOIN acs s USING(GEOID) WHERE '+' OR '.join(f't."{f}" IS DISTINCT FROM s."{f}"' for f in acs_cols))
 moes=[f for f in acs_cols if f.startswith('B') and f.endswith('M')];metrics['source_MOE_columns']=len(moes);zero('MOE_negative_sentinels_absent','SELECT count(*) FROM tract WHERE '+' OR '.join(f'"{f}"<0' for f in moes))
 zero('zero_demographic_denominators_null','SELECT count(*) FROM tract WHERE (population=0 AND (under18_share IS NOT NULL OR young_adult_18_34_share IS NOT NULL)) OR (households=0 AND (family_household_share IS NOT NULL OR living_alone_household_share IS NOT NULL)) OR (land_area_km2=0 AND population_density_per_km2 IS NOT NULL)')
 zero('tract_counts_match_master','SELECT count(*) FROM tract t LEFT JOIN (SELECT tract_geoid,count(*) all_n,count(*) FILTER(WHERE is_mapped_specialist=1) target_n FROM m GROUP BY 1) z ON t.GEOID=z.tract_geoid WHERE t.clean_retail_rows<>coalesce(z.all_n,0) OR t.mapped_specialist_rows<>coalesce(z.target_n,0)')
 zero('all_negatives_subset_of_frozen_frame','SELECT count(*) FROM m LEFT JOIN frozen f USING(id) WHERE m.is_mapped_specialist=0 AND f.id IS NULL');metrics['frozen_negative_rows']=c.execute('SELECT count(*) FROM frozen').fetchone()[0];ck('frozen_count_matches_manifest',metrics['frozen_negative_rows']==mfest['frozen_comparator_rows'])
 rows=qa_pq.read_table(paths['m']).to_pylist();byid={r['id']:r for r in rows};ids_final=set(byid);hs=qa_recs(js['holdout']);sample=qa_recs(js['sample'])
 ck('holdout_100_unique_and_exact_frozen_sample',len(hs)==len({r['id'] for r in hs})==100 and {r['id'] for r in hs}=={r['id'] for r in sample})
 bad_hold={r['id'] for r in hs if not(r.get('decision') in {'exact_address_goods_confirmed','retailer_confirmed'} and r.get('official_address_confirmed',True))};metrics['nonconfirmed_holdout_rows']=len(bad_hold);details['retained_nonconfirmed_holdout']=sorted(ids_final&bad_hold);ck('nonconfirmed_holdout_excluded',not details['retained_nonconfirmed_holdout'])
 qrows=qa_pq.read_table(paths['quality'],columns=['id','geometry_requires_review','entity_match_ambiguous','retail_scope_flags_json']).to_pylist();hard={'entity_or_address_conflict','legacy_osm_address_conflict','geometry_review'}
 details['retained_hard_location_conflicts']=[r['id'] for r in qrows if r['id'] in ids_final and (r['geometry_requires_review'] or r['entity_match_ambiguous'] or hard.intersection(qa_json.loads(r['retail_scope_flags_json'])))];ck('official_address_never_bypasses_hard_location_conflicts',not details['retained_hard_location_conflicts'])
 duplicate_ids={rid for g in js['duplicates'] for rid in g['ids']};details['retained_quality_duplicate_ids']=sorted(duplicate_ids&ids_final);ck('quality_duplicate_risk_members_excluded',not details['retained_quality_duplicate_ids'])
 target_groups=js['target_audit'].get('duplicate_review_groups',[]);target_duplicate_ids={r['id'] for g in target_groups for r in g['rows']};details['retained_target_duplicate_ids']=sorted(target_duplicate_ids&ids_final);ck('all_extra_target_duplicate_group_members_excluded',not details['retained_target_duplicate_ids'])
 details['retained_explicit_target_exclusions']=[r['id'] for r in qa_recs(js['target']) if r['id'] in ids_final and (r.get('exclude_from_training') is True or r.get('retail_scope_decision')=='nonretail_confirmed' or r.get('storefront_format')=='vending_robo' or r.get('target_decision')=='ambiguous')];ck('ROBO_and_explicit_target_audit_exclusions_honored',not details['retained_explicit_target_exclusions'])
 bad_quality=[]
 for r in rows:
  website=qa_urlsplit(r['website'] or '');contact=website.scheme in {'http','https'} and '.' in (website.hostname or '') or len(qa_re.sub(r'\D','',r['phone'] or ''))>=10
  valid=bool(qa_re.match(r'^\s*\d+[a-z]?(?:\s|[-/])',r['address'] or '',qa_re.I) and (r['locality'] or '').strip() and qa_re.fullmatch(r'\d{5}(?:-\d{4})?',(r['postcode'] or '').strip()) and contact)
  valid=valid and (r['retail_evidence_level']=='official_address' or r['confidence'] is not None and r['confidence']>=js['rules']['minimum_confidence'] and r['operating_status']=='open') and r['operating_status'] not in {'permanently_closed','temporarily_closed'}
  if not valid:bad_quality.append(r['id'])
 details['retained_basic_quality_violations']=bad_quality;ck('address_contact_status_confidence_rules_executed',not bad_quality)
 zero('verified_labels_agree_with_proxy_target',"SELECT count(*) FROM m WHERE specialist_label_verified IS NOT NULL AND (specialist_label_verified<>is_mapped_specialist OR retail_evidence_level<>'official_address')")
 ctx=qa_pq.read_table(paths['ctx']);ids=qa_np.asarray(ctx['id'].to_pylist(),dtype=object);xs=ctx['x_m'].to_numpy();ys=ctx['y_m'].to_numpy();flags={f:qa_np.asarray(ctx[f].to_pylist(),dtype=bool) for f in ['other_retail','food','target','mall','university']};metrics['context_counts']={f:int(v.sum()) for f,v in flags.items()};ck('context_IDs_unique',len(ids)==len(set(ids)));ck('context_coordinates_finite',qa_np.isfinite(xs).all() and qa_np.isfinite(ys).all())
 ctxmap={r['id']:r for r in ctx.to_pylist()};ck('every_final_record_has_correct_context_flags',all(r['id'] in ctxmap and ctxmap[r['id']]['target']==(r['is_mapped_specialist']==1) and ctxmap[r['id']]['other_retail']==(r['is_mapped_specialist']==0) for r in rows))
 excluded={r['id'] for r in qa_pq.read_table(paths['trace'],columns=['id','retained']).to_pylist() if not r['retained']};ck('excluded_master_not_reintroduced_in_context',not(excluded&set(ids)))
 extra_ids={rid for group in js['extra_exclusions']['groups'] for rid in group['ids']};details['retained_additional_entity_exclusion_ids']=sorted(extra_ids&(ids_final|set(ids)));ck('all_twelve_additional_entity_exclusions_absent_from_master_and_context',len(extra_ids)==12 and not details['retained_additional_entity_exclusion_ids'])
 # A deliberately conservative, independent strong-identity duplicate check.
 # Exact normalized full addresses are narrower than exhaustive identity review.
 bgdups=c.execute("""SELECT regexp_replace(lower(p.name),'[^a-z0-9]','','g') name_key,regexp_replace(lower(p.address),'[^a-z0-9]','','g') address_key,regexp_extract(p.postcode,'[0-9]{5}',0) zip5,list(DISTINCT c.id) ids FROM ctx c JOIN pre p ON c.id=p.id WHERE p.name IS NOT NULL AND p.address IS NOT NULL AND p.postcode IS NOT NULL GROUP BY 1,2,3 HAVING name_key<>'' AND address_key<>'' AND length(zip5)=5 AND count(DISTINCT c.id)>1""").fetchall();details['retained_exact_strong_context_duplicate_groups']=[dict(zip(['name_key','address_key','zip5','ids'],r)) for r in bgdups];ck('context_no_exact_strong_identity_duplicates',not bgdups)
 ordered=sorted(rows,key=lambda r:r['id']);rng=qa_np.random.default_rng(20261025);positives=[i for i,r in enumerate(ordered) if r['is_mapped_specialist']==1];extra=rng.choice(positives,size=10,replace=False).tolist();base=rng.choice([i for i in range(len(ordered)) if i not in extra],size=50,replace=False).tolist();comparisons=[];failures=[];count_n=distance_n=0;maxerror=0.0
 for i in base+extra:
  r=ordered[i];distance=qa_np.hypot(xs-r['x_m'],ys-r['y_m']);not_self=ids!=r['id'];entry={'id':r['id'],'target':r['is_mapped_specialist'],'selection':'target_supplement' if i in extra else 'random_master','comparisons':{}}
  for kind,flag in [('other_retail','other_retail'),('food','food'),('other_target','target')]:
   for radius in [500,1000]:
    field=f'{kind}_count_{radius}m_clean';expected=int(qa_np.count_nonzero(flags[flag]&not_self&(distance<=radius)));actual=r[field];entry['comparisons'][field]={'expected':expected,'actual':actual};count_n+=1
    if expected!=actual:failures.append({'id':r['id'],'field':field,'expected':expected,'actual':actual})
  for kind,flag in [('mall','mall'),('university','university'),('other_target','target')]:
   eligible=flags[flag]&not_self;expected=float(distance[eligible].min()) if eligible.any() else None;field=f'distance_to_{kind}_m_clean';actual=r[field];error=abs(expected-actual) if expected is not None and actual is not None else None;distance_n+=1;entry['comparisons'][field]={'expected':expected,'actual':actual,'absolute_error_m':error}
   if error is not None:maxerror=max(maxerror,error)
   if not((expected is None and actual is None) or error is not None and error<=1e-7):failures.append({'id':r['id'],'field':field,'expected':expected,'actual':actual})
  comparisons.append(entry)
 ck('brute_force_all_six_counts_three_nearest_match',not failures);metrics.update(sample_rows=len(comparisons),target_supplement_rows=len(extra),radius_count_comparisons=count_n,nearest_distance_comparisons=distance_n,nearest_max_absolute_error_m=maxerror)
 zero('all_counts_nonnegative_nested','SELECT count(*) FROM m WHERE other_retail_count_500m_clean<0 OR food_count_500m_clean<0 OR other_target_count_500m_clean<0 OR other_retail_count_500m_clean>other_retail_count_1000m_clean OR food_count_500m_clean>food_count_1000m_clean OR other_target_count_500m_clean>other_target_count_1000m_clean')
 stable={k:qa_sha(p)==before[k]['sha256'] for k,p in tracked.items()};ck('all_inputs_stable_during_QA',all(stable.values()))
 report={'complete':True,'all_checks_passed':all(checks.values()),'checked_at_utc':qa_datetime.datetime.now(qa_datetime.timezone.utc).isoformat(),'upstream_released_at_utc':mfest.get('released_at_utc'),'method':'No production builder/function imports or cKDTree. IndependentSQL/key checks and directNumPy hypot across all context points for50 random+10 target records, fixedseed20261025, exclude sameID from all requested features.','interpretation':f'{n} counts rule-cleaned records usable for mapped-label modeling, not{n} manually verified or certainly operating retail entities. Half the holdout remains unresolved, so it cannot prove the truly valid retailer population exceeds50000. Holdout results describe the original frozen frame, not fresh independent validation of the corrected release. Computation/rule execution is verified; business truth, label accuracy, demand/profit and exhaustive entity deduplication are not.','checks':checks,'metrics':metrics,'details':details,'input_snapshots':before,'input_hashes_unchanged':stable,'feature_mismatches':failures,'sample_records':comparisons}
 a.report.parent.mkdir(parents=True,exist_ok=True);a.report.write_text(qa_json.dumps(report,ensure_ascii=False,indent=2)+'\n');print(qa_json.dumps({'complete':True,'all_checks_passed':report['all_checks_passed'],'release':report['upstream_released_at_utc'],'checks':len(checks),'failed_checks':[k for k,v in checks.items() if not v],'metrics':metrics,'report':str(a.report)},ensure_ascii=False,indent=2));c.close();return 0 if report['all_checks_passed'] else 1

In [91]:
with notebook_args('--data-dir',clean_out,'--report',RUN_ROOT/'validation/clean_model_v3_independent_qa.json'):
    assert qa_main()==0
independent=json.loads((RUN_ROOT/'validation/clean_model_v3_independent_qa.json').read_text())
assert independent['all_checks_passed']
print('Independent checks passed:',len(independent['checks']))
gc.collect()

{
  "complete": true,
  "all_checks_passed": true,
  "release": "2026-10-26T01:06:15.631223+00:00",
  "checks": 48,
  "failed_checks": [],
  "metrics": {
    "master_rows": 58513,
    "label_counts": {
      "0": 57965,
      "1": 548
    },
    "predictors_no_null_or_infinite_violations": 0,
    "csv_parquet_all_columns_exact_violations": 0,
    "master_subset_of_quality_input_violations": 0,
    "US_country_CA_region_conflicts_absent_violations": 0,
    "exact_CA_join_rows": 58513,
    "GEOID_county_string_format_violations": 0,
    "source_address_coordinates_unchanged_violations": 0,
    "projection_max_error_m": [
      0.0,
      0.0
    ],
    "retained_ACS_columns_compared": [
      "acs_period",
      "population",
      "under18_share",
      "young_adult_18_34_share",
      "median_household_income_2024_usd",
      "median_household_income_moe",
      "college_enrolled_share_3plus",
      "households",
      "family_household_share",
      "living_alone_household_share",
   

24

## 10. 与已发布总表逐列比较，确认原数据未改

比较全部51列的值与NULL，而非仅比较行数。浮点值允许1e-8的计算精度误差，标识、地址、标签和其他字段要求精确一致；下面还会检查发布表与冻结研究输入前后SHA不变。

In [92]:
reference=INPUTS/'reference_release/california_retail_clean_master_v3.parquet'
with duckdb.connect() as con:
    con.read_parquet(str(rebuilt)).create_view('rebuilt')
    con.read_parquet(str(reference)).create_view('reference')
    schema=con.execute('DESCRIBE rebuilt').fetchall()
    reference_schema=con.execute('DESCRIBE reference').fetchall()
    assert [(r[0],r[1]) for r in schema]==[(r[0],r[1]) for r in reference_schema]
    assert con.execute('SELECT count(*) FROM ((SELECT id FROM rebuilt EXCEPT SELECT id FROM reference) UNION ALL (SELECT id FROM reference EXCEPT SELECT id FROM rebuilt))').fetchone()[0]==0
    conditions=[]
    for name,dtype,*_ in schema:
        left='r."'+name+'"';right='p."'+name+'"'
        if dtype in {'DOUBLE','FLOAT'}:
            mismatch='(('+left+' IS NULL) <> ('+right+' IS NULL) OR ('+left+' IS NOT NULL AND '+right+' IS NOT NULL AND abs('+left+'-'+right+')>1e-8))'
        else:mismatch=left+' IS DISTINCT FROM '+right
        conditions.append((name,mismatch))
    differences={name:con.execute('SELECT count(*) FROM rebuilt r JOIN reference p USING(id) WHERE '+condition).fetchone()[0] for name,condition in conditions}
assert not any(differences.values()),{k:v for k,v in differences.items() if v}
for path,before in PROTECTED_BEFORE.items():assert notebook_sha(path)==before,'Published input modified: '+path
result={'complete':True,'generated_at_utc':datetime.now(timezone.utc).isoformat(),
 'rows':rebuilt_manifest['rows'],'columns':rebuilt_manifest['columns'],
 'label_counts':rebuilt_manifest['label_counts'],'source_code_embedded':True,'external_project_py_imports':False,
 'all_51_fields_agree_with_reference':True,'reference_column_mismatches':differences,
 'original_release_and_frozen_inputs_unchanged':True,'independent_checks':len(independent['checks']),
 'run_root':str(RUN_ROOT),'output_parquet':str(rebuilt),'elapsed_seconds':round(time.monotonic()-RUN_STARTED,2)}
(RUN_ROOT/'notebook_cleaning_reproduction.json').write_text(json.dumps(result,ensure_ascii=False,indent=2))
print(json.dumps(result,ensure_ascii=False,indent=2))
display(pq.read_table(rebuilt).slice(0,5))

{
  "complete": true,
  "generated_at_utc": "2026-10-26T01:06:30.730686+00:00",
  "rows": 58513,
  "columns": 51,
  "label_counts": {
    "0": 57965,
    "1": 548
  },
  "source_code_embedded": true,
  "external_project_py_imports": false,
  "all_51_fields_agree_with_reference": true,
  "reference_column_mismatches": {
    "id": 0,
    "name": 0,
    "category": 0,
    "basic_category": 0,
    "address": 0,
    "locality": 0,
    "postcode": 0,
    "website": 0,
    "phone": 0,
    "lon": 0,
    "lat": 0,
    "x_m": 0,
    "y_m": 0,
    "tract_geoid": 0,
    "county_geoid": 0,
    "county_block_id": 0,
    "is_bay_area_9county": 0,
    "population": 0,
    "households": 0,
    "land_area_km2": 0,
    "acs_period": 0,
    "population_density_per_km2": 0,
    "under18_share": 0,
    "young_adult_18_34_share": 0,
    "median_household_income_2024_usd": 0,
    "median_household_income_moe": 0,
    "income_relative_moe": 0,
    "college_enrolled_share_3plus": 0,
    "family_household_share"

pyarrow.Table
id: string
name: string
category: string
basic_category: string
address: string
locality: string
postcode: string
website: string
phone: string
lon: double
lat: double
x_m: double
y_m: double
tract_geoid: string
county_geoid: string
county_block_id: string
is_bay_area_9county: bool
population: int64
households: int64
land_area_km2: double
acs_period: string
population_density_per_km2: double
under18_share: double
young_adult_18_34_share: double
median_household_income_2024_usd: int64
median_household_income_moe: int64
income_relative_moe: double
college_enrolled_share_3plus: double
family_household_share: double
living_alone_household_share: double
confidence: double
operating_status: string
source_provider: string
is_mapped_specialist: int64
specialist_label_verified: int64
specialist_label_basis: string
retail_evidence_level: string
target_subtype: string
audit_evidence_url: string
snapshot_date: string
coordinate_crs: string
metric_crs: string
other_retail_count_500m_c

## 使用结果

完整清洗输出已写到本次`RUN_ROOT/data/clean_v3/`；后续建模读取该目录的`california_retail_clean_master_v3.parquet`，按`model_feature_spec_v3.json`选择13列X与`is_mapped_specialist`代理目标。`clean_preview_30.csv`仅供查看。

这些检查证明处理、关联和计算一致，不能证明每条都是现实中当前营业的实体零售店。原冻结100条比较点审核为43确认、7问题、50未决，后两类已从总表隔离；不能把它们转成0或把修正后的同批样本称新的独立精度验证。

本Notebook不做全局插补、不训练模型、不把同一家店复制到多个虚构年份。后续需按Tract／县或更大空间块分组验证，训练折内处理缩放、重采样与调参。